# E24 — SGLang speed track on 4×L4 (SGLang 0.5.21, Qwen3.8-27B) · rev 1 (DESIGN rev 2) · FULL_A

Goal: 1500+ tok/s sustained accepted-output decode for Qwen3.8-27B on Kaggle 4×L4 with SGLang 0.5.21 (RedHat INT4 W4A16
+ DFlash2), with the SGLang accuracy gap explained first. Six sections (README / DESIGN.md rev 2):
1. **Accuracy-gap decomposition**: vLLM 0.32 FP8 / INT4 target-only oracles, SGLang target-only (fp16 / fp32 SSM),
   DFlash W6 / W4 / forced-reject, cold / warm / cache-off, multi-boot A/A; M1 greedy rows, M2 c16, M3 FP8-generated
   teacher-forced traces up to 120k, M4 decode vs prefill, M5 rank probe, M6 tool-call probe, M7 cache isolation.
2. **DFlash width × streams** (W4 / W6 × S16 / S20, W5 or W7 by rule), equal-N windows, parity rows.
3. **TP4 collective critical path** (NCCL CTAs / ALGO / PROTO / NTHREADS) with profiles and env proofs.
4. **Capacity census at 90k / 116k** (Mamba slots 68 / 84 (+88 / 92 if Mamba-bound), mem-fraction 0.80 / 0.83 / 0.86).
5. **Prefill / decode interference** (chunk 2048 / 4096 / 8192, prefill-decode-interval 1 / 4; D0-D3; starvation).
6. **Correctness backports** #43567+#43559, #43029, #41736, #41737 (unit + negative tests, trigger evidence required).

Attach (DESIGN §2; FP8 in every notebook: prompt ids come from its tokenizer.json):

| Input | Mount |
|---|---|
| SGLang wheelhouse | `/kaggle/input/datasets/foysalemonshanto/sglang-0-5-21-cu13-wheelhouse` |
| vLLM 0.32 wheelhouse (SMOKE, FULL_A) | `/kaggle/input/datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse` |
| INT4 model | `/kaggle/input/models/foysalemonshanto/qwen3-8-27b-int4-redhat/pytorch/w4a16-g128/1` |
| DFlash2 model | `/kaggle/input/models/foysalemonshanto/qwen3-8-27b-dflash2/pytorch/bf16/1` |
| FP8 model | `/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1` |
| ARC tasks | `/kaggle/input/competitions/arc-prize-2026-arc-agi-2` |

This copy runs `MODE=FULL`, `PART=A` by default (the code is identical in the three E24 notebooks).
Every number is a measured counter; a missing measurement is INCONCLUSIVE / SKIPPED, never PASS or zero, and nothing
defaults to GO. Outputs: `/kaggle/working/e24/` (`e24_results.json`, `e24_report.md`, server logs, per-arm summaries,
`e24_raw.tar.gz`).


In [ ]:
import os, sys, json, time, re, shutil, subprocess, hashlib, threading, traceback, base64, gzip, tarfile, inspect
import random, math, signal, socket, urllib.request as _U
from pathlib import Path
T_START = time.time()
MODE = os.environ.get("Q27X_E24_MODE", "FULL").upper()
PART = os.environ.get("Q27X_E24_PART", "A").upper()
assert MODE in ("SMOKE", "FULL") and PART in ("A", "B", "ALL"), (MODE, PART)
SM = MODE == "SMOKE"
VARIANT = "SMOKE" if SM else {"A": "FULL_A", "B": "FULL_B", "ALL": "FULL_ALL"}[PART]
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
FAKE = os.environ.get("E24_FAKE_SERVER", "").strip().lower() in {"1", "true", "yes"}
TS = float(os.environ.get("E24_TIME_SCALE") or 1.0) if FAKE else 1.0
FORCE_COND = FAKE and os.environ.get("E24_FORCE_CONDITIONAL", "").strip() == "1"
BUDGET_MIN = 210 if SM else {"A": 480, "B": 690, "ALL": 690}[PART]
BUDGET = int(float(os.environ.get("E24_BUDGET_SEC") or BUDGET_MIN * 60 * TS))
HARD_STOP = T_START + BUDGET
MAX_LEN, SGL_CTX = 131072, 155648
OUT = Path(os.environ.get("E24_OUT", "/kaggle/working/e24"))
OUT.mkdir(parents=True, exist_ok=True)
LOGS = OUT / "logs"
LOGS.mkdir(exist_ok=True)
ARMS_OUT = OUT / "arms"
ARMS_OUT.mkdir(exist_ok=True)
TMP = Path(os.environ.get("E24_TMP", "/tmp"))
TMP.mkdir(parents=True, exist_ok=True)
WORK = TMP / "e24"
WORK.mkdir(parents=True, exist_ok=True)
INP = Path(os.environ.get("E24_INPUT", "/kaggle/input"))
R = {"mode": MODE, "part": PART, "variant": VARIANT, "rev": "1 (DESIGN rev 2)", "budget_min": round(BUDGET / 60, 1),
     "fake": FAKE, "time_scale": TS, "force_conditional": FORCE_COND, "max_len": MAX_LEN, "sglang_context": SGL_CTX,
     "start_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "inputs": {}, "identity": {}, "setup": {},
     "sections": {s: {} for s in ("1", "2", "3", "4", "5", "6")}, "comparisons": {}, "argv": {}, "errors": [], "order": [],
     "shas": {}, "api_drift": {}, "gates": {}, "not_measured": {
         "coverage": "needs the Duck / Poetiq harness (paired ARC sample session, DESIGN §5.1)",
         "adoption_correct_outputs_per_gpu_hour": "needs the paired ARC sample (DESIGN §5.1)",
         "agent_end_to_end_denominator": "needs the Duck / Poetiq harness (Y11 third denominator)"}}
_SAVE_LOCK = threading.Lock()
ABORT = {"why": None}
def save():
    with _SAVE_LOCK:
        tmp = OUT / "e24_results.json.tmp"
        tmp.write_text(json.dumps(R, indent=1, default=str))
        tmp.replace(OUT / "e24_results.json")
def left():
    return HARD_STOP - time.time()
def log(*a):
    print(f"[{(time.time() - T_START) / 60:6.1f} min]", *a, flush=True)
def err(where, e=None, **kw):
    """Every caught exception is recorded (E23 defect: R["errors"] was never written)."""
    rec = {"where": where, "t_min": round((time.time() - T_START) / 60, 2), **kw}
    if e is not None:
        rec.update(error=f"{type(e).__name__}: {e}"[:800], tb=traceback.format_exc()[-2500:])
    R["errors"].append(rec)
    try:
        save()
    except Exception:
        pass
    return rec
def find(pattern, root=INP, must=True):
    """E23 find() fixed (e23_template §13.6 #13): the shallowest hit under `root`; two different hits at that depth
    (two attached versions) are refused instead of silently taking the first."""
    hits = sorted(Path(root).rglob(pattern)) if Path(root).exists() else []
    if hits:
        d0 = min(len(h.parts) for h in hits)
        top = [h for h in hits if len(h.parts) == d0]
        if len(top) > 1:
            raise FileNotFoundError(f"AMBIGUOUS: {len(top)} hits for {pattern} under {root}: {[str(x) for x in top[:4]]}")
        return top[0]
    if must:
        raise FileNotFoundError(f"{pattern} not under {root}: attach the input")
    return None
def _dir_size(p):
    try:
        fs = [f for f in Path(p).rglob("*") if f.is_file()]
        return {"files": len(fs), "bytes": sum(f.stat().st_size for f in fs)}
    except Exception as e:
        return {"error": repr(e)[:200]}
if RERUN:
    tasks = json.load(open(find("arc-agi_test_challenges.json")))
    sub = {k: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in v["test"]] for k, v in tasks.items()}
    _sp = Path(os.environ.get("E24_SUBMISSION_PATH") or "/kaggle/working/submission.json")
    _sp.parent.mkdir(parents=True, exist_ok=True)
    _sp.write_text(json.dumps(sub))
    print("E24 lab notebook: competition rerun -> fallback submission.json only")
else:
    MODS = WORK / "mods"
    MODS.mkdir(parents=True, exist_ok=True)
    BUILD_SHAS = {'lab': '52d0d7423ea78096bb41bef5567de82cd2955af7ee670f2e8e7f8dd062857ee7', 'patches': '96a9725130da728286f312632a967d504f0f760361ddf9f10933088b34dfef33', 'hook': '24b4b4788df326cab4ce1dde628ddb76c1e340cd491b7cfd74633495f1283b70', 'unit': '86cb257c1da9648013aad17e3fa3e944b19dca03ebc9138c948e7e5d869607ef', 'loadpath': '18325602071c4f56becd5d8ae5da89f50b79c8f1e148110f5a2f2bffa9aa3190', 'trace_summary': '8d89ca106e166c97204f1aeff931a50a75e203999c876f248b34e6ee24212b46', 'notebook_builder': 'ce14cbe114ca4986a95c5eb69e099d32ea13babb2b5d4819373c8191280a71c2'}
    BLOBS = {'lab': ('e24_lab.py', '...'), 'patches': ('e24_sglang_patches.py', '96a9725130da728286f312632a967d504f0f760361ddf9f10933088b34dfef33', 'H4sIAAAAAAACE+197XbbRrLgfz1FD32yBhKSBvhNepQ9TqzMeMeRvbYzd+dotFADaEgYUQADgJZ1PT7nPs3d99hHuU+y9dGNL4KUPJPkTuauT2KTYKO6urq+urq6utfrnYwm4nIrs1CFfZGp9yorYn+tRKE+FGIji+BK5SJKMyET8eL07btnL1+ePBdvf/dSJpfCGU6HIxeaBdfyUgkL3hejvhg5o9nAdQZjxx4eHX0jc7USJ+7kca4BZsK6wKfiJLmME/UkzwJPuRPvJg23a5U/wc/55Rp68DQGw83dhb0SUQyY5VdyNJ0x0nlfIMp34mqbXAP8i3QdXhDuR5s4SVQo/DsRF7lIbxP9Yl/I9XqQZoMkLa5iGMQG8KFu+uJiCF378vpC+DCk7Qbei0RxpcQmi/MCUAVwhYJOZZHexIG4zWL+KoL0ZgPIHQGywbXwFVBMAclCIaMC4CMMatwXeSFhEBeet7kLJDT3vCdfwvCCCxFm6WajwqHAOZFhmNNrUfwexqyCIk6TwYwQ26QZDMl6fvL2xe9Oj4jo4v/+n/FwAiPYJqHKoI8kfKJJWL4xvAHaQLunBBcpliPe0EWW3jTe1FR/okaT8nWPXvCc6cjF2egfbZP4x61KVJ4/SWEO1nLzBKkzIBIA5eWljJO8oM40o9xeKbU2EyiDseMsl4v/+Ld/fwoEBhiJTAIlwjiKchEn4sKgscm+5H7xJ+CDo4HoTdzFfNQTyFcr5E11synuRLZNEpxTH98UCfKGuFYKJlKGN3GeAw1FsE5zQM/KATbyWwbDAT5FmJOJ45YwQRpgfHkBbHWl1iF+9YoU/rtWicjSWxEj+bZJAcBSRFwWtV4a4PsiXK9vntzEH+LkSUe/4+lsDv2+fiMe0WdhvSeKjYDn4+S9XMehLBRRci3/9Y6GJPJ1irgRuTdpnBRirZLL4gqpnFDbdINMI9fA4kmIZMFXRCATYP0jUf3xFcpEGsiiThePiEhIilenJyJUUZzECNFIM7OQhV1dAiNk9D4yr0oKkEmYzas0AxlKxA2IQ73HIE1IwfjbQlwCg+ckWIM4hDfjADBGSc9L6kyXFXWmS2ElKczGWqEKuVOF3RevX715d/J8JUBki0zJGz3piBl8B6qlIJXIUjCJQ/xyIax38MNL+Aiv317FwC6aScNU5XVcgVjiSr5XLDYoCmKbA8YESuaFl6ShEk+EAe1lKlDxptCP8lvpbTKg3QdP/0qIg5i/A3AlwqSzar3KzWYdQy8BtE7Wd6JIDX5Is0zGiMKzoshi+K5OsgxUtPoggwLaAgNkzC0yz0GhAzUzFa7E6ckfT94g8qAScVYY4LDe7RvgeWicC82HBr+VJjhyEbEhElSuU1CJ6sMmRWTwMSu3OBls1hJFGbgPGcZMpDOqJhI+19k836hA+CBOoQTWYTW6NWAvM3lzI7MBaKubuEAu06xuEY8CTfPtugAypwGoo7QmWe58PDNd4ud6l8+/g+m7AuDpFtQuAE/ei0IC28AowGokObCjUKCja5rAp4FbYKnUOn8SRgih0du81tt8t7cRdKdUeAcafQ06HSbtVq6vqUfSeRlobIVzg0bqVJ6iogERu1ZZgh0Oh8NGp++MgUCpI5kRMkMVmv+FNAJpdhoCGDuVkUWiZ6TqnpASRfV5J1LwADItizST8kZbOzDhz16/AL1QhOvYB9zWd0+1ao1v2BSxpXmqbXG0TchaAdmKbZYASgLYAQb3P96+Oh0AQ8ag0P5VopcRxkFBSDE8zdY4/KM83WYBWt4UGt6Q0j999v3J2754fvLdsx9evvP0Vxi6+fjm5H/+8OINfnr97N23v+dH3528OTn99sT77sXLE+/t75+RD1A95pbP9S/QC/GelZOxToAK+fEpsLldqoPykRh8jdMk5kPxsQfoo97v9UWPWvRW4iMoShbj8K+gRDzzGRgtWsPA/wrqMwHH5FO/LoHdf3o0uQgU5HClTehfEYtPff7Ri5Moxe611ezp57kHqMURdIxP0uveJxgjonJXHyP6JRngdPwu28JYQ4UmQyWBZgx0DSMSR9bA2wQAGyUBdlAPDRobWVHhUw1yhfx0/wiLdIuGR3PxzRY8B0L8ru5NkQbW3ibrCGoIBiy9Zlfracu9u79jGWRpzkwf4HRaUkQyXm+hS9ArIKKAjmZsdD+1R4hN7ad1Fk8BEcIJ6MuO9F4mApsO7GJ8MMtQGyQJNHVePK1TGwgM5gk6ZsqzptD0JgRuUpRZef9A2adlXyWmvgAqzJZc52gQyF97KjS35E2/V9se4DsaHZHF004yDdOu+mHngEmwMo01CbWD3Tb3sHIIUmqA6sWGPmCchfLSayu4uu6jjQFtaSQI+jKmDpHkH4F1clUYyvTZASKlR/QnM17+yExmBAO7gyF4YZxZ70GvdjBnje+gFTLBBSu9C9TaxRX4fqBvVQaWHv3997HUyhFVJvhNoUcGDj0Xfm4fffvyxUps7oorUHfoaO+seMRHYqe/krD+len5SViDASIrnr94I/4qBgNAWLz+E3yEB7Y4Gwx4uLLvwzP8LMBonB/1er0j7hksxOVGAvuZ71dgTgBN8/UveZqYz2luPmVl83zra0NbPrnLYZn37Ns//PDae/vDd9+9+F/iWPT0VPcqKj5C/gFPRNzGYLuBTUSeChXTJJmVIXinMI+qmj40usxnR+++b8Mvbja9o4Y9gF8s7cj3zSrBxq6ZK8GJleAqgNDBnBixYzm0j0pLwkDIK+8bB7RvHJi+cSv6xuLbR+atJipfVbaJRs8iQHPCRsYWxPXo/RrMzLJO1JZ19pExbNDFR43PqkLR/nRU2TNw8d6+eHWKFGKh7R0deW/RxOGjPCue3MgE+DjLGyuR3pH3/OXL702bPesVaPX2GzSYe2GVqwZo+v2z7795VrZUNx4teJ/AwhGlTn/DlTOoRlB+AOzGl/zqN29e70fXq72z1wHEAX338tnb33vfv3p+8rKE1vLcqmZ/OHlzyu2Mt5VuoFMQ2+0avNj3qv7S0SOzjNX24PYqXYOu3ybMyKHRmKxpLBbuAT8cBJuxOxIwsLt1nGw/6HXxMQH6g7y8XCt+dJWis44gwNfr9GSQIUjz8xSvRM+fuWE09edTR4bBfLKQy2Wg/PHC9ReT5SRwl8vIH0dzZxLOl+NQTqUzcud+OPN9f+bMgkmPvRHiBgA3Ws5Gi8l8Ph6Nw+lU+e40mPkylLPpcrpYSDVbOs7CATkLw7mj5Hzij2aOihxHLYPpwp0bcMw2iN8omi+C+cIN1WIRjGeBGs2DYBpOJv54Mp6MnPF44U9COVouocE0mM6ns9k0nEUSBjAOpAFIzAXwAtdZzNRY+s58LgH2dDmbRYvJFEgwH09mSobLqetMlnO1dNyRs5wsnGk0C8NpNFqGyyAw8IDjcLhAOhnNZnM1W8yXajnznQhQGwMWAH8RjpeLZeiMZ/OJctwAiDOWzsJX4Rggs1LQ1KuxHoB1ncCXMwme+Gy5mE3Hc9edSBj2aBYCGrNw7CzUZBrADAXuIgKahpOp60cAfRlMFnO3BZZZFdF1I9educvp2I0my7Hvq2U0d8MJzI47CuTcmUejcD4CWrqL8UhNJrMomk2lCvyJMx0tcPCfarzMehk0yJ9YS5FPZpZ85C4YR68R3ljxcuIr0rZoD7VOJ9exwblNZ3uXeSdSOVN3OZ9MpjARajSdOKO5nIQjH/5zJ9Fy4S/diRMsAjkGlvSlGiFHj93pdDKPgkWLecPFbDkfBVMVjaG7+XzpQCu1BK5wgvFo7o+DIFTAb0sVzQKkh1y4EzUO1GwOHFfNZsm8U7kERl/ApM1H84mEuZnAkyUw+zKcR4sZIDULZiMgZujO5yM1decgJeOpA52rULpt5lVLN4ic8XTkj0bhbOkufWeslqE7k5F0AM2FL8dqNgr8RbAcBZOpPwmmS19O5yghUTBvMu9irvzJGP5T/nwxVjAEgDRbzubBEvltsQAJDmGg7mw8DsJxCKI4ngLfRPNAuhPX2cO84chxgdun7nQmnXGEAqaA98fAlzIYR+F0vogckPZJEKhgNvdHrh8oZxHJCGRs5Dj7mDcKXRgYyKpcjKXru/NATeczByyqq6Yw7sUyWozgBxgFCNfMAT0AamHmSAemyJ9Iw7yDn/tPPTAHboQNfRp/+KAJFC/Hk8VyMAbJpNXLS5joyQD+mnPQjJzYKgSE0U2KRcGiG5zNmn9sH3nUvff6zcmzly9ffeu9eonm/HHdR40jWEXfehTji9nfxTXH6s9J25vdpOl6CM6yRwbX28TodqbwF74LrviPfXa9vDj80C+Bnjnn9i4sDHFdvx8ypER9KLwiA3cNXwUESzB/TvhV8HIEt8VAJjnFGB0Df1RZuVrD+PlXDQRd6vdyjVHQAn/64IUZLPM4CJvTYxsG+HiHPqcn/4KW/FdEn/pL4Hx04fWIQoZxMojW8eVVwTsKtQAxr9nqweF0HdYCxMN70KNoJqOXqx/XKgEMkUQ/7+T1zOS9fX3SydifN3k/9QTiH1gheSaGfiwwRNJs90i8SsqgAc6BjlzkgSwK9O/BJnJEFMacNmdpWGNfogCz7q+MAge49jPY7Cel6S9nGcjteYhlyNgoLCajAfy1wJiqxFV2JLxLmCucBl7P0IbBeu1l8hb8J4968E6+f/3uT1pAYDlfYwwUvqFKMKQKr8Yp6IU7hIGbYhiITTNuEufe1Z2fxaGX38rVUZPcbxTGT7h/aBht12vcdcCdnBoochGl0AjiNiWMqgFJ78FpK9iEdyy+k8AlR0c15K0Hvw3DAG4hVTO8lXGB7X7cKgxYHh8Lp4Rjk7WlZgEu7WGtCQyIUkMSQ7GQBk21tfh107SGe7M5NKTdUctuo/fM7JvW90sz3C8AW7LdoE3xGrB0WC1Z37UgcaSN5ZGiYbhvlekd7LW87GOop9odvU23YJlonyW/2ha/RgZ6kKjD8mUxmDgTR1inr7x3r/5wcir8DDdGy7gBozEgNMQ6TTeVwJ++ojd2Rd5MM8zSb47FszB8o358QyGQ4bevTt+9OP3hZHXUYUDwhePWCwavVWcAeYe89T91EbmKVSYzWC2Cu8qRnb2vGcnRb65vPqt5I3rkgUOlsgS6XMcwTVknEHu1F/Yj8XZHPjlj40Ze0465tj64/0AbArS5kWselwluPeTg5qhwbx+H2Rn5kQUeoKGwwUjywhZfCwdHbh0M66MTsE/W977Y/QsZ779xDOgO1JWq4dsdtfrL8e3PPbE/06R+9oTaRhf9Qq4Oxnk+x9WZLxaD8XwJ75CDs73xdKYJyTJQPRd5fJnIYqt3+C7ke/D58NcLUoTYoVdf89bYqUGplXil013O3mocvsHn59q75CCEjZu24DBWXLPZeP42BOygHeIIqxlAUa0tewg/4arlJg6yVDtmefyvSgyqnqtMkRJvgEP6qp4v5JE/XjYhMJatZaY2xB2J+RuGSHwEhMakJQ/TB2ilBU2cn5sEjyh9RIsRb5Hijg7mSGnBkmvwLsI7cYULU0lZVJakbKA42dZ3ax8JMryYWpcr9hn8NIP2NngpKgRBEREm9sCon1LWAvYYZ5wugftFw79nYsRXTQo+3OCPF7Mp+PYYKcJkFdxc4oQJ8pvWmCmGNj+HtTusxiixUgMt0rRi+G9fff/9yem7XZZ/JF5cJqleEfEOuPFVW56KWSAOazR9wXlhSZoMXr+G+clVn7PFDgHoi26xvSIfTq5vcUPaxyQdJTnXEbSn0691K3VKCQa1aMiMPPVWaV5YYoq/8G4+JtiZNKm6kBiy7IjJT08W2UZQp5ThxvlOJqBOIaA38vrIwemtmBs32ZF585J74d3hZ/DWcjYfjJcLUKaVJ30pKQR0yJck0v3u2buTbkeyEmRtuZoGCCe7aXvApoUUb/J8JW+82ziE5ctxh1XDVT81oRSrYa012fT6j/WpIV+EPh3t91sAwl4rq+V8n8XpQLQkwW4qTtdo+0cH/YHj5prp6B5n9AHOVWvkNLwwzuUlCN0lpdhh8rRCJ+l54/H38HT4+s3Jdy9evtztmVjfLDsRQB+/Eb0EMxEl3cCCJY9AsvkROkewBoNVrLxW4kbdpNldvwM0tLhVrK1hyVsKJaUlliqZjIlRCTIotnLd+nXY5RDeN/X3q/jV0d/l51Yy1enk/jplqmHz0Fpub6wM43ZornPv+j3NVIbqpoP4/xTiuUOG48a3/y/LP70sd6+G9nHPoOWZdb37cC1w9MBYwQP1QtOM78mhwVUR2PGFIyydr+JhSy9OwGPETsi9Qrv+VGyTbc6HWNJtJp69oYldV1Yd97m7rfoj8a2ZJQ7ugTJAdI/+LjX194h11dnfJjc1xUvD7lS8P8+wfwWK8ZDW+gmU1i8bZeBTHnyoox1oeHga2Uq8dAczYemEy754OXWcwdQFwWPa6s0x2lIDoud0POQC9ST0qRcguHSJ89oJAPDnYfpBKL/Hl1njfuDzEDpRmbb238dlDjyoVD62kxd4ROZSJ7kiPovxYLpY6h5fTpezwcxZcnJAcYUrBL296gFgGqjWD5ypjHmgeKYu15jjQZ1EJ8UyQvQ35hXghhkl/ptsgyHtOU6X3ovvkc5m35XaeDD1GJDxPGFSRvHIEBmhHDeDqVWQrtd8LC03zXQOI2b160bF3YZWPvz7uz+9PvG+/f3Jt394cfq7vvgWBoHc3S/jGn3xFgmdBLTnrN8q0iy4MrukFcZ6n/Snxhh8CRms8eBM2QjmADc2fsYhdSZ1gKoIQnOkKKIditb2KqkLPODkBcB4Q09v2RDzcWxdycjLgbUtPKRk1zfyd0SAtkUoOETYDd+pJE+zWo+gA57pY2IUxrhhAQC7Spuufc30SBvcq0bLJrO7IbxWwSBZ03qOBLfDSDNKWvul2ZA+WW5tZxrdJr0P3rEHXgNOGJHwMJAT/P5aZvIGKAIqklMRjh2TuQDPjl27tQf+0+BMoSg+jtXew++L3rfgiFGGOvlrTFnqq7fDJa38gP8MLsHjID/WopFv1I8mBvnPwEC8kmosn9B3RrusPqjAsrkHsyPF3akPoN09f4u6lvJjOlIhggp1Y648UCcduRToF+9PpaiOHdZw7G5Oh6twV+HAUcT9bwLC+DIeTGudbhzm1zGeVmZIlfXdhWXvyQl5JF7r1QpG/kqzisbr21f/Ivj8F52wwbNjaDIp+PaHP5Jty4fdUGuMEarAjDPaQ8wdgvYPtSMrcABSCa1Gp/7h5nuoeFxsN+v7uiKOIs8zwBmqjVwLu4FGOgJ33NDIHYJmH8B232/2ATZ9AL+Ir46FFRR9+7+IDn9x+m7hfQuavCvT7ZF4jidm2YVkAHhM5YPQCU6ZDMHXJJh4PrVQMsQ8giDd3IHWHO5oMU5QSYqFF1xvChrlPbljZdScSb9Pwe0R6TJBTVN613rs4RasACEDPGSBQa/9THhw/ui0serOeLtXaKskuM7J6kjK++efLCDLr2u+cKre7ksh7cxFREQwB3L4Pla31sC1H6Tk1PpvJ/3fJCP863u53iqPyRbuJ/zPQPQWwR9oB+wHZTV3j+3htHkkqqmks8hig8kEr3//p7cvgCEEHT99ypHPNcqrnxZXWn+HeRe893GGwcvB15uru5zPp3IqfnFXxjdxs1BHIHTclBxZm0/wdjgnVf/3mKGyoR43KJUYXOEdNt+TJ/yrYfNu7fL/Of0gef4LMfsvGG50Rsv7duAfdOZUvHSno9EA/loIS4+INkt0urupb9KncicUxse+vW9e/XD6/NmbP+0zXVxlgCKxfVEWQYF1e1KRnoSVcRqiWwxr/QzPzNOZC0zezzBw6AWb7QGvIthm0D+tFTkzfyDcZosNnks+poaDh3V3Fp/vgsEVCYB48kSYYyK4PiHgtWerrgMrdVK0QHxZfsQYxuMd8nacCvoHIy+CBmljsNS27LF8rFc8ps8mAF4uk2LR7+riOZ75Rb/eEfLQb7bjHuYHnMjDDukjcfrq3Yl1vbmSN4P8cm2vytI9RbZNuMAUfIClWpxsc9IrNKQuUIymkFfaceZiTpJqi+mzILegLjCF7IfTV1hxIk0Lu0MZ8XSZykHHeBLJol2faly4r2g1aQ9TY/eF8zDN3u7iwYLx1UH5aoD95QXol9zywcPTLR3cOqkvXk7m48FkvgTVqms2eVizacX1rgZcUam+1JLCeeKKG5iJeLMGpXsqT790ME4pTznJHvr03j173Va7mzSnOmfwrPz4BW+m0L4xE5WMo9yUVZsst0/lo+rxrvwqjpgjvhtuZGgRjPxsNYCG531hOcBh5j94hPxWY7h0iyoA//4K1JKKojiI0QCdraj1OUyX6eBLYZWofn2Mv9rG1+sL/E+D1bMPMIdAV2CrxBqMKOLL+rIkScu7/EchSfU2x5hp6q37xt437zUF+uH0fQjxflFZmVeycm+1CvFyNHcHozn6JKbsmIdlxzx+1ZQk44MpK01ZmV2CogS1SlXItlSCgvhj7v3uzcnJ805npVgDZVJZjEcVlZthd+6rpZd8zK88xrdRN+e4TYFVNT/E+XFbAXOIhdvGiQX/MAvwS7j9gcD6IP4R7mjARKYbD2v8dULD6mLSj9fAPPiyKMHp1xEe9QhcNESeHNYhdBiDHBxo9uoZnyeiUDcbLI0IbnopYiUJO9ZwfyMJH4m3WLFhE2NdvrwxhytdvgFcdZpMACETUIeUElb9BC3bQancS2RSDeY35tOh2StJyG/3xYCGY/XiJOqhJBKEfROC50y5Tw2OoQyLFAGDcRqPqlfxPEXr9bioz6KG1hcGlyaXPJC1AOg/Lj8dHdE23ZWF5Tn6mGJOy1f65HFZD53xobXXR6oK1+NiulgRbh3CF3qvBy/CZ3q9V73PP+svn6BHXVyvLN+hyyytxFmZWgIIcVGPvmifXm09ARGA3uZ+4CwiVzlqPhv7TjDz5+PFbDGbL8ZSzp3A9+eOA5+W0XTmjqQz8d2ZCiRWzYimvquAsw70XTtIt/OM+3fn0XQBnQZTOfOlAkx8P1r4s9koGEUjXynpLEb+NJq64zB0xhIQdcLl0p+6rhxPxr7p/7yvKUKVmXYoQnVEEIWdag67DxkxP1DzhT+S/iJazhfOeBy6EggVREAgXwaLQMH/bjCfT+UEa/EEjrt0Z6HjzAJ/7C/nO4RpomDO3DcfcNejmRpNln7kT0fz6dyNlmoSzJbjsVpEUTh1Jm6IyMCcjNxpCEj48MCfuVM1VdFMurMdmlBVrr1c0jzp03qiZykYyYULU+TKyTIay9F4ifV0Fk4AX52FG/mRCiLljOZOhGWDJjCr86ULXDIdAa8s9nJJ+9DFzjPuXy6CaLp0nWg6ipzRxAF6zOcw3AC4YxnMJ3ISTkI19+G5dBfL2SSQi0kYjJfBYjRbjKeH+zcp+s0Hmg1COQ4XWIRlMgrVYgmwJ74Mpz6YE9cJHbWQgM7IQelZ+ouFDIANgE1lJCNAY6dnzKAr+zFZhM0H3HEwdoDNJ44EmvpB5MwcNwyDGUhr6IJ0zqeRWoyxLMtcyeXM8cPRxFXzMUySDECQugSDipc1mYDq4TAPNlKS2o80TjKcTNwlyN7cmYaRXExD31GT0HdHS5iMUThxR/4C5DIAbJ1QTmfOcgFUCcbjCBg2bBOj2XtLJOt5F9j3zHGA3tPJbD6eyrkE8ZwtpDtX0cSHxovxYjyRwAtKLWW4DKQbOKMZYKQmi5EzBv4/2Hdjd7DjKeOwHM8nkeP685kbRXIKkj8CHTgayekC/kSzmTtZTpZAGSwqJecjkA3fdacT6YP0gLgcxqHcRGk94b6jpR+6KpiCOnQC6YeRM1mG08kcWH0UKN+B/pwZsOp4GoXQcOqMFsA6I9cfjcfT8Uju8gPW1GvzwzdvXlPv7eDY7kNNkXDphlMQtmk4mk1HITBeGI3UmCqCjUEIlmMfiO+DqpLw62IMwjqagLICzTqTk12sqJZXG6t6XSSyI7X1Y+O7JtViMokifzZdhCNnETgqCEFCsQpT6IdOOA9nYO1CkCp3GkgFTUPQKgq09jxypsBQXUjN9yHFVZU0FnUXvf2IcZtN3NlkClrSGfmg2MdAjAkws5STsRvOlqORM/PHSwXCNfKBiX1nHiz8ZRRMx3ICWmBeww08AqpIBzzzHboEyUrkaQaLJuvj1Rk7G+e0MrzCdeFV/smmb+CNXVERD+1NDGPwbHLL/nTEFRMBGoFFkJ+O8A0vweb0I3s09DDChyUCZ15yXiW4NiFhPRSd72d5UV+cndtDucGqJpaX2Ee/zALuSq3BfcvZawOXyvK1ewZeeZxj8AKL4Fs+FeSt5er6QAd/qJIgDZXV2xbRAMS47tfpKp5DdtMA7PBKfQjjS/BzLdu4iRtZXOmqtJlaNx3DNB/iz0OsHG1B51zWtS++hJbDfAM+rdV70rNLWHiS08I3NBiqYpECOekh8FgGyh/XIlE1Ct0VpubDyyUoKmTk8aUK+m1Mw9SAi5sNhh3gMazTqxqgrU6hFfR5u9tnNCTwFkGsPY3W2/yqdgQAxh/ld0lgwU/AtElq2fxjkd2t6q2Cq5s05P7gGxaqZTLARz658d+Ek87hD7+uPgRqU4hXb6lIe+3Urcw5MxuAmNwiAkrAukiDBWJ2ydPr9f6FCkFd4MMLTD+QTC06cksFkjFmq2DBUmC9fM4v5qKqNmbX/UNQGfjfcCBhmtd5qyQTFjtGJCqAVDPcUNfq6SLHKNdUajZIk2CbZSop1ncr8UXeE19oAu8nvUanxuG49MWRre7pVselsWwRFtyvd6fnU18UwttCFlb9172Wk0lxd+j+An+8MDeL5EhvKfiKlNraXVHlf6y0nVxWc1nnWA2g1hfMIOYx9vpYoL/w4gSWqLEu/d0W1fLImGbjE/oHQ22Aj1pR4OEuKeQHIkRf/BF3N/k6AivBAwRUQx7UyHA4bMPufZEbGlmYBmkpe+h5WHnS82CIJc3wYhS8LsVqEuq5Witg+/pNKr/NwYh8PaQLVQRWW0NpwHtEri7wDFOS6hsKEKlA50/iCZuScsxhuOA+O6cHsCz28Z6J45I7WRVWnIJdtn+Gni18zT5zGAynAR03lSwuuWvY9zrUDRdzpvfwkAhWqKbWh1WLpq8eTGkryX4SxFW9fk4ypDJUOUq5RaP5SvSGPS6Ck4DNCfmnHlK11zo/0sB2V1Qb4yXM+yKx7Y5jjYSqscdJs8W+ce6o0tbANf/o2vxs0Fa7RC4+YNSIxb1uInvmTSz5a9uYUFqzvmjhWHf0KmxvaPMHPA2shWNlML0ahOf9Of/yGP7X3/W3s8d/7p1bZ/8b//nKpm8AGBCyuZa+gdQ8v9QBdj+gNlfcDGn3xHJtnPyb1snQe1iK2pUOQJ6uYY6Jo1r6a/C1rnZNNxwQ0z0Vj2Xff4zjOnssH/fFY//xOTZExhYWX0BxDZ2DWtqCEAVU0VBfilTZKWRYEoqyGlMLRQRnkeNndzhW1W0H9q6UnX0gOfnACc9a0rPeWf/P+flXPV3Jn8j2gcWadxC0qtgnYhRBBX8qQQ8hizc1a4iDYTEzedYAscnfuNvQlIpqG6JUkGrjEf3ac/Hc3OOA4UgO81pcyFSfVaIFnq2LAd6iSeNy5frWB31DhSYQZkmX88A/wOCTauD8Lg3KkKFFpyNzsOA9zEkBI2pqok4K8HgrnwJ6C7GhKYqOB+YsWE6AJ7Fqn7WjhoTqrtZgFMKab9IgdXNOWzAM+q0JAc1Zp4dm1KiaXqLQeRm1Bfp6dNOBts74oJq9twWdlIjosAT+VN6ppX0Di2TtmEszU0WI5LImKUmUUsF4HfBFCGZBVgV4vfQafqOlCDfAgPA5FUQrv5tA8PmnUgRpOFF6VgdTW3nhb0M+aIJFDottftwDBG8wc6aHiw+Zp8lxjwZFN0HhNvl2QxdQVR2ao2t2b0eLYQcGGQKwow4eiEKddge6STwOraMPjoF96HJIlW0aVOvv/oAx9XP7qI0RDpKaHRvI6rZ8AJ9L3UU/42y4rCgYATzl0xjpWY/H2MODOD0q0x1gth6zAmWvlJCcJiT3XkgqNGBydahxSd5WG6Y1t6EiOAH45Ln4IhRFTOoYMWk9BPcRj5ubiz6Q0TkJ5IZuMLHJa6zPSkMWad60lK1TGXo4L7m263y5WClmsL5FM2Rl8tbc78eWPqTZJEtsnPJtgl4CErYUM9ZtHz+V+gLhgahTLzUno+0oYXddXgcuzO22CTiDp0g+RLKPr7ackdob2oJb2oT3xQ9JjE2f0wv0zF7tg1/6AbtGZsPVn0hboWJjXq/RkS70wJ/Bq6/ddyT+KqpPhkPg4wZ8zhjTDb/CKxgHpApCTMlZVyqMwTVNbNznSJJKtjd055ylI0hniNV5bWyFFtScLEQZjKpIhZAbSvjMPSeFUjlF0Fmt/VmPdu+Ik+PqMWFpTIc+O34NLRH+x7wmI4g/Bb34lVKd6saorGui+2nV7ALFp0bYmmi334cfu17efbESWTJyCKbjvaZcN7VA2UjPZ6/OPvRjX4+2ZKStDyPUJKe/m4Eohoipx4UubmSVmPebFLB5onrmSq2e6YPrXKTZTc2wglzDk1uZhRXPPsO7dYRlnqOob5Mwrd0qauzvhssJJDoB9imrhK+0HaFNdW6vU+T4esyKk9WP4lhf+ARtdYeMPnmsdL8PLEUtvozGLvn9ig93/VgjeRaAiqJNd+RqMj0wRGNrduBb5idqxJaq7nbVbBaApmCHi5SoPQ+xQMFvGlantSqAmaHhfxHWhi8s+PoEnrG6pqENSYSsq6axxI777R4rLMnEa7NrIjaGDJjhU+cfnvH6SoXuR+PYTKVkSY0a3qw44gUeppe3F6XlsdqXYH1Vuz9V5yWyo1cGmkqFR0eBYr5qy/7vhhfMr03nuRWsRpVFWJ7xjGolQ74ueWelTGgXmyrw/h0gtYe6X9bONZEj6KXzMhrTgd10Pw2A8ja8Uu2hCTzot5kbuo6p6u9nO20YU0SyfB5wlnRWPkaxmDhVr9/jWAbBtQ92Xbtx7B4EyPvGWt84r0DgEhfjdrQbaLWHPXRd00Z2bj9uj1r3GXOCkA5h4nKZvOnavb2cKkJ9mluTC7yC1d/Ga75Q8ogvmKNAWelD4EVvxzubDiA/jbvCduMxfukXAYSGptI7JpoKTV2ke/fbntHR5ztFZTyJysjkqoNJ8WBMYZUX0h0f0/cDAsfhtMZE7rsXp5Sj1nK2i3U4j9ewLt59aC6gAgKpDOtO4O0+NW7YrfT7EKFI0hrTMKfUWKBxDR/xFHHFAR7UCh2p21xo70jMFfl/V5UtLN098tWqPUZiivV5w2b0MUCO7LTjEIA30Ax4711IPpREpTyU0X8T3E4IjZZX36ZIem0MXGuXjwYmb4/24pFe7ygMAMYqolH8pZzBr0obRGas17V2+iV2Q5+9fkEWes+dq6VRfg0LBLqRirUyskI7vIgRRHFBL1/Y/fpV8ZhKCqJ3e6XIRNbu8oSprl8kachDvdZUn9ist3nT6GMlAkNC6nMITKPvgy99PhNWbEVK94bPauGiis2N1d2QLUUSgSGtbc+WN8+uWoHu+k205pLYxhe+MVY/MLfG0tc6n7YvkV3powM9ihvhZ73ypU2q43LvP2rFzuhra7M+Of9ka4tEdOZOAirbo0ul6lulOQ5Wu0CxvJpMautkrp3HTvX15qANPcbezIRBroYN433IZeJUgU+wZEUVT4MxO8O4uMQJ3gkxdFChjc0eetjNhW8trtAFpb4O67OqbC7UzSq9tVhF5YlR6Z3oa0esFHjvrOSPc37zI61pmRW154+8c6WXtI1XmQv1i+3Vn33v7pRxTLU8rO7poKdj1r3PGIN5hxd5vdqAzs7vjes0ggyl91tFePBWnVZQoZqMK4k0KZ2Cyic+ULiTxsLSfG7iNngxb91W0fB4Hx1G0YIJ4ysze+EDSr/WfJ7jTb2Ry8+bLkqXn79bKhI9EH2FbXgA2B5/pwMe+xpeOZRWjsB9zqX9qdPzfsiK0N4lOGvMkuil6d4Jipfa8ryDXVsK9bxxgwvtbmAAcpM39jh0dlTTA63Wb2FzSUgGTyZ3Vq3JB1rAV03KnS7sy97Bs54t1aMsA7xSvJYz1fuiMoG4P4xxUO3v4Ma2WSgRdNs4GDB83CjAyAVwivX+nugO4vied+MqOR/SCWoghq2PJ3bQ9OhAIaQNkcOMybabO8cbHTQ4dAH1sWU3L0wvHRX8BlMTiQt47UJYXHeu9G5sgff98q3xZjcxv8KiksYPuWj2dFHdWc3Gj5f5pK1KH6N5tzVazPK+4I2MwfnLwaLmdHobqWAx5nbdEzKUGwIrxutBsE6x7ihmeSR3vOBDPdSdYAIYC738MY4ORbRal3bbdltW8MXf8pu8X9sZXNq53Qj5mvb18H0zrQiEWA0h4g/3CNJD+jJeA0wlswwzIEUJP35qbEi2UxPrIyX3UAdZfnNcY/fS6CK+5Yq+Yv+OSp17SMIzWhKjRLglFg8ZtH7op+naKuGAzNp2ZxrQzva7vp+JZIhuLq+79XukhqOw2nnXKon3re94J3oo3qhoiyUZuTboNoFx0ZfZnMogaD1EVpoqdSIra6pYWo3ZK+Z45mD0H+ENjiRhjpzO3eoI97FvwHFZvXHOS4ry4ufycngujXzD1xjgKowuneHMp2ewOE9hHaMjKPrSAHNTxkqzNks0riFzWtKwA0wD2aR5MaAe7H5deHHZUagEywRTtJGKg2JBU14ygou6ydSAnvLaHI0qmwizlg/o8oawCYLqnnJejjjlu8Ew641dV67PRiee1iBRD1r0/G2rmHJG8WwSc1GPuIL8M2qAp63LtU25MGmuY5iI8BOxttaBLMXG/Tu+Z/e+sSSj3cCWu4zGxWzwXpv4ABkaDA8gB4cr0XA2qRcL+KoVX9SN2l4IzT7Zz5pR2c372nD8jXM8dtvuc7c/M7VCU4XphOzEHVaGWnsmdbejS6c9gGZfcAneGM/AwWcKvN3xbnCRXtLC/qm+vw7dkjVGD0zoJdyXctagJ+O+T+MaknKW0vFuyk3NvyHWRJD06b7FMUM8tDgu9XstQipp9wBXJO2ViTGuZi5qfmsVj/z4yWbFXpmG87o9Afi7DuG9k3QoHM0xlJompTm1jDLUKszu7bsaC0VDaNkA7HZiaffNZWseNyQXn79pD+8Z7/++nftDSYs1MOXSTqtgz5fXeUdSGfFJXdOQkiU58ACEsbItfWFUo17ywtcqL7je31dNbuozM2bacdGrTeYb7UDRLiP/qtdoGmCLlQBGbQO8a5PJXAq1x++pLxqarkr5KzEip5fUUoqTfrPzJsv40Ol1uRFV9kEXP7akDe3uHo2rhQ1bMICHiw1Zc9Zj7DU0s8axpkHRXKmge8NTQZ1WsmvXt5zKsdSYBQtfh9Qzs1bfsNdh9qfBd2zh3J8BDLiUPN6uJ6Pz6m1dsIQadV+70DoY0i+bd1+Gx8M069MdRAlilUtud980SHRoyAbifN9gduIOuBw/PLgqSXr37QPZz5x3vwMRp7a2Mi+5iIRZtd0IppSHgktiw9/rXI3wOuQQG+JbnuZJfBmbHj3AQ9lZYu9qLzClWvo50n4sGrk9DVtas5Tlb/sDspzzVcVW+1WIdW9m01GlaD8aLbIqI2gFXzqDloZ8WQJaBg4ZeiPs/umoHrdpxGsaer2VgNfSjrvOXt6E+5tWltAB2KXl1s78bv4evVJP30PWqiXH0EqHM7qqTD2AZOF+oU7wUmFc0J13NzJkr4wQ7RqI3gz4hvbRuBt8W2fiVdk3uvDckak0aOb9XoNeNibB3LM7yD+euagma/uOZfLOzpZjx1YiAG6Lz2cTf+8ENGQ7y+xuTye/hxZ67M0TSW0C7fdvPm+Quhdt5B40xg7v8EGjNkxEXkmZolM6pWkWX8YJJh7iYQPaLEVdrsWM48q8XSoToWQGQpTxWs0e7ga/DhvPz8iAaHkUrTg3JkLsKv3d04HQTmsv5mHnvGvhk5f7B/Uor1kZnLUduvP61VUHrdH++ddxhvb0t6ceQwJdC9pyVumoI5JxgPPV11GbpzpUSfC3dJ8vVcKsR0mID+JcL7uLtAyM8AOKjPAEm3hK6YR/xozf6y7hMUJo05SzzoO9e72g5hnLGsZ7XR8K+ByXnleb4ejnnSOOzaOOb7YJ5mXr844ooXoOdEyJEw1YcKqAFPyCQruDUKfuoXtbW9k74PPtoLujdg5haRQP3/5MpGx72qVGsQ+cdGw50xVztHhCj/0Bp+Sa89pmmvvF+LC3++Azc/fzdOU077cG+yW/TnKgNVMPp9q+x804dDK0+eJhb7atRPYP9qDwRrx+Pqg6q8ZneovVeIjnZp93z6xWLxCjVu2NzDaawlx4OjTLLdvLml03su4+1ilizvKBoiz+K7viD3SZa1ttD3PFfzk33FflPshP4Yv/w/jWFDb5x3Su/5nd6H9OT6iq62D/Mh7PPd7EL+05/FdxCx64Bj7IpHVrb8L1H1t2fdVl1csEK23MVy1T3j86eKlTw7yvGsb909EeqqL53mve+ecD+bU/4Jkns/9uThVTlorNxV8KusSGTiiZs1Rmj77QJ8urPXpKDMce9Qb97RWqLN65B6zv8vopmKF4pnexr9LcnKSRmaLuzbjYwKkf8UphXLYNlTvBtbVeVWJBm7Lqg11uzbf2oY50xh3My9NmXgulECS6lkqpLnEbfJ36vsrq9Qf+8/e3a4m8jQ1uHZLVL7XSc3/KLe4z09V5GXH+e/a0O/d2fuLd7gel1DUy4nb2uz8097ofvv19fN/2dwc9SzXJEoDsCJb8Nt2uQ7FW8r0Stew72u5+amQOb/wgDfkBxlxtDTX6WX3+DLT2VPfsizePMu7sktv37ZMf3u3cXaU8fLtz/1ZkqTz7Tfe+ax9yZ6IO7Tcychx42GXLKsU7WXGjJuckdiuzbMf/7sgmOJhavmpn0TUPALY3Dh50jrBFpCAFnynZqr227RF0zDmM+jwHXgCpE7Ors5Irca1Al6KqZxX/8CX6RxO9rAwo8fWnh53Eb0WbymV5tSDHRO12m3b+NDZuvPqgE5PNqPHmvjDxg0LEpb+LP+uTRPftRzZO3bVG2iJuIzS8c1FG9y7tXjlqnFXi6GzzxBmLktZz1lpFeM5YZ/LZnXFGHU3vnCLKheX8PJMLQya6dTbvjydv3r54dWo/VHXjSHT5Er1ZvDcnpj6eHfTv3fa9VyF17/22HNroYVu3lZUocwzKLeAyC1sD4Kbde746MO/xjOb1mlxcQgIvctETnKkg5aRnvC991Qrq/xYH8rXxBLcJGqALBHdBq7AB32IV4A3czK31U6544zrpAj5ehPlJd0GtCtxeP07jEBo366DTtZvKg3fU9MGjB4e5LyLtflHtTbyQoSq72fJKIlLDUStdTzN4VKvR1kwj2GVZxMiwSqM2GyMWJbXCA0jWysrgm/XaGjIDoaETnNdnKzHAC3WanZ/X2G5dK30H30h/M4R+RV27PNmPZTUVFpp8UgsA7KhtHT/oOCm8u0Y2ne0GDEoFyU3ujQqUJR8NJ3REA5jJWsGLA/rV+C7MWaUsl5WbSJCrMztkhmgk/f1rRT2cT3+vQmkhwalWK71R98l+QKrW56SFgHE8fU5F6PGISC++2QD3Cf4Hq9tuwV/oi7/kaYLFV/+ctPMFeyhvzebDKE5CD29GsR7nl2uZXD62O15cp0FuHFhgv63PpTY9LsDn0aXAYB1yql+n6yfw4vHsvAMc6PuksBDRYbi9AUX38TFvPD8G2g35Yw0GR7Ifl71AK0ToE9590zOFMJFmqDqs95u7PlVJSrfFsTtyaudPQJ9d8CgvyqR0eIUufbmjk6t4wdMGiI7qEgDBer+klyhJhYKPv4rBiwsuD2CAgV1WayyppwPEmHLLAHCVLN5whng9doYnbxMsKhDIhDNTQb2Avt5TrpQOk2x9fa/eMNsm1hlyG2CD/D54gX76AKuXVswCujeQG7wTwwOSbLaFPhePYRnzUZNL/7unLECt47flRy4P0N4G7IgRmglamaEC8qJmkGnejEFexxRlsYLNMC9CPD2F6X29skgglx/EVrBIPRu4q3NivV6vIxKL25HEa7jZklv4Uhn502OsCrM+dAxZcAyLTMbt+AssMgsmNoNPNBZAmyUXw7VAfjOKs8HYcVbn5gm8oZ+YYbOchexzlQzP2Y1n5/VMRmpKa219CoWkpr4u08DOjG0BvCn2EJ6Zxuf2Lkg+IlwapDhHmcJfkGYPpQ4LGXuXyM14dAKD7V9kdjXXYfN0WWUCJdtA06kW8BsJlhj09vt6eE6iQMDDjYQ19fBZdrm9AcfmNX7LrFDlAXALkvC4dzKaiMutzLBy2dvfvUT0tLdpTr7qFelmKMPQkxqU1QtucPEbXKV49+axpZV1n8MWd1zplHzUygEyDpxJPt2Bie2oZGZ2mR/3voSPWPr8mKqE4W7QSvx2c32JtPxa0LYhVgg+34fhYEC+mAGiiV9TcML6j3/79yfYaKCf5k+4lb0fJsxRCVKH8wT3xHac1qN1hWmVCvIAVF7EG7h4f5/UNU71dvFKHwBEJ4tI/YTJW8LECd8MacIRcs7n6fK7HG90en+GqgBX8PDZVKtgK4JPSo6jnGV6UQ7xm446xFStSw7pA4Y7S6sih6hiCTJ+0rWj8NXhJt1Yjq1jKLWDzM3MY4RUiQ+MQLHQMEn7ggiOfcryEjm5rrFBXB7yCDUldoKtcljVYNWfa3iaSn0lVnIIrE0BOGbpRtyr4YuY4q5VabTqXZaCVTvpu8Mbr8bMid1cw8CwhN3qfed4XGfvWvTaqO/G9O2O6mymbceiS5+DaLkpGfrdMcX1j92m9nJ0GLE8EqjTHnkCXNBh6Evr7RhC3fNQo3meRh4ZWH2IC4v0HAD/f1otRFmAwgAA'), 'hook': ('e24_hook.py', '...'), 'unit': ('e24_unit_41736_41737.py', '86cb257c1da9648013aad17e3fa3e944b19dca03ebc9138c948e7e5d869607ef', 'H4sIAAAAAAACE81963bbRpLwfz5FL7L+AjgkdLGtZOkwZxVJjjWjyD6SnNk5HB4IJJsSIhKAAVCXUbRP8b3Dvsc+wPdMX126G40LKTrZnV37HBGXvlRXV9etqwuO4xztvha5nBRREos9cZnvBcs4Ki774qePnwReikLmRS6Smfjq9c63r/ZEGE/58lsBdT78cnR2sv9XMUnSSFKx4lqKKM6LcD6XU5FfzcP4SriZvBW7XbG7vbvX29nuvdr2/E7n8Oj8+KdTQe/Ef/7Ha39PKAC64sdPxyeHwfnHowP1/i8fX3XFOJzcpEkGAP3nf+z5r93QE1t0+QYuu1gykndbakB7wXWS3PgLgHdPfLXzxu9cAGy3Mr4VUS5ieSszkYbF5FpOfXGZZjINM3kp3LyYzqNxV5WIFtwhj8TTI82jQvZSACe8kvmWGmaRdL7PkqT4YSuBqvPwIXiMk1h2CXP099snVRbGkkXxVS5kOLkWBx8+/hVqiwg7KsJCiruouCZc5uFCKjAz4crd1wE3EPCzvN/JEM6CCgNwD9hOmkV5EUHPIkzT+QO9A0BEDI11uYcgufFoMjM5SbIpAAKtPIj8OvTFEcKEE48VY5Et4xzmVIRilsn8unP5PSExfSiuk/gH0Tu+hA6Ticxz4WJPcVLIMaBeJHdQEZ/o15MkLrJk3hdFtJDJshDfiJtoPk+v3kIxmJR8kkVpIcIpwBMKoI79k7OfOzOgJZx5r8SKwq8IC5E/5D7g4nq4PRLux79evP9w+nH/4j3OcXQVJxlQ4TKeAvJ6x0AiiGEAS1xkUQH0uyWO4+lyUiQZXJ7/dIKzuCUOPh3udyaABCmmUQajymUxlbNwOS/EWM6gTUByBjiCamrqsTeiFDnti0u+VO8uxV2ynE+FbgHgXyiQ/n3Lp240UXTurqOJmncm4t4ONJ3MQ1qhUHJyI+544mEw9vTRulMQiEUyXc6lCIJZNJdBAGiGed19s+eLff1Ol+3MsmQhYC7yaCoruIUhzcJo3pvMkxzadLMwHrwL57mEtdsTRNN9PbuIQ4QQJ0IcvpuH+fVPWbJM5fQgiW/9ACb+NpnfSiAQRJw/SRYpgPb1VARXXI6KdMUizG4CJNBoIkKiOkZGBkB5/Y4Qp+Epov2b43iGbzP5eYmEuv11Lk72zy/EeJ4AjrLkTiyW8BxoEcogPeuSO1ByBrNaYCFffIoVD+hD2bg3i4D9AJiT6zCKvbfQoXltXs1lGHu++Mue6myvK4owzYFJuckyE9MsnBUy+5rI/Ta4kVks50Ee/V3ClI0TQBCiOofr2Q408aAJQ2P1275YpnkBUC9oEQKGpJw+4IK/DhY8+QFgJcjkTGYynsBajoM4jIMcSEHmQ1guvy2iezmFBTGP5hE8zQLgIynQAk0QTBSSuOj9AOPL5VziAgjuwvlNUNC66IrbXAS1N0zxUPXjJ0+k82UuDoAAoykwk3NV0rd7gaUDD+cPFRwD3NAnQNjDiaRpinI1KOFGsCruxWAAyysNbvriM5WA0uLvMksAZcRe5D0srXlSIIY1BDAfH3UfgN0Y6Cec+50fEd3IprFmlCGdp0tkAbF4d3x6fHEENIQP3opLJElbAHhAMcz6F+G9+O3//d/fYO0Xd1IySRZ3CciSLAeqz/3OwckxEqcSIqLXQ/kgiE/+sNUqK3o9lBRiq1ikW8DVxbDX01we2dcIHxCjzkVdhoygMjLPP0GPvR7xaXr7G4tleKlWcE0YfU/t/aBrt3ZJ7IhBOzw+o0eaV5+PoL+Wf1BkWjykkuj5t1lKf17t8giAcsXpSLT/Q5GkcNjlpQ48xwU+rQXGygFq+CuAEs71TWNk5TCgZzdOSjx5AkkbeskAPf32QeqZxbkAdvo9AvVDF0UjL2ngVZcoCCfX0XwKi7IrFD3BIhOPwDrhAbIU+EH2wUJ4CRQ9lQUw2fzpbXu/NEAtOf4Z1LXgE5BtcPbhw0VXSGDG/PAvH87+vMWiBPWnQI0sx4EWtABgHqIYVAOk3Sy8A46VedClgbK31ItU/CmgUWoc4gMe70rK6ZwBHv50/uG0Lx4dLOp0hQODph8Z5gldURVi7hJv5X0K/AGvCDOOcEsQ+B1LvWRM7H/qYUlEHpRU5TqryulrbF1LOkc8suxDVj5HPYhkIlIeMmSNsyeowqweK+slTgDHt84TrHWFshXj5cEYWOGCp9oBVJ2fi9/oJwBRFbzbPzn5cf/gz/Ds3f7xSQCoPHh/dBicfrgIDk6O9k/Fb53j04MPpwcnn86PfzmiF0f/dnR2cHx+dAi1Gi/PPp3S41/2T44PA6WcB+cX+8DoyucXR9D5z8fnP2N/PC/XMMCOA4wuCMd5MJXzIgxKJCgahcE3ge/TIxBmQBghSTYgr7iiSBTIsME0qIh6YKaTcJnLDpas6CSkdaDtUABzD+dIiu6dDEGO9kmcWxWS8a8w/ZaKAGDA+stBQzm6B+NlkgA42+IOtFhS5rEgThvoULm4A1FXADcH4yScEtBbApTWLbU0gYx2Beh6YpkD3xYyy5IM1ivoY6B4YNOo0yE8Y6mb8juO43TUKMLsCsgEBmjus/DB3OSFvhyHudx7re+uQXMC4aNvf4Wlo69BSl7r6yTXV/n1sojm5g5U3rC8W441I9VPHswl8kNznYUTiSq2eQD8PO90zo5+EQMgfOAwZAySKTYWrmXHOR0kp3Mo5jrEpJFgiE87XudgHwg6+GX/jN+Dgr1/+lPAT4FRY1H17E/A1CrPL86OLz6c1p59ODt4f3x6+OkAruxXNd7pvANVEAq+Ozojznj+cf/gKPhx//wIW0HVR1VG5kUNH58cNUGj7mDBXRydnh9/OD2n517n5w+Hn06OcESPasR9GBuLdVrudOXnWeEDmQKP9qczVIgdrwKnQlJrXdYZcz9Jcz8HDrdEA+BW6oaeL6jVPsd76mi28g4G2Qp0C6jYQ1Zs8cMtfuinD7UhtI9jA+hViS0osWWVsDt66nQ6E7gDPfSCJbd7dD+RKbJ6j4V0Cm+hFEhH0FWvQzJ23FS9LLKHUpST3ZikMnZTZNRjx0OBPavK+kwWyyzWK9Bn6eDOfGQPruf51/J+GoEGV7ge1ZMEjvhwfoSsoWxLtXMKjEtDh/xBBriYUXJdd5FxKTinMCUJW7A+aMxoobv6Hngx/lIdjzuFVwtghWiWulPQAO7B0AdTfnCRLSWXAIUSmiQN/Bvh+HDrdKo4gEeAhTuYCDAekmkUXw2cZTHrfddAC0LsT5eL1AWAu2KGshKV68FOVyslA1CbDGigJM2Bk3APBLXGwDwJp8plkdF4BogfhQPgmuia0SoFKfikqqAJCpMKdlUmLrHWpYdjBoIptD6jnBJCi3ZQ7Ay/BGUsltlDFx8CUU18ZM9EORpD0K5G9q9JFLvPzYS2qD1UMprOGKJc6mEBPRAT9X8mwC7g2m2pQXxV1/GNwQ7gdRqE+9xsIWJcZVkbuoV5IF1LTmDxUhfTaFLACDoWrS6gFMzUV6Dg/Tf/M4q0trKIPNRDFzX4Lum8XU0NRCfKZ5UPXAdVU6frAW2T3Ckp6IPSNkpX5CU2d6nkexLPQdEPixVGkXHLka8A3kWzCEhszL4zBYuhn2IbpxfYko9/XI3LlNhrDkUXITBEIzcDNb6tHVaIb+EliFZksqhh9UWdzPAxUhiC2vIaH9eEiVb4ofQjarDJDVwhU3iiYh8Bn0Twg+ZSxF8zgqGjnjgjGgzSI7SUMttymCnCA4vh0jrnbibXN1Dro0+OKh5F2TDeBriqcmfUtR5hK3mgUU79QjtDxxSlu3opapcdfFCBtSCf73FwBUiY3HWCAJY9mkVBgKh/iVyALhwgoI8+KrCfPgbnn969O/638vHFz/oZg49aINigYKEyGZZLbgrcl0y0QZWPMAnrWQmw2bzw7Gr1GrolI9udsnhCUwE8CRBvlVMuBX6o7Ay8K28CIgNy2j2Z1iqSkbA4w9H9Eyh6tLjIrahcZzBkWmZNqzgLIzA+fwnnS0nyz3WW8U2MrlWt9rMj+0XmiBeVwasu9ehJhOVm/F6zKzW52aLIpCwLVspV5GJrCdUKusipHeY1hDkmmwH/AK95WMyj+Ca3hKoZ9S3RN7vcsSNPrP73FfEO8u0j9+kbtzwYJIAaNEgYR4W1LxHF1XENHe4MpxKXRnY7xMU9WjeDTRSGKcFNuwEujXmYFyOvUQ66ozLUW7e8JeuH+g9T7h+3FvwrWYBQo3fVtiYWG5jWJz8xaxuXurmpl6EVz2wok/M+llSPhnA/olUJF7QsycB3H21Vd1iMhjtcqqTjJ++p3o1ZKCOCWd+6k65wPYXZgVkb5HIB2dX1PEvukiJo1FMUyzU/EvRTotB5kffFixzXhYs6gis9EMuIhSAAAeRZmiQiW3P2EU4Zso3ae0UX5hqXb2VcFgsGLYAKJ8t46loCTPRApoHBW9EKoMo/SClAN5prbSqxVhAgg3M1RwhR5Proiphcu54/AzEGmrg/SZf4N4mL6GqZLHO4wU1At7fj+UUyj4y6rgb1iDIsRTZJ7wqfblHYkhMTxTY1/WoX5cR47zU8YRPdhxtSwaRLtrxPf11nBgVD7Gz8ALIBLYWppFJkw5iBBKCumsFsDgzUAhjQE4WsFuX7EBu69Yiyb8ltWmKFsdFAwMhAop1hoBTlXdoPhB/tBzW61BltKjX3j1CrIgWJte23pGEQc8OFwjuI6K8BJRRUMNpZYuea7Q8yWlRNcIKaNif1xrhlv8F3uWRVOEHVAvszIjnGwdMQysWGejfuCDKAOXGouFxRM3gNj0A/yFxQeh09QkAw2SM4jAVtDwNo+IQXPNlyhtnyiAZinCRzd8a7qI1BzDxkJVmRowrvVsURbtKUbsnRMK7oWrN2PWtGwM1KgNC1WfFcQnF+UPI4Qhr8QRD5nU1+yY0mC9qpypNlBsYb6XMVw0y1/zVvnVbdeNrgmDa362D+GluBXSyGeFUePPQNGoKwrB1SN58zeMixOBDa1KHnubxCZMY+EkYfHW1IA2psAbxdSFyHUBPI3yvpCAuSv47eef44mT50qiI2yimwIQYcxSAcsMa7ZUwbn4dy1qUmDtBlAXcekwWDge2DLmrjDVd0AzmOkk5qepx/rWyWutMHaCyasFryt5iUSnk1rDU8QrP/bzH+NyWaPalSToUAyN/CO7SooLdxhkZLvMptd6+aWdfaFmZa93xxjAEiYIRZnt7ccvR27Ya+znndA8m1e5BbyEvY1Gi5hacJbi9W0CkUOsmnQSEAwHfzRO1QCrQB0VIG6KL53OZoFRDJxfxW7cJzt3rvwVcGlwN0EegYBqe0IW01vBIw0PTHYbOHWqmMk89hXxy93t4Vwm1gvxoRUfFUrVJQLHYUbKql5MBApTfsv9nettRINeKqS4UBK+0a9i3iD4+RdhhKx59pLM6JK9KkAWtTZOm809d+HPsztf7CObyJY+vVU7tbpMnp2FFl+UjivIShJi9LV61Xro2VbH21e7VqW682rRm+JptHDtDtiN/3j9COqkWFLp/q/ss4b+Ub3Vq9jmKOKDBr2FKO8qFyOI/q7KQ0I8+WMeqhypA0YSwtUSlORZk79FsEjMMMyLA2iqEgENpYG5iASN1A0VJOFWCKGeZdTdDPsUf25BY0q7gLbuhwTLrkzh7uKabWc/sxaJqVx692n4YEEyvuU4k2J7+H6wjo1pksp6HCxPuu+KnLgSjwk6tQlIGgnc29112MTnmFW1mafeiwFYB+bbCK2KX2T1HRycVLrsSci2BZhPEyhKKANpdwZ1CHVp3CoPvewIZwAayolLo8jgH8KPQPpko94ggvXsMJsHuU6yW1YLM+auMKULLmA1cx9gzESBzMoxvpNgoqc8GDgWz7bxiIQtlw9wa/1IJ72kWsroaRtkebdXiA78XWFs7I6uoTAAujf5ZXQHRhjJokuuzIS9MtNdz1mCCXFK4NVC3Actup+U1wn56noQy+cu/9CUpZdArTGMpbbKrm6UBmpraAgZUx24RSL6bkz4ErZGJsokFnNfMfh6irt1R7dIB96A3mPivSPNQo56fUpg+lYM6qJrtCGf2girW2r6HdUdXDgBich2MJkgNMK8Kj68S0Y8/EQjfo6AeOFM/Kx3jjeS1+qvsxwGVw3PJ6yPFiPbGDSMBeW71GjZi2LLlrempapne8wbxq78yjE8vo6nqcZAFFwwXQyTMzwuCP9LSsEEBluxky3k2aBE1up/9su6AIBjLM5pHMENj8uab7okT3Jm2nSZQD2qYbY6LS8lMTy0yYL3KLKF1FcTQv5P5qVAO+gCwX/qJpAlIVPWQrp6oZk9VYH8P2GRnpt6vwqv1GMAYTTsOLF4ALzKM+czIHO0Tnc0iWKQ0ebhWzoxgTreusUlyQycyiK9zCcK6j6VSiPocBKqT1k1CCBz+hWwbR7/S1aHHG2BUKPweZMIpT4sXGi8IifgOFyUFJhqOAnyfjNFEBmORR0/Ife8MgxFyz+qpnB00HdxlHwGUW+YDph2IY3UbFgaoPEkQuUpmF0IDUdYAguUqtbGMsCsZFmN80qpL4UQoJkHO9Ma+hKX3brilZKlLVYFm/44/mxp9F03z5goZ0PAM2dbKioed10G//y3RQpQfMAEEzd55cBeTr6Qq8TMMo61Jkqtq29BqxASd+W3Cta6JZg2iaD/COHXrQDkX2DqAeIKSI0JNBT8q+dUkNAd97m1oJL18+3vTFLdci2Qjr6tZsefnAEBa56z1ZQ8+Xi0WYPQCp3MgYUP0ZyAVQkH3mwVujhqef8SnpTFhYA/tZX5Rj7VgaTJDMkCMhN1su3B32+FPUN9CK1O5SNHNZH2q+Av5Mnht0+QxhVnK4532t1XjJZeHWXawTrmwcqp5Xn1I0VHFoAQjvcF4VHPTIRdwUFDjweXUhxFP2uW3eHOLNuuKw2fwww21V/GHfU7VZfos/nrcGkaOWjpFvBZ8DYlrkYyxcaNDHOenteLg5su3RHQ2unLcgjSY3WKN89NSxVeAbyVvmoP0+PpVu3TAmOcReM5Cu5CrAeHbH66/caquEzDOrEm+QTaP1Q6BTnH6S3ojvSsJsWDJvyom1GDVVG2BL0EZZ3Sy+mjEwVlUMo7UNNl0RF2lLPQYVNAvdxNqGkOlpdOEmFWKrqpgaGPF8zKCi2jbKIUhriqHnud9eqRwF21/6hSe+B3vr1comcd0bJIRMjWRtMhpeto6e9zdq01O2yXuqg2fFNvEcJQItilDMDBo48dtOT6xj9y9fMsMvmQMxRNqe20BOdKqqo57ZETPAtczWVMU1Van5bLe6JW7jq/qRHg7Cw5H3nznw0fuh9TCJcJOUfXR9cS1DmDx9qs77I45Qe/HXVIE1nru2WMc2X96mSAMEE8Zp5ZXKOOAB3hz6DYS5rNiSMjv4DkRsMgnHfId8lWgyXi4kmhRgFn2nT+vB0rgZvO7ywZjBd+TAqDODYhfoYheJRc7tiWnRKzbRKMqRey2UrawbM+FEJgFhASAg9dfQLMFVI9guyh4rwuIZH/X67h4p+L6vfSj6zEF/Qyf2K3RiPynyV8e8+FyQcL/wKBbMmTq1meIuBO+l6QMYKN30YSIWd1EcNPc1Sdqg8IPZ39mz9aiGuHrdIq6gORB8O29KWs6/WD6lf1gyAaOqdovkjY6qNzv/8gqkaIN/r2yJYqWeY+bYXZOZr+fjUG2eclUm8FV8u1LQLmbmT12gNGsqZlrJzQqvtllQ+tfU8BATL6YBc3mkWnxgHG0qhGCzNj43qruKxTfJ6Dt+YeETBwXs5o1Wd0K9qA17GNTpo0ELJQMZrKGlm7aanZonRfMvRsxgtb7wvLJwU2+9ZqUjkC1GuhlTi5FuqlTLAi/Y9t/Uemva6aVbYKWhrhpU07eWql++DL1OC3EojhTcfKepq5WsVtf67DQIiV1V5KNi9kz8eGhbKiPUJ1fp7GjbbWO0BbJM60xox/JjoXdtTevoN6Rlp1ZgtfWcvOjoC8eWQ8Q9MlY6sadPdv1XermUE9rMMQsIIxycvgJztQsMj/uz8qS3NqHSid+qVwHi/tz6xjiuwnmYLdxcTrTXZobnyglf12FOUTB8YggP4vBpe8eSN9jGNRSeSy63XHTFLAMIa85vdqCYcxqOPmpJW0sgfgWxMgSjZOrUrc8/CghfgdDVnTaK83iQXLEx7bfKljG5rlw2/Yy7qhpBzmHAGIijHigoB6/2tvXS4807th4HO7uvXuvDJG3x3nrHfdCIz65ok+nN1bNb4mbtrQoghyE2g8fp4GFfGbxOuTtcNr9CK6LxV8LH7aOZfR1dpA5omns+2Vje0vlGc2stFYpANxvhdF/pzTjXqaA5Yol+X47/wxX2kPvqjZ+n86hwPWAlapt+WYRj2jvHUuWDWjecOIEXLwplLAwq/1Xu6zfor0ijKYfYX8kCrl2jCGLgCRFNXk36sKVzPuAxBuMntHJEcFYEnQXjazCeZNYDwhUcwgc8SUWkTVQEXEmazx9IUaQFgBNhUFVn41gA3I3Q6jAvFtU+7pVWOvDK0HfygJVH+cq1D5VlfBtlOEVm9O5ttzqECYer3/rz5A4YiW6ZWK6kUCQg+pu+3RpG7d0oN2S1c5CpLp/ge//hZzrOd3L849n+2V/NWb6Tw6D66PfGSWiU8TFA6nMfr06Ozy/MOcJfjs+Pfzw5Cg6Pfjk+ODr/472VGUx0NFjFPv7YFcH6QxwGt/aaHlnx2MCQumKI0z/yTPA1oRyfbR4xVG9fHwBw2opR18EXxkAziTAXUtvGKbACWtHl8WpYaIH9nLjUEw2oCadn7xVoMsWNvXl52IuolrDkBAGyqiCgyCSvhgDFT2k4WhiAWMXTZopDqBMaeqTQZt1tDA2xtvOgYYlzPFqwXfNJKFGuZFaL46R5qkOd5CWetcG/Vi/KBqHtDVRwj5Wz3IiAzULeW1BjOjGCApRAjgobWeG8Oj4sCFS5Mqp3dTOkVzda8U1SCi5Qb0aRjwonhBJAQUF4C0NFKeR6zyCIcq9gA6aK88XjZ8OA2rPAwEhXfkO4dbfXDT1MwzFYEMUDtcIB6K1NlSWhQcu6zqKWYGuHVdCqj5pSeKB/Wh09rgz3iwK01jijN9u+XENSDHiFGmCIuLG0aoybkBvqYCPbxaPWlFaX21irRS3so9qYKdt09jsCOJnjGi5t8gL4aCGHRQAggB7We729vd0faRcW2GW2B3e2kW3R0OrrtLrhWRW2QWv2G89SxYAzjN+aFIqKL4O6ycJDTdGrjKZN2A3qYqf16Bc1aHKWqeRnbJeO5XV4G8VXTU7B8mukYkA4xVfL4MgQ9dAGNITFy2I1xCuhRXhJnJI1L+DqRuyfHq6AWrbATLKYdhsoI8byyvsiyC2O9485cKTT3lROIQeYfwYVpNJaVkfcOI6EtjrVgZzGlv6Q97fsTWNVWR3cwSxDYjgaGXda7RARE2joY0Q3nyQqjx2pA0WqQTyURK05jlcJhyUeHppwjko6FzcE2i4HNpcxZk/8pwFdjb329AX6RM22v22sgfuueMDR/T1K7TZVu/fY5D0C94BXD/2NNA9K6hYLCUtFclxjHx1DM/LUYID+jE6MxlXSmzJk+MpV/SLRccceH5RZsE41WyM3sBlAEzTRE9YOEI4bMIjHhKbVw/IKv4qGyqxF2uMwrboeFIKWXUo70Cz93OFx1c9K+3+pZBLeecboJ4tXvcInvCBT60HtBLm27lUdvvUsO1/V5VuvzGtkPACt6YOa1j+nG+qbQ5vuPTdcsmokn8cnjx9T5BiTuwrsZjKkHRJEqvdUG0kbHLRV1Ob50LmN0Plg5gbvbjQqbEPUNW6WDRJclUmo/pBRWHpSEOd4RuFptf+wOoD0f8EAWsB/UoYtaYVV+mVj6DrkbRXDhH6HDQzE0qXzwQNRPRaMhm/pWFyi0NILxYBaIUFoqUqS6qiG5Z5MqZX0D7aih41Aob1+dvTu6Ozo9OCIIA/O3+9jDk82bOe8oLHnalE92Frp6qFhTfWj8sAKHjtRVra9DvJl16aqPNXJJRjap80002e7pCa135H0z/6zPoKnjm1eANNPbUpCXNOL9jwGfB6aORhpy63Z0dgwI6WKM4mmSndc21AzK5vVztKwNbPUPHV0v1QoEfh0VUG7kKIZUKbIy7vB+CpZ4Sy4quxdn4FFFW1No+1565qjVdJkw1ZXJ72zWtbJSq6xEqBlqMXDurlppK/TDa5FHdbigjNYDbO0ZFpKyKmF3dVIrD42qtONlasAowxnS0/8H4o3nKW2d6nhyyGIWsXraMj7u1VdT2uysyW8BlFt7lO8977MvfNM16d1tay62m3PsMP1OZWeg0F/HrcBVdga3QVrlE9fhhS1MLwt1ek1kPisY3O+hlv7/LaV/GClWjBSmh72yS1g59WD3//wlAhWStLSSgE5AGYZptJT7jX+wXRlmNykSwm7/mZ5YEHiQvFqKR+GDqw3lRP3a/ZTfu1V66QZbrGZ5F+5S4ZF7ufLMa+xIJchwAcSeEK5N3O2bThdBmMO7rBVY4xwp3h6In1oyf11qXNlK5cpZg+mHE7pw6VwrY0VSpFSyCyFGcEz5uMHYcbDScrLPKz9Sg53OTVnbycpSVmdHdHPlrE7TB+0r6fcUQJp1DtGxaWHaYMI/ZiQKExx+z/gbMIUxNSlM8r6Uu0s7uwqpwbgCeeBMIpaS+66k9SHWUX/D2q2w5GD+QIysKk83u6aRxi+7w17OyPEqimuSzGWqaLt0qae1MGR0r0NyHTxDUb4ro97V45rrD5Dxwvi90Uu3GwygJ8XuUfSuB1VACMvEzRWuwpkWOzD3it0F1Vt1UbCBA1feSIdzJ4JbX26mYIaFFLkFu32QiWxBii/WYvy+6U5ae00rrYeW6aF3eRQidmIVBowQkB/6QSwaVptj3hP4jfx2CnPJlkbq7hNNs5bBlbxdek4BROh0DVna/RRnC/U6J+NbOi2hi7YlgqrdQGKBUwWmVWSBylvxrC3h5SCd8gidByDxQvVtqW1jY88RV1WN/y/YH8f+ELtgxM70G58NZew1udzs6We1/M/VyJJcCY4GWElFTSwb/qGg/oGAh1mAkHLdg3yMx5GUt815i8jcPZ6STEyi7ecfQXEU0I7TfprBnpbqMsfbFAeA1+cLzB3+qUSxpfr89d9ZfJM88cJvr9LspsfVFbp6p42cWn9yYiEnDYmkTmwVeTdlJwRfRhZeOcXIUz33/mDBcU10JDq8N93trfFzz8apq9y900TlVh3nuBXNaB52ZYwxk6HpzbO7Q1kp5Itm2msssXXto9tKmLeWEWXemi8nmi/Xe2+m9xM3rMpAJFgngvh0AfzLXqfOo0wjkoYRtWFU43IqPtnqvEfJrugyia3wnvT7OU590rFp/LUsodNedoH/AP4tVQDK5MEo3JNYkiM6eZkkIQ+b8M9ktRfphhK6DJ2Bu3mGnoOEFkDR0Hw7A6m19zZTTttxq9q0LZ7U5O8q0wJmZoEieapSp7I75QPoXy8yVGvSppIkCWP93jmC1u69yyvGqUatHMIGqNcyazcaTnvuv6ffaQsrSU2M6fLnmxVpsTKM/PXZteWExiWGQmn0dT6IAgyT6IloYzrlTO45FQkytdBCQ7otFxlUBYTUcPBgdyQxW4FdTzVMkpWidga6a8rckoSN6EwHh/1SSuWJ7fjiycL9GsP29XZlWltsUBPJbRXTMl8JgEepS0Z6YZELc1zYw59LAAn4ldulWWUw1uPpcRy1OcjHLPR6/B3I1RJ2rUd2XsMigFU7VQc7zfoBTCfJnAMqxittqwpafECRuRmvzbs4kZqZ5N1uplYOvt12O+9obRKoOKTonXHCavgrtZC1fj4SPUBfkx4dTuQmMOEdf0BVKVrWHwDq8b5xeGHTxfVoInGyIiI8Mhl6t+FUaExPig/vbS3XW1CjdrqSG1RH92nUSanzfZTH5WUllwHdsevtr3V+LcR+HzCL9M8rEiVDdHYVO6sMX+umsCulae0HpxRtvRYk8arjpSQKUSJpkHXQ3WJU1Rr2yibIMlXicHilrpDzcagNayBbQw0CU4ohbfeX6IazCVGXa5PPmxD3kb3MG2G8QDaUK5K9eURfqJ2l9VnSNQz3r3V3yThh9r/1VjXbeKfq7T7ZrpilcKwuha7ZSqGbTmemqPOjnh4C/gmXmm7jsle0Fjn7S+uog7CMh/mj5wpN2+1NG2WtchyjW7jLh9UjVZr3jyz29dShkPx1olv22HOWCuBqGw5kEcdz4tLIKuVgRP/Q1k+D06OyZxbAAm6YXZ1a/uB8F6HIaHQwnuM7sc5oXdgKJUJGfGJXiu0f4ult8vCaGiYZyhktdZFljCtK0e7UGAO2CqiJLn6sx3+fna1xFyBH/Euw2x2wJaMOk/hShxshLGBeFKDP14Xoc5Bn1XkTxkKV304cUt9NtErHTYE+0AY4ErKVtjQSCifp344nQahgs11+KtHpO58XiKTriUtbqlAav+XVOCk106Z8t+pf5FKDYn88npQGs2/a1DlDtSXALq2TtXH31JbqT2Vqsg7kwhk4YCzza+rX6pJ13KeDhwk1lLzzITbnnneeyuSBX4+hr4EYXl90bfgbDKTqjvrG1amXzKN3d///ShvPQCK9loAoEOQlSBYxoHx5FactuQGUal/1/dYfp9sfblS2yTHDwXnlAT8am/b39ZwpzLTLhzhqkg1nBRrKlCPycU3e9siF85KPk0tkdeH3DnkG5pFRd2v8y/b0Mx66I1CrBec8ltpYnT1A84qp9PIeesni3VqwkcUW9ggR1hnFVUvVzCLletP5wKnGKeUk6pisZx4vrdJPJC8R3zvYkxQaH9/qeW8+DouWrpktB8h9NmTEPpsTYW+8SYUy3QOVmWZSSTkXOC5On7idB3e37D3z9YzvKqCG/pW7M+mmu5ylYpbb/uLW07XtcxYMxFO9BERg6va+ENfcYs2xJuzWaHParUpXUF+6JdnYPBOLV+8VEYhzMYz8b/coe0rLvvU7erZtzpe39Xz/qRVDr8X+dYO6p9k2FWcflXDovxEQ9OE3vAcO2UM11ET62N2W9xx9teJYHYQFTCstm8c1QbP24RmxHgMMkb1mdqz/WWqVQST7BzU79iqbfhcdjioN4vkTHlZtK+rut+x6UcGV36NjzFfs1zJYcNbzYwoNUQtNwVmVCZPA1zhtbVHSjBXP4tU0bO3QcuG9vWsEcsIAtSFg0DxDHbWAK8hDRmq/3/v4KpuonsAAA=='), 'loadpath': ('e14_loadpath.py', '18325602071c4f56becd5d8ae5da89f50b79c8f1e148110f5a2f2bffa9aa3190', 'H4sIAAAAAAACE+V9eXPbxrLv//oUc5FKGbBJipQVRaFDV8mOnPjE27XlnJvSUTEgAZKISIAHALUcXZ16n+Z9sPdJXv+6Z4DBQknZbr2qp0pMLINZenqfnh7HcY4H+92+WiZ+oNZ+vlCzJFX5IlSfvn/jx3OVhelFmKokVj/68/kyVPtXb/aVm4YXaq+j9vp7B91Bv7v31Ovt7Px9cY03Wein04X1rqPeJXmY7X40r7pSSxjPozhUkyTPl2EcTs/Vhb+MAj+Pkng3y/0036zH6Nk4jC+iNIlXYZz3VoE33OmqT1E8DdW/i1YG1P00V8lM7Z5zT3ejeL3JVZSpV58/Hfe+f/kJv8qPA/Xu1SeVLZPLMFB5kgzVvwf93lfq7YvdTGVRTN92szwN/ZW6yPDua361o9RlRAAaHCh6F4Rpplx/FXeoinU0VV9/NXi6v+/11PtNql6/O9lX2cJPuYFzdfDV1yoDEKWn7mCvd9j96rCHp5OQQB4S/LoG5tSZ3t7gUcazQsBHc5la09UsohlwE2qBoDMPeXQJgXBw2DtQ37/g915HZYmKcnpzGat1Gs7CnGCuS8rA0NhRrsph+7WBo0nAcu9p7ykqzv3zkGDx9Gu1iuJnVHwWXRH0nva7dE83aZZ3l/4mpobyaBUmBHhCopimmHoexQS748HglR8tVfe5mvnL5cSn6c42k1WUZTTdyv13X715ASz6CNQiNFzSBGXq+w8nKgb2qP7hbv8b5WcGSah5alowa9Afqss0ykMCWbJWbrZICBf4SUbw+KUH3Ij85S8qiFL1RPl5sqI5S8PYX4WdHQJtNyXIEy4uaaB7EyoShFkexYyLVK4LgHTUbLNcYl73vjrg+eAp7gCo0TRXmU+gDuMsIdRY+tcAwnRBaN1R03mabNZqFa6S9Lqj/MBf59FFuHOZpOeYYH+1VvsAzSH+IQxzp/5a7e1T3/2Y5mGmMsK96YJu1XrhZyF1aZoQdrlEvp9yAgZK6kn52+sTtfLjaEYjYKr0MRk0/6sk2BD+zKnljGkcgJ0AP90sD5bRhDBkSd2LEwL7545C7xhr/x7FASbD8IaXHz4rgmyeMSUScY7TcE0Qdz1l/gjXujQLOU32NM+G1PYmpm5cr0MGHFhNRwWzjvp49LYBn3idJtMON0N9DpfUCOA/pseT0GoEqBaog331NnpRoCw6uAR1ZEQezAKEahjPedIYxT2qlLCCUDFcjqfJ+trUy4gQqNVmmUddoXSF96jbl/4AizqK+GI0i0Ka//VmsoyyBX0liEWVXgMr6LfrX/ppSG3Rz2q89ufheOoTVkhr6GtGKGgaNYwFjAboAwwmulRuQTKXoCqaoGztE/vT88HdmxF1hQGGRQ1TM9zTzPV2CT/mqERV55x4aXqt1klE8+LG4QXjYZSF2TOuiiZ+TU3lCX+VpBHV4S954jI08gmje5vEUU6dMH+aBVz6hKtBMh8S+0nmJBEyNSLKnCua5ktioe8/yshFttB0hMTsMHaCyuQaxP5EAVTUEZoF6hB3b0e1/51HyyVzMF0d8J2wnciXiHXln2Psph/PNPYTsgHXgdy+TBToJwAKAfeoZIdLjkkC+TkN19BZpeUPH48/HH08Bs2+fP/hZ/z+dPTm9XdHJ/zszdHndy9/wNXH46PvfjY0y1IETS1J+mUskHwip3TFACYwZElMLf4a5WNDxi6Yuk/dwUONPx0qmhHwaw/PY2L7Yz/OLsN0nJwznk2T1Ro0wIVK7GJS/vzdkQrS6A748leEIhe7HSOhvtjf+2bwddmFoIQm8fwrInkiOsLodJ0CvVbAB7Bd6V1XeifMccdxnJ1oBfahiLMtiA2Z2ygxV78CJPo6yczVPzfhJjQ32WJDE17c5elmmps7EpbgAcXtArhG/SseEB/ZmaXJitEbjFC/+EC3OztvX78g/B2ob78lpWbnw8f3L+nW2QWiODagvjAcdprEhFMx+F6FXWpqo2cr4hSE3echY3+28/L7j+8/f+Bqs+tsd5btCkt0dnZ2vlDdv/JPWcqVcOudIJypMXPscTJzARNi9BghzZUrnJxHQgIxA0v3Cr6bxMx3GTa7XDLTjIZgkvtRzMTIzJ/YhvuOxAS0w5irmKk3Uby56gEjmDum18MCKSeod6Rcx9EfFG/W9DjJeqi1RxO7xIV0WkUzvIGQVyOC7TrJoitHEVtkRUgKFfVAuoEkoVZg4l3MdEc5MgzH67EMysMrIsdetl5GOROw6w0rlANVg/kdvZRirlcpQJ0iddflcp56PlJPhw3SW2FQXOJ0cEYNr5fE7l3nH//o7/cJAI5yvMY3VK+7xjjpYxrKuscqNBSHhbta91JoKWvX2XU8IkX8eMx70JfVmjrCVwDzaf/MG7ZyAzMJq3VH927vrOxIGuabNJZC1BmuanAm4Lanmj8Ir6bhOlfH/ENK1rBeTeWLHcHJoIKNFfzYUL+EB/SCKDsfbzISILUZ1jXfOHlCLH88nzhDRVQWB+6mx4/UrhqE33TUgFibMyPKrJbBk7LIbes4oKGGw2aTYZomKeoK16kbeqfDw/7ZrRkXKT5j/4Kkrj9Zhm7b8LYiZ0hiY5Y8GDuBfIyYJW44b8PVkWl86LRMvR4EUbxrYzVNrqcekxWxt3/PlK79LNuxasKkmrFzv1H1um3cQDce77o2QkbmxtSiYowRqp/wCmflXxHGMxKilfyB6Gd1UVjxWLRT12KFy2gVERNkVPPQoigvzAxJA9EsL0xJu5GPdVWkO+6JfXQxeCZ9Jr0uDdUmZgEJ8t3EXHsY9Lg5zVFltqmpLApCkqdFEySlY8L/5JL58A/vP510mLrz1XpGhupuEF7sZouVx3oVKR3U2HQBLTkwKp50rWC+1DpGRozTmiKeCZFVgns0ph4DmAjmnmLTTUr2Vk5FdwpEXEFR4/Gjr2hN35fTgUL3dcEpW2GgUcExT4VpjBnEA8ZiVcST2lpR2XPSpcve49lzrSYckD1aagUXAyX2UaacYlodMApfLTak5sab1aSmf8mwGQktpDSTohFzkWT5eEEDSpNkZSHmC0ZDVoQZLVgTnlA3yaaewnIWvSRbwfpJga82C+DRuNok496qrrHQBNULJCFTHKLu9IzvVj4AXONlZrLppQWxcn5RRc9fk60RuCvfq+NejfZ22idgW3VXbr/DZbvKRX1EV31PT6UGKRGUi49YWeDxMKuwyN+yb8X6KeD8CtoSgxJ+nT11/PEjmTakZeRd6m5OWh6J7XMiMfdOb5YYx2RxsHxhy65iF7MlXMCcukJwIRlGSivJE/yAG85w4Tpf/tz9ctX9MlBf/jD88u3wy08ORBkb1FSYlKHpekPmLrUHY8HhAdGLm1u6CWZ8JXINAmcNnsZFSgBD9M8ynuhCQbTZ8PpUV3p2yiqWd8a95cJUPT53RGukuxkN2gmvItKN6c7weXngemVHAmHnhouBTHeJs/Gv9vbBWUHKpf0IJSxpRvPLLQRlC1VRx50nIJydBug0tI3Aa4qkVYQRLbUUdEhiksI0FOlYEY0ss3+7wGb8Lisa7p3dVuELCiO9RADLekuho6yi3jzMWaSf8IsOITyrLAdaqylIs+2bggfUvru9V2ZWu1XwrXsomb/TL4oRMcuh3hlZbosJpk3pNh7b6hqzJuszS5rYn+FxTYWrzK50iYiE/SfcpRhz5FoTjk+hl2slpH2iqY57JnqbwV3V1iqqmrifYNCwyH2YOt0YUF2sUBGjjcFOztwsnWraoCujgeGh/VUGFhe4ZHMUvIKK9NL5Mpm4zmOHEXndizKu1EV/z8Pr0dJfTQJfrYeqyyaKaHPjLPpXoedbzj6qsaNilsGjg33SNskYJ764IE1pdCi3JT9mFx9JM3Z0h//cEIONSKnXPu1EvEOhuKt1pXWPIfsK9UgMxxV2V0LGCCFIn1nWrjnydx0VsA2T96mCWUZsgsiqIyx7HaYzYcRhqlGBHUPwurn0pZNOCM8mm9ksTImvjYgcSWFYlK1dLtBVNKC+1YOpsrMJNcoWMUg8dhlmBpYkEPGl17BMMaZJ0wKYUDXnlafc8JORGI1STQBlHUJ3EHYPWodJzeb9ChLdOIArRAGb6TRu0beGGnROFk4LThWQsv0UtL6ajLPyKXqyi9YtfvXXu01sP/zOdEkWTuEkLBDyNzoIXaIjyxX46uj1m+Pvdl8dvXnz4ujlj15PHRMKX0Nxx1oFMTf4h3Px0vrqb5/evxO8A2KQ7pdghcyVdZEnerFDVod8lW5idpzC0x0FXRmMKIqwIjJtkERwpSWkTgWsFV4urpkqZLrBL0hDJn177GbhctZhXWEkdjvZMaMGBlgCFx/0xBPEl1QenoUPFecNr0gW+pjHtVZrYNKipxY35ec0LyxMeGDAldMz8GRtg4iUoAcG1mNxu+oXt+UAQ0ZcPTrU1Slct3qgjx+DzVsjC8uGqT79kZMX+GqGy+SgB8EyrEp1jmmHQGyJNfO0TbYV77qqbETEo/TytgGkUwOhM6M1h15LIQM4SA/+oFpmPFtusgXNQQE3LDNosDUARKNoNl/lOc33p10SuJs1Fv5crq9ZvtmLUIO7o93qdfzjFuo4gDFqL3xLYdLeAyly52TeBx/9BG+rkAEDLmijrp+CZZauEVtt4QfsNSw+7kGejDOSINGVWz6VB3AB9lhBrlfRY4Yh+gr87r1gs1pnrgFBh6RjQKgwGoASeIFmRAKWUCyMpwm86iNnk8+6h7WayfQwbswcRkDRobLYdh2m8CH99Wy9uhKnGfvLZH3NC16k73wkTkZ87Rj+PD11umtSdvxBrzUNt3FJVtnrqMgPNVhE9GmIk44ZZvqOzPdzmmJdti/ahFnN6L2ht3U+SF3Kx1pgjLZqHkVxf4Jlj5FVKYmcOK8Xw0JNCK3o5lbd//eFcgOtWnXg5s/C3IPQM4vtWgXLkk1KAovlf0kofmBI2KZeVpUKkLTwDqOexM1X94OkaBzQv64T6b1t6zlD8wOjV2Pxlobu+iSvV+u81Fk/kQKBIAkS4Zc+iVsJB5GW1S+6+C/K7XcnxAgDz+i3jKdDRVS411H7HXUoMQNP+57lKSg8HE8JUfaIEyvTfuF9RTVE3aR9a7W67Np3UfYrL1hJIe1QYjX6lOpjjV1W+5bXyiXcD1P2LflqTj25TDZLrC74FyGiUArfk/pXmCYICimWrq0eszfJTTrK0ljRDjHXxBNDC95P6ZGrO2E6fgb/zime9j3xSEmfqFpSvek/LI70WZHJNis35urGhFhcY+bhNeqzP6Vuuz4+faJ8LGZQkQlu8alP+jk+/RcZgSmRKDUzGJ5V3UtpZiAtbBX1zWh8kxLMRzSaS+VWVH1R2lb+tb6ahVgwnRi/HlYZzsPgmYS7gCEt0VN262qdsIDp6kLMEOjmbHtfROElqewEpZ2GHcGLQRcWssewXBKRCfTmFOWGZ15dYtV4NQcSqNfvmUm6joxBIEII0Ffap7pTNyZiAyygheCmWIDEP5h10GR1eDG/ozSVjfZLSL43dCE0amKgCF4B2W7KhQ+PPZxZGJ4/AYfb5a494/VDDiDRTsUy6IW07k8cTSRRWDIyXrNPCxua+kFzYQcd8cQ8YwyvcTmZQ9PWOQSeCatpCTkqbVA2DQrML1gMqRoV9QGwES4Ow5s4bdPJdbcSsYDVKOvwPd11N4hgG7OJPhocVOV6abTyRLWbrbOoI2V4Hp30SWuhpGl3zqIe5sqlqW+6Swgvt79chjMIsrjxQvCdXz9X/fZ1zocA8g6g1q1wGkVhhqNhcGtxXXjbmt9ih7cT2MzJChSF4XeDtfQnRLtdHuct8OaGpufWaW+vwZtaSy2M+r3lPYO0W/EI1P8EqCTRpUy1EL8UzeLUhe+YkMVjqmcf8qInaFjzm9VAf7cOGSGEazQyvKNlpRNwbfZK9ICaqsyO92UYrt1SunteZVnT4mB3M6//MfJ+ILn+Rkr8/5bY7iGsP0oS/68i9zK8QGyhhK+S5iP3es4QWjYy3uEbTAOWRq48CSgNoWgB3a8APPkQXZcyvIJ6q/5b3eiytxVt6vSq/BTN0HdX6tuRqVgUQHNTUOImFkIM3F+TSWZIsOg8h/8R0k1L105HL9iOEXd43UHsohnkyIWyTSb+QYcDdFugvFrLovHoq8Ge8Vjz07kfxaNBb/BVqbKgR0OFiFEoJlAiNwgjUN8dHb99/07bYdkzxcSRWfGKyo0TViYYIsUYPICAREDhCnL9nWJCa87YC3+5gXcaajhbllpFMSGijzLE+42/f/3T8fjzh/HRCelCP2ziuekVq5sSOQo7QVR/Joss30zFXawjFTj0XiyCCTfl8yfUuzjoCFM64Uo/JMny+CqcbvIEYRNxEncDn9TWuJyvCN0nbY3+JYqIckYn9KInhtvfraBq18SPq/7hEHGzEiI9OOg+3TPuC2+oeIlF+bm1PMCzTXCf5boumVMU8uVlR80TE0hqlgR4BYLMG+h55ZyXSw/U7Yso2WSmBnbeBSkp8RzdCzWY34iRkvvXPXWkbUE3JyTOIkTpcSyRx12RVTb+BNVk2+rJeuoj01CGFTbGZDhFSxJGz4lEe70eHKUzdojpl1Qwvi0DQy6wGrKFAwiuJUFCZTg8s/cpWq2X4X/iWvMY4OuvoGDG/VJU0Ve99SZ3f9UrWhikFV0AM7nimXhJDyKwQV0vLExs6aAvlhdko51pZ3gsodltTo3C0JdxuFFw5dVXWCAc7pVPjMHo4XCLCKSawdt0H2FBYma21F28M52/Ryqi4R68B26/91VjSaetgSR9SN0tkrahUOCPJpK9UjSB8zAfxwn3pdoRLbsEKY5X6/y6TcfQXeL+NCG5HcgFFAhq0eyaVVnvj4ynvfcv/CzcFulXFGW8bfGmVyU+z0f7OHU3hZY0qyWkLtFXWKUre4BGgrsd4sZzkk5RB8sk/ioczZxwsI+Ak5volpQ8YaOjk3TT7BNIMirVT17DEx8MUVJ3cNZhDQb06tHfWUHGubhxuIsWJcuStWuCeRDYkvwrhCuhLzUtL9jZMtAMZTkmLQ0/E/lJO8woSWrnW1x0IsS1o7TQYthRWiy/LiIOVtXsg71+ItXv8/rRrAh/c6O626+JfRZFk553cRqd7dyNlM2QA+EQmJiGQifItEWV02+p6bZVXD++dpni/WV0EeooBXvCtgd6WhICQGyRBzTQZWTFpbCriFfw7nIxF727JMuUZ832xT4vtZhtA7b98DOHxHmxvwSDuym/7/Vnt4qF9U2BKBL2MeztzW535akV88uP1fcvnEZv7QW3SsAfhvG8eP2wLheV0QxMZcPQH+8hSwqmsOHONuPKWJTPhcDaOSgm2zAuV6ZYx5hsMW8iveDPUXY0p4P2ciVBLaP2EgWDABU0WerSDIOh1BVeAXlaKmbtI9IcxG18jOCmMlRBMJIZkfdwyMgKIDdRRB5sNzJLjlZHJH74rVXgcak6brdKHwr8h03A/ZNQTAQ19gRhroaT39nFJ4bL//5uFXDptIkKrMTyO5rDrVJhy2CycPg7gAEQJPOCBXQrogU4aZmOzfrpU+IDx4N93t07/K3U31E3pcS5NUZRi/uhKu/suImafd/v7VVCyCo6jhXCv01r2SIXt2tj4ob461dyZX/mlP3pO+PvTn7+cDx+8fPJ8SeOz3jx/v0bh9fPnM+H+uK1uXh1OD7ef/vUuvvq7Z51d/i2b74dHNDVHj4url7pq8acOC9eFYU+P0WF+/iwuHpVXH0+2KerQ7wtrl7Jldk0Yu02Hstu4/pOreS8g4gdstpPFuwmWJG0STfLcIhteodd9h68OVYL2eRK5DzPeYsxIon0047en8hlxU9Y7LDEbuhrDgcgm7qIYINfZpEsQ7I4X2KDXzbUviCYxJNwHsUdbWLzDa/JkSFM4hDhGrhlH0R4lZvSQGXzTndBlv2oa4K59K5bVhfwNmMx16+AuIGbLXxsU4OWIDssuEzWvtWMFxzNthOAtBamuNN0pkogE7yp4kCtL4dBM5qJX/KwdTcYFfGIpR5u1cle+UuOI+LgLvZAykzQROrltOoyzhI+ON4F2dvEa3967jrf/qcDR9Sl16IxUvlvsThKADok9k63zxkO9/Vn5kz8oIpB6maxvK31JgCf5mgScL3M1bBYLC15SVOrRma195D+XyxLkyIrXYque3HqBH7ujyWWAMFBiKxsPoXhQjRwoSeerqD+Umd6UR6utkbgEjjO1X+MlDMer8Lc50rHDhNClEW8tXMaulhWjaZ5Laz1aqjcK+7OFWLAGwuledbmr5exMyGwWqvp2njm1Q393tanGduczMI29bbfWnExTUJyUrH4u0Q9NlVYMwZguRMQJf1/Tv+PaYzuhNjdOKR/znFxMfDM0jd2LuSy9N20X/ro2mRwPxrdPIKTbOmvH+nvntNn4h97BHbyiMYf5pchyeSb8/4tz8bN+aDWbdIKwk4x03Vgo2Iissn9vYnDuY8MDSQUc3jc4Oc/ryF1AKeHLVc4jP5CgumZvzjN+Fr6qHXfir3eHkxaljr1JghdPfMzoubTbTs1Y/V4xD71oHWjKEhsgqnZomA2Zuf8llQV/krjoVhc0p1HPNpH3q255+7RfR1ZuwMQJaPEZnY3vppEGuDtGk/l69uCKgiQrYShq4LiRgwzOXd+w0bNcsQYkht6vfEY3pTxmAFw65wO9/p9s7bAKsb12BLFNRn8o72+T3psd8C+flLZ9BYsyRxiZwuph+9sk/NmeQMrpnqhUW+snlnBUg9dzm8TZS0Lg9aG1JblwMYy4F3Lf5NSJN63VLd9ie6OpTlpW2YIu85vMLDbtuXxmva8bZ27ZTFPT5G1NK11M04RI/seBJ5tOxbsuZEv3N85Gds8SBaMJUTqoWA141qEV62r7jaMNB5qWkA+AytmB2aDDpSWNbiRWy4EjkitlYGPNN2FV+twSkJ+rB+z86GgJg5/zxeYzdUmYzUzMAFzRR4U4gpFPhb5sqvYFpO4eNn6JSC2og5BDM90aR0Cyhs+hppyORrHjuWRKishOxJRKPor5rUtsBF9jDh1UY4kTaZJ3oppWkTBboQgW6SSMftzpRbzQc9iC8MaG3F9zuCyYajoRa1P7z9/fHnMCYuoEwQjfwo+CA+UBS/R9sVqDAPPNJYww6SubMl/hPwifipbiWszqG4Y6mChQ1ITr24lgw1yq5QhagZd7J1GHb1ZL9MeGd73zNvDeJBjPxsX3Tbh/dKktYlRz+QIrsHUnXtDRBiCHSMEIysc3kG50xAL0c1dUPibQzIjtwVT1JLVg3GelDuCLNNhts1i0CQ3L3YgQuGe10pDPjZ177qA4pKrKOPkKjRk2UADTz9AVQm257Fzw8WIeaIZwjO9QY3Rkzr0H3oS5tXnd3cG+9ChJ93dH+qLtN6IaWnu+24xfWy6LExNpnmJGqoPHZZJ4f/X6zoSYDovpGQbNhjrQMJNuZxEiJY1mUDseux1GdGIvAEQ51ZbDWXUlt1Wl2gGrFgo1qj0a69F22NO1oxEtoBQOC7NvMpOK66v7PEFz8OojF93K0GyJk5WFoFKyZP6vDsIi65iAc1H0teRDGcUk0VUh0RHWuO4l9grJqGENVopoW2HcOgGdQ1WGMdBv9/rd/TGHt7fKZcmbMMUrNpkBJ67MVu4OrNzC7npM2028UYrtjiLp6SjKveG1QQ/8G51+LRno+efwGqo83OzmQPJbWyJ4FTHpB1BMKGbyuS8VSlIWtApOR8zp6aqxrIvtKW6Wau9IZ+C2ZmPCcnp8oHWx1x4CQGevmmwuKIdenjaKh+KzU3NuioA1TILvPGhsA0470yp7tXByX3SgunsVBrHUmHQsExrglM6wbmLorj1be1ZUTmBNrjfxNUVWUg9b3Lshiml3/313lvm67ucNk+SsCySJAvHULhcSd/DgqmDDBcB54MkGm9mCCBWs/LTucnjMUJAlss5Op72PQ7LKhJojPb27ZdWmgd2nRTtCMctczUU9h263FOmCxBpH4/eKpdTsHTa062QRhJhk0GWqDgMA1moqKX1wJoCQ+NJpbsdMUhM45L9TqK59tXJB8MYJQsMPfTnyDdhx5OzQ6EE33B71KhmCiZ9QpSNgygV5aWZc4kKMHC+VdY8IU9Qr79HY7Cno4mlUCSiuLniwvHAXmUvfDHXLUJxAX9jDY5tfGmRFvsoaSx0V+10FeJbw06aPdZ0EzwwdLNSRzMBESd0DNgPIaZVmizDmm10VLFFkOaUWccmjv65kZybYkAMTSLfIpNlxltH4GfRe3IIjyQnq/N//tf/3r3c9wcH3flg73B34MiaBj+fzAYH9EQ2FPPOBj/jLcZUHeC58oPQpKkFDgapP4O6tlwiIJAD8/xNQNi/pz70OeAyTYLN1E7rumdtZmpLREBgqZrQA44bx/se70YMXbMX0bNN2dPhQHvANaxJLACkwIFHq0e3XcT0ImcsKIZ5ojzRl9TsraOnpppC1BgyY0IXMn4fl5bu3r7xAZRZDIpwhZHWXWo7i+2FxNFTlGhBwCXZ15xlsSNGpOFzxNp1M+1xq1viVLXdzQy/pbU2+1yHG1XREWEPttlNpjBmkM1eA6Ddb/Hhc5Ocd1cYa3UDpk6zet2RfFlapINpipNEx/pUEvqqZhu7yuU3OuSTt9vVqIpQpAiRhNk55iSvsMywB+gjp0bltlwrpKNjfE8cOenpcFEirJDztnLIp4yOWTiHoOokqcjMxLGzWYgwAKJUk1AWRElQRhYqrNxJ9KOkamIgSAVwwSJlJcDqwt5nj9HdBCOmUzOZhSw0j3hbXN12LXVV/kg+kJSK8iPrCRVQmshLrtTY8TwfiLtAMfFzEXG0FZC5wg5lDTzHq6UCwpcdacEb3rn1VMKlTY5cAAxw99NlFKZmNyS75EPJRbvJ8rBk21q6pSt2a9F0R/MYKVcl8soKpmsYT7LJV6DbvyMuSkLSC0cF/BjaYq3GiNGAe6tziF3hS1ktkq86S0MbBGVuE+gxbrkaO7Zm1LY9eF7uND64pOGQzW51xM02Ts5b4g3v85EYiY9i345srtYaqVmEWmr7czYiI5KNUHZwsyMFGcF5IoEFrKajjIki5C3G2MzaErzTKt1L/yy14lzqRd/FsA3/rK2v1AsCUhZ6kJh6xnnnFYlgztMWhucI2+PUG02VpkfIGaMe6WxV32CqbtnMW1uHZveGcQew26M25rrL4zeCvGr329s4UaK6C8qzQ0I4dcGfvVVDfjp35VmypGB5WXYs5yG0ky8ns9mp2dgkr7SnsukQF6alN4DxKEv9AEsgxTAHB14hieWnc3+uqFb5XLtvLEfXzfzaxl0evB1Ho7OMD5VD7BmxJTst+R1s5rydL4vENgLd5TTozI+uiQOvHhKSVGXO0uo2/twsz/27+4Nyw9TvSW2kG7gpMhrptA2NlEacvoxJxX4jqNeSGEbHElVyIJlIMauL/Hk19xuDF9G7SDHFrB/sS7BujDSew4LUWpsEhaBJn7P46dg6Z1gLwHaY84xXEVLn8rXa3RXt0iEp7/ByhYCfE9NcSGYa0afrafKZcP50PdrSmrfs23uwymw2cP5ZmtUfUiQsNyzz5Bo7tvakzmq8uCEEyn7VZIKRMvXRFFkYdv56jn4X2/59WcgeRKOtJGfnHdtGYxbhGERHHBRWd8dkskwQHG6lMSjO+xgipdj4zfuj7z4cnfww6qvzMFxLxlEsfVbPbPmFWPQv5YEvnDtLW/RmR189toCYtk5GKmEldmtwoZF5XyQ5REBU37E9EuaICf6BuZQ1SPU+0ryLDuHPGN/r3QMSmto0KIvEXMxTzy9LyJ5sOQBjKOaB2KuFY8S46kRvtt0rkgIGzr9ncujBdbHiyrYfL0fvmGzwxvrrMHfTRuFcH8fAKVYhNpVrdALjU9D52cVcZNOleRRHT72zju4Q+1RLzR/D60nip8FrYH26Wee7n1i0HmPPJPYVykD1gnS2meLYCx3TaewhXvUumtYL2T4C0yUtdWEwM7at/LUh3tIOLbED3IltvZWs+Kww9vL1mbV7mNdrV56Iivbit3YaRrdJTuxTl5tqrjT9sJaQrMzbV6gLeNSTBHAmkx/2UUWZYFlNV9AUtckh8k0hInoUu23IgrvlgZYiK6915NLoVhdnkSC4nuS+kuJ+D3wJOF0mw2NMKLIBA1fLdyZ9Pd/aYHJM69KSSVNnOVatBHX0tJZ7NZATBRre/U6d/K2EsdxrXkOhi532Cdw2eYb9lMIGlYzwj07MReMYcTMF/MgktwYpL+1R63Q4mpCQgc2Ey8CPwWhvO0hWWH1ZYVOezG7UkQkO480qxF4Gi6FqaaqDhFExV+kxO8YjuYUPBndCNqSKZckSe77aCaf42Gqnli9WphswnpGeN9tknDP8MtEsINN7uTmchA/OKXgjYQ/3yYr903PMyPDAvy80/2G4yfZpjhBPiA2lzxo8Ufdqp9iyleV3rGbAzVS4nagolGTslexehtF8kdciqRmCpPusTLyrLe4E/K3brlo4SityIjFoO2ZyQ6OVXq5nemBPa7xq+in4FDI5M8TyR686SjzRRiY39a5RiwZm9QECGgrYA+1P3T9b+LbDBqs5OotDj/MQZNs2NzdC/JiURL3h1YRMsC5oS8JB1Z+KCDmTePi2eFjJkCzM76xSvMUdBBoXjxGX837X1HN2zOIcO+0yg3ZJs3EqauaZKLn8gNXHM2/nYc1UsMukmyUMa3XG2WILaaB/44bnBrPYHjb7tN/cdXDnCBhIXPfIbqYBBismn6DoNtSejrL0Hu+utCipMKrTVRvbNHi7KiKpBBnOtJu58N6zKsk5F1Boo+u+R0SZJL/tnACyeHQqnKDsG7p75m1JYc7i+8xWpeqm9eqPs4Wtm/YqfbzdeSBG1TpeKB33odXtzp+EDjYqVOnCZPfUh+m59c34pY1hm0aSr0UnMJFPiuVSOVNQ9jxket01xrYFnd7FGHk1s47UoeLYQE+U8CNedEI+FVcfvikns22WvLLR1M358C1z7EyMxbTivDyy13FG19BeaSrkrCjoFftD+mmbM5lYOTOO9SEMh+y+xFknSwz/mk0K784s0hV78nFhvzHNbNaw4HvFFyxwILwZ7NvSHHBaTrtafnB+WSRZZVjhzC7SPy2tqKNQRlbii5ygm7ieEbRaScU2brT8+LFu2/sfCKKpnccogdo4C0S80uvIwlu6kb1+hAWka0zDOMARdsrlQ9nkACLvmTqlcnKmFy9TtO+gO48CyQ6709jAUhyR4CHMMuU4kkawScCcrhdlAeFaa2qS1rwdwlWx3zuASgcGWz1+oT0DF0Yum2WyvJfqUx688pSH8kiP1u8xWOyJ1WvLLurjnTlmlYS34fCIWtSFuwNDti5DMWPSiXZ4zRDTUtsPgNe1HMomTcs6WdegAUVM93ddywWGD3gfVODyWOEqWvMI28zfhx2vJf01GboQGwVsNKp0zgp6v3YQTRTcFaO05YQ0OYSGLNjo4YekGZOrflAa97P1hDTT6Sej1lPS/kAYENVanM3BoQs2nIj4cJoMsR3EVf2KC73nAPTdqx4orTOY2qcUk0hKNnMSDiQdsAQzC8dY1iTFgC2hMdZLsh7JqTeDwUHfK3fcWqezioyjCtaIeNBmGw4ig/QHwHBctD5UVS2QPizCB+9evnyjsjVULD5b2l+uEpzKQby25Cm/Bw2MZVeb/iYz2E7qO1umdXb69RkOT5Trb8680lwlzW9JwB30lSszIs67wR7d88QQ65xhpyb2TaOsesqnQuXhn4QcUVI4uCtsHTlxSSAmOU5kxkpbkTJFTh2rBRAy4gytLLUuBryvfoxe6IObzcRbnD/kk9WqckW62VEzlomaxPEWJ8VpTObbSrZktNX/BqFL9F1Hf28rX8WZw8UA/+4jz291FD31oTx4uHnU8Ovd90oDS1GpS5xav+3wYdkAxAQiRCHZhBJocpy0l8Pa5F6fs5t55oRjOfMYDcgBG4Mh+4rRzDN9gMheRwcNlZ7mR28S7F/j1hCX90jcuqJFQEGjATJN7h32PeTci6kF0cv5bI/8nwFOcA+w7Vxi8YIklE2nPGqkNAYkaWx8Hlmh6jN5SUc72qXFA2orAcTGHiwgbSgq6b8RTRyrjL6ITbSJLIJ356m/RsjTmqaZ0OvpwZ5i6+h4MOj2D9gbLVOkk175eYGnsl1J+ApvG5Kce2bfV3lKs3L9SZYsN/C+b8t2aCmeWOcYH7/7Dofu1gB+xwEnfDA5TetYduY97tRAOPpG1jkakBt9vccv1om+/wp3i5CkyyT0c360JcbPCvF72Kkq3Emdod701NxWjluRkswuTUlPL7g1TlqpjlNX00SQEVFs4udutbjX0Y8bH9SUdAMeXX8FPkXdplBRa6WYfcoGRsVctq6yNw/PMEdnGEBYoqJ+skVHZ94aOTrkwXkoE0crTmOnMnyxsMrynpzM5/wjxtKWGA8FnmItz05ZiBx7gpVRxku71/aZ5fZY+2ZstYNyon+FnDyIVKC+ILL2X+PIkhJ05RdT1GSLmgLfeuD5dl4Jzj8ji718idkBM8aJFWXUjPCSEXGTaGY6wOsFg4qbJCuP1NHZwoojfViwjFdYe2D6coRt4/b2IRnWrBEkSFNz9/b8xiYKkStIQ4qY1enopqxPyiIKGVsqeBj3px5kB4bM5jaHKNd0yivUW86hqR1B0zIA8TjCtw+TnLMYVyvgDGpZa7/vt1iK436rqeFaUNCgocbCByEhI2LH0i224mMj1zHLE9TuTtVzwmZeOHE5YE7nO+nLuVOCljjNVU6v5t4heRrTiteaY1MqH7ZnYtJhHfU8TNW2eLmXAdACU0Mpv6l+C1+q1FPsiBL0kURWgjfN2bGYBHOAyowh1GKKN55WhekBlBLxKYosB31ksmcbtLXE8vsyOg9LZXRJc1w9AfcL0jd9ZA+CArOJgal8doOchPwtze9zOMHWS+Q4Nq2hIWLUXtlWtiDJea53ccu+Zlnny3ZagFRwk7OOeaT5yZmOKmkpqPPAWUngOnZRUwHjq1c/5TeyBE9VF2uiYbvIbU+vWGLFc2lmuPMgRmJjwv3sj12VhU54wz+3HOVezcxY69Md3EUP4Lklyv78njuovEt6JHSpIhejSAnxg5ruy5Gyrd002kI9ExxLuuejFgWmZSC6DpMSjv11wyo4OwZ8hiuzhkYcm6jwViG3OHBR3aT1THHaPr8htLvdliFOi+U78sNV1LL/Cafl316f0Eg0motXOWMb99col6ULnKDsYrvRiMOAdkOyIviFUw31EY80L95UPSGbDOnJ6NnRu+/H1N745dHLH47H373+yFEL70+I4SChlS5Rvp1cmy0kOtRGR0+JObbX79I/e88UsaKYKHaXerzLPpRdPZxd7hIXf9ofdJ/2D2X/lHbBh2CV62U0jfLldWFgT4xLA0Ouxa3Ve6ijKyfs/5wvacx8AHTbUJtFx9RfpxZu6px8fH3y/l37Z3lKpjjrqs7J+48vfxgf/9fJ8btPr9+/+9QomhCajkmbbjTw8vN3R7p6jjuzPppuAt/h0zzH745/Ov44/vTD6w8Q37rDu/GF0zHFtNRAdWSVR9DLAHXYg4JEQ4MFX+zvfTP4mv0vyE1xsYvTyD58eKO+6X1z6AHDq2njqQApPcsN74pj/VBSXuTRJFpG+TW4RTIzx3OKe4cF0IQzvmcwh8EkeKOtOTGrt4PZeHv07vWr408n4x+Pf/5kjQzDoquxYBLuZkSviyiehSnuGJx8IVOgoQD3AB8MjkO/L6ZT/AosnC1L+M58veE1NW6RQ+suFyGS/9IVKE7HVuCWsJ34CHX147vjN2ST/DT+8PH41ev/OpaeCwrwdB7h6s3rTydOiXy4fPXmiKbw3SuaStxq3MLlu59evuQL/v6H92+P0ZSgBVrS8EEITJUPFKdScLfJXEdudxwxLfdE/SVTOD0tspHI9pAzyXhvvKb8ySIh9jDUsTtTP02vix2RT9RkE9Hk5v68p9wfeELm6ivitvBIfP/yk3XOwTTR8QOrKCYczOQ0KL28R9IGbM5E0yCKhrM9BFyAlwmJ1cjr8YwawYkHUZz3vHp2KMmJd7rWUSL1I5XPSscps5ASKD19TnPvcrF0CiA2m3T1ihnvhucwTGJfmKjedKN/GBF7+t/1Wp6a36n88L+/RvGvPl/lYATWoqlsRudAH9fsOlI6dZLkFvHMJAlJdHUwooQo61cldJHx0MehdS723ixlbRWblMvAxK3pl4q1Koy8lNlBETPU2EYRbDlRvkyT5KwiViaG2DjBcSiyIGT2qda07sZ6T3nEtky4dRBKUDty+6o8cpu5zpXJZICv9SzyC2c8Xl8LIY0dE9RwxXvusm0DWVf2hAV6C5jXQ7qFJIuu3HJEoBbnH/1t2bXWlfQytXxadt4pQUyZdaZ4+r8q5zG3F35qTij0Y3NEspa6gRHIOCORhtvl/VggaP6KeXvAEW6LkAR8xNnl4O3Wh7EEYRbN48bBijfnQ3VRSS2pJ4f6Y3LI8Iyc26tIbexTJurczEGN7ZmQcfC9lR9HM8Dl8WNZcrB3gGO4HH1sSsG8/Ql+Tw6eglVWLNLrhRr/gqCTpBynkSCFEwf3ZjjkxIQu2+ynCOoVbMbi47lAAP1uCjQavHSTFw3POeTL1SGlnL6QlNtbHeRIukHq12rUyyqckVMDp9FIESJpOpWkUtewupHHggSp2wTMAkzD4tMbfUFK9yaWTLE3XJdRoPXUW6lMrTNygRDS49PzszvActthTCHhfq23HBXsl+ZjXMg3Vvg6WFQdE/50iv6Wc06PdTCJrS0XyZFkgUJUIX7TKROfuQZbvBJd9Gk9fqZMf3s8vGLaTcFRHRmLronGKr6KmtKqx2Fe6Fsr5R3dgW/1eGtlMWrnUvZX5lbCO73Hznxw4qev6YnrVLptHzDp577JQCtzZTrcmIqCf1Ua6+ldq3DVojJri+AM4YDMb1Gwo6Kk9wJM7fV7KeltZeIAjcW8Gzm2ECxeZbgMTIvZNg4Ai69d/o40JSHrpc17sLbl7Oq4Y16AsfRq7+HZONaliGkJzWCAIELAT6cSn8PIt+vwKfRLrxZghVnW6K+5kEUB9E4vYAgtyIbGcZMSPsdALZzFJEFNk4SDnqSYaFgFe+RMvia/3jPdlQyKhzBBkzb7A0c48adIYFVyR73u6UMv9Jc9HYHlm/53JMn02I+zyzAdJ+diDcAtyqFXTLGyof8yWoeZfsCki76ZdUTEt5ZHgLYTVcWLLNBpUmcdatUM1hWyMwBvkBzTfzWLM800M8dpzrhQpz5JPNl0lq4Qfy5damIPpG+Z69kSBdD8V576b76Qr1m4roxoKSrVD+53XxXszEqChPZvWyLAQw5npBFT5XQzoZnfEjYCOUUlJITJDh/RVMBOZ1MoyraR0Vb6K3mDaeUUXKmofnjW1qsaJ3B0P5xezymMAxCnqH98ityfy0vuHI8kf2QM30VDbQV+Z7qBEs2r2ErAYxyX0znLw2nNdn7iTFvWX6qpBJLJr/qEz03ubclBbBjLX5eJuMZw3HlCfDU5D+MMx9TlWEhZp5JqIpx11HJNb5ejfq9vnUVouJjoiLrPQx2YgGPz5mkYcgTnPzdMNWBqRd4erYcsg675nrRokqWkaXBHFGICte/jWRFNKlECgwNlOshzhZMhuYfEhN9rUNzLNSEndWKUCt/kSC5igyVIZAMMnlEnsdEHD50zr30OJMFMh91wRUWmu8gldIDzsYrazBuqj9/t1BZ6fT76SpIIm4BLRC6QSddV1x5SXsjYxcLrqGuz8QR9IK72fwGSSwqWtaEAAA=='), 'trace_summary': ('e14_trace_summary.py', '8d89ca106e166c97204f1aeff931a50a75e203999c876f248b34e6ee24212b46', 'H4sIAAAAAAACE708/XPbtpK/66/AsecJ2Uq07KRfSpQZv8Tp5ZomGcfttaPoOBQJyYz5oSEo2WqV//12FwAJkJTt9N17nsQmCWB3sV/YXYB0HOf85AmryjDiTGyyLCyTP3nJlkXJPvz0JsxXbOx/65+esGNRhWUVrMtimaScRVdlkXE5UDD3/HbNyyTjecVOngxZzKNEJEXOXp48Zafj0+9GJ+PR6WPPHwxEFafJghV5uvPZh6rkYSZYdcWZQ7DOtwBDOCwsy3DH3Lxgn0SR+2kRxqxYUsebqwIIQCq8IROFpgFaQ7bkN+xqk8clj9kv/2BxMciLii3S4oZt1jQa7vmiKK4fCXZx9stTGBMV+TJZbcpwAWCLfBQn4poJYANbbcIyZuI6WQsW5rvqKgF+pGG5Ag7dJNUVDM64EOGKw8T+5yqsWjy7KZMKubMF3iwTICnJiYa0iMIUSN+UER+yt0CROD75ll2FkhNpknNW8qXwJoMRS/kqjHZsHQI+F+Cfvf0peH/x7tXrN+fBb6dskwteDWlczJfhJq28CcvCHIgqxbGIrni8SXkZREW2LnJk7rGSYRmobv56N3n87Xj0+NsfBwx+nhWbar2pgjgpnx/PnqnuIHq+TG6fj+b1kyR+Prp8P3pWrYMyzK+fz0Yv4a58Pp+N3uuL8/ri98vzty/3o5fnL969PJ/7JDafpLv6E/A2QPVEgBsFE1XpVqBZPv5yPY+59eSq4prnqK4wubwqizTIktskx+k8OTn1nhJTFN5jhReUfAmzIP0DpCRFjXmxC0DJV/wpaxhg0fKfiv2X7y5e/JcWwkXw8vUFA3s5rrI1c/uYe4IGcPo9qP+IgcgOiHJ6ApLbVElaSyigOyme70ePvz8FfQ/B6tJwBxRK4tV8RiSdNGXHbATWV8QckWUcdFXbrJhW5YazMI6FbIhHpiRtgTDXpoQG0GROwZKRUjSLNej0Yqd5BPx0kwo0d81DYBgHtd8xVIxHdAOqRxx4l0vrZeAy2OV76vGU8TC6YgmyuYyufM1FHHm7LsDxSIcTSE/lXoOJgED++8O7t0O2+jNZIxkobiQ9Bx6hqzl7/3rCGq/m0lgU6pDlmwxkzdcCLKdYTx+Ph8z3fY+NnrM4iSpW/7x4gzDWYPvgzfjJE0lAIIHugB/sGZoJG40IHDi7geM4gyRDosGLrdZhKXhzD15N3yDZ+hq5rq8Loa/KeqDYicESGADOKk15VIFvFUy1vSg2ecXLweDF2eX5T+8u/ggufn1z/oFN2YzM+Sv2royB1VlYQTcxIT4tk1JU+Ai4fpMAMDcKBR/BFc9FUiVbWA54CKLwfPaSg8fmZViBeyRIrnZZ4KAkhhH76eVbtuCwbnDU0TX4sNWEvXpzBhKMwCNfB+T0gmiTAe/QTEhdPXZTbFJw7UBTeZMIzq5AgxzZy3laQ9cgwQ3H4MLLTOPCOeXIDkCWhuLqdb4EAlEDgD+ghHrgMQ46DoFz2xC7s2te5jwVCgNjSxyd4OjJpJ7A5Ouh1YBAOg8boNQEK9IiBMqcpo+Dio0eh4U1vhTZuQzTdBFG17Te1lNhbrgESTGeclxRkS1ew4qmm2IB8S5Y8SwDHpyBHu7AttT0GFJSALScRZsKCBKTScy3Caiw7PGMqJxMQPuf+4TDdfIoSgOgDBbRTcSdISvp0Uu+/ZnGBGdpekFte3zeedhcAZBAQtlHZSFEoHAryN6whXEV4lwIIwD5ie4IiGqwBxR1Z7xtGsEyQXrEEWqV9037Ks7BjJOqyKl5uRE8DkSyyookRkwg+b18WPJoU5bA7r3U4T20wuOYp1W4j8KNQI0u8u0+C7NFuBci2wPwBpNWJMIDjuZ6D7+C670AkWTh7V6378FR4H0GXjTJiywJ0720AehaVsa8QQEJWJkJvN7DUgDsxytqqjvWWkK9/4F2/pKWhT1dv5fmt/8AyFOuWuSNbqoh7JfZVbgnPQmWN/FeaczeGbDOT+n8ggvFhwr4tJerj6Dr97AWxj//tl/j3+B621DaqC+Rihd7pat7kf3wY3ALrhavxvIKWMZLHkCfBfTZC2Bg9XNboQzTMdTEeLrfghctcFmI95u8RK8a78GSA3AwAYhhL5J0s19x+FUWEPnu4M+a71+E1Rn68N7JN0yIivVOGdgemamvkzzmIPWIPPFeKvVe0q67SMUMDknScCrQY659/mvp77um2zUt23ZaxmJLw8Tdmq9tRKaiax11Mp5F6526giAVr/rkInE6MJd3b96cv7h8/RtNxv3SyXiDZuFzI2AW9zHohTDCXdPda4ge0dFC2xpDcXu9nA9+Pr94e/4mgMcI4y9HSgSxrNaboJmOusMpfR5cXpy9OA8+/Prq1evfFd12JIUjjCd0azbRQ2/w8vzV2a9vLoNfzn4PKCCE9GUK4cSTH/yx1fj67eX5xW9nbxDXt8F4PMb/g7dnv5xjM455Mh58uDx/H1ycw53BhdL5XwxSPs7cjx++8QDnIIhSEUQQeXGc8OfBYAChHIvQ9pLlzsVldMhAW6eaF2rBhxBHen1s5KsC4jyIvyjGwycQoxOzjiWXPIw4cuyBAIGiFb9lrhGBeD7GTAgYB08xFqkwonYcDzK/G166HrVe8x221mTJpxgxTFkzFX/FKxe6ytZkSR1gAcZM8C2kQJNalUtebcoc23VXwj+1BI6EABOJUGThL/T44wKER9ORdJpABZCjtZ8eQ/zUAY3a0wMaHj8ENI5WoAXvdJD2UD9VOl/eotJLZZ9Y5gzUlbe+jPfcXsw2gqjzfAF5/LVmYspztxGHx56BGpOWNvCa5hlIak56KmMxJRO8HQy+Ao79634wt+RhhmEl/IXAlrQ/AD8Pro3fVi4maooJMC2883keC8y7wMrBfr2OLmFQ7yMEGgz2XaLf4zkssIBn6myq5egHfFKWRSmmDiRLaYgrlzl5a/wDh0vTTWBdUSmKzLgUGBm+RFeQjExP2DOQCOSSGa/CKVpEY9V/JByiccrFisUnWCJ10QXillEKINP+Og3oC9Vfmm7U4FGOirkqFnCQ0TL3SLFggiiRBOZiyjWEWeNKE0JOAGEXhwxEMGeNAlqk3GEuSkSHvAiFEKAbwCxUOnbMJAFTcb1Z1x4FkmHQLnK4mC3KUEc7FMqg2wJnoWDLRrCLzRIALH1UEdfgo1f34AX2QK2HvqjuRq+601dU9akM2hlV1OoOoIwyoZ4iSj+le7fBAhqou/g0UunhrG2n60KY5IzoUg/12DfspKHc8h40uMSlwLHDm5srTNcvyw3veoRE0buE2MZ9ZCnHI6/bG1SUPZ+y8aQ3fmrcSGsQsLh/BLSRDvX69/YP9pw1OoVuBxNRYAlyajYCNwVOan7P+LywLIzAIG96h0mL7jRJNjfIZ5PR49P5HSRkBY24Sw276oiDDumjrd5IAlAwmQM5OGrwxTy+k7ezSXJocp+gYwK9kN471ecuJbw2lRAsYsg+9ere9R26J83m2rKPf043D8j+3yDKT/2C1PweD+5jam54kB4pIK+eQSesxiA+uJ9jcOGwj9XH8mM+dLocwTHfTFvMxWUVnoNU8sn/B2/vd9Zf5LRt3Rh3nmPdOck3HXOpeYIB39zpi6M65FflrtsPFmFc/nFRgIXML8ObQFZ3keoh0mXPjN9GfF21l7tzjBa+kL8hFuN+C9ONHO2CaW5yXMBiDA6TfBumSaw2jnKIHBlyjh3VuzSQQ1SSpw476hL6bzIEwHvYFL5IqjuKjEAePWstCKhXq00iJ/dReZC8f3kYvObliAryWKoRA0r/2Ct4gGUcRThFxkGSJ1UQuIKnqHwUWWLMBaFkeAuNEHyCWggjHsGePu13TKm/3UDx2pRA2A2gZ8EGZ69q20YYVLfnh5sxWaWEtn8Z/0pmV6PnbBYhANwDqMIUMA7rfHZuQ6RkNaiQJFz+7EZU9UAaabdRBgmWntU0YoG202CxEpqte7urDAOhD8WTPv12ndhpsQNzlns7bXIIvoO4LFQM+gqw8TY6vg7EOsxFL3Mhwg0XEPkbfI0g10WurujvfFCrUhjHSov4tq0uimPWWgEWhaFHIpIcJp1H3OXbIW3VeOiO+JaSfmd95XjsP8Dn/t7yuS1/K8sMuLWoR8ITx+urOdBqiOoyrbHgre5sUohQYRE0ykiT+z18vCnRB0J+VNXEwDMJf+yPu76QJG4PgGjJ610I3MvdWjrwoeHMvQctSJg3dUtBNhpLlTuru2HLs2iO7TCz/h656mADQO9Qo6Cp0t7eoBPWQWjaH5bWo2c4EqPS2ckQyYDJ2C63mwsRBbNxD12y5aR3StIRoAJ3mjCFs7yJIhtFXWFEZbUemIzhiirRC772RwZ4vH1uNx+Ab3iznnUNrbDrL3pjC0o7GzflYZTX9XD94UevSzqYZD3QGXaG3OcY71aNFmYfCxGQflTiHoy6I/w3aghp40BcZyN4GYQ5MDtUlX9HulD7ccuOM5iLKvyqmmKvtWR91h9pS2ucvC94pXb23cxflcVm7Z54QzYbD9Ex0a95j3vq83J/z9N9qRNrsAC4PpXUNdg2Iw/lGNFB69ft/T7gbqXBcaf1uHpNXGzEDkRMNmNMznGcn97/Sq0Mz8DA7MgwMNCWnvfYqrazJnhwiy0vU1A3rLwJddaKlmaM4mEVbQrvpv+4w7RVgdIKc3SuaNq6tXAjy8e9cEwxyaMFoIVFCeS5EBSuuJuDvl3z3dS0syAA1UkqngVBg4ciuJbgI4zneKPYOHpGaECs82Fjkc1Dq2aeoDHKppPJ3J4AgG5DTiyQybzjmzEh6FEJSTpoA9A6AqK7+Ug9D0DaWrMwhzsAl4bwwb1olCyovdHGmFc8ws1uOtfSo5Qf9NEy5NVNWIIrFZznWD19Afr6CMfNfN+fP2IyZoTULeZpAGlkro5FfX/6ozdhv51fvH71ByVNJRcc40a0HRamaaNjRT56eXH26lLGmKKtt/Wy1Liv+xUXxjkSuYOSfsj4Vp+ZHj8H5WkzVGwyd4sOBHUJcqRtDxIf9Vi4noxiMHKlWUqfSMTKuj6drwvwBFCAPt3cmcjkJp/avSmdy/cj92P8jQeLRiEo9/IXIbC1HmZtNYCzMJw70ZFJ9pvYqT0AQzQRLwBxP4aBtiABKS5M2t4jnZgCWDQ7KtC55dMRw2I2YSNyL9Bs25S16dRiAXAAGLDobtBgv83C7IRXX1NnkzHOJqdZ89hRXIgTEaFDNc+R0QERkWz5lPKlIcw2SjexOi8Xy6fN9sollUvoMB44cfR3NSzaQ2nukgo1herltHkZygOvTxV5AvdPAcpQH8NrPLn9HANe8EuzueaTllgisGMzly6nZnXbvAY3MOytgQRd7gSEBPzlID5nwgzuOSBA3P2e4KGBSs0RNR9a6XjGzvk8bwSsWD0x0jgKU2eakk9FkrvxesiWudwQwusAbwW5cuHfhOm1QSp1WqIVYJ95z45qLwpjChqTBAK90kRUNj/mtTnQqQO1viFgb9L2YS3prFsG0alM9dug1wZrWJlti+RnjFY8vJCtnaF3H2IyXXMbSp0oBc8lj+hVLVNoLZ9SnXQoXCvH2lIKOZLRqagRFYvUQXWl226vtTmfvbtpJwvRqNeW1dvGowwfy5OBlAeVvDB/kYdiF9OeYxutati09/BG4xPkGXiGeVpVuwf0y0VTg/PZhTZ7qtHp1UFSGkiO6YeNK7BCcTzRDhQbCgNhFD5Uu57H7IR/NzDC7ncfKM7GBb17ZkJid5AatgyBznjCjkos9bp8WJ+8MDhl7h6Rimh6npu9DuKRAgEc/skST/bDdOmcvjVatrqydm0+B16iEMGXekSirYuSEEuuXrM7TQdj5AYtlsW0SJQu9C7M7XpoVxhLKlqhOzi0Y0+b8/irZYmi8rF4xreeKatz+oNZAUmrXZhravd0Vto9f/dK5lBD9meagP3L5GoBLMJdA+YaadaQvT5+d0AsxNTD4gdayWo71ZlWzPLIwW7OR/H1BP5T9PJIckCV9fSu4hDLc5Z3yzrcUUXlVmCjJOg3VqCudENrVxcjj6JI3YaI9ravHUeJaqgjpn/1oRV59FyFhe7tkOXx9InSEkM8yJ3bugREIR0wAzwejdAHRoL6eLwM8qR/aR+QV9Cl76lTNNUV87MU5BOHTEzAQVkyBw+l7pesaakdAZE1xk5GaEq5FzxD8+BACB09l9jsFVMcUC+itVYEldKg9f6FJV/Zgm8w+K0kRy7lhJewfa7DHc0NubjVd8/NvFaxS5RRXeaoe3p4mLJcbfDwY9/BLVXrn22JAnIMQJu/RTNU6cF2bk69WzrrQa+3R8gJ+VkhKnwNKCtysAhITOAf0qWZQBBmc5mtOXecEbEwnWAlCIwDhOlKcunMOWWTsE5tMca9UkubJ6Ei+0XjV22eN6iu8UUB0ncQmawi1wzxjOM5Ag8FCl+XT5omCPawJACj65rmCDrqAqonVcgsxrb00qwlSM1XJNen6oxjpK/bp+poA6umHH1INOzWuRAw1r7xwGkmgjWsBshfCH9gPKzJ4zEMgQvFdCeCSVndJIa8RuDVnQ+dY3HEFYgoKJbqzDH6AADkEkLkukxHkf2aDw+AhfyvoeCNJ3fCQTx1Ptlwj3RgZpgjccDComNC6RswCJAeGx6B34PphhUONxqkCwadPkW9Vn4aAchLeKa2Kuihuran5jQsCWxxED86ArEHY5GW9NAeqt0xMcPyyPTkoUDx1UUMi3VgoeqAdTRcFQW++bfTL39ArNOLlPyyjQNNpUU0Wc89pNn+EwbBA0uorf5N4QP6/nUN/x0qR8IdVkpQvfF0rAjIfdDTk3lNhD56bTafNs2f7ziSb9dg1PLVU4fxPrcoVvvBCUeK0VZb7e2oYMJMxWy3DuW2qtcAkUTrEipZhfQP4Wplu8iDvmaLC4e0p3I+Mymeg2OZWV5FFqPw/Ia0QGOb++oeMLah3wfoltrpxDE80KGI9q9Nb5hmj/cLsgQTQPjtbuXi13giZnfkIfakWht6vlxWnO/yW/b48BYRhbddRIP7vJ1G7tISRSkUFaquZB0N5t1yfPjyXp9Qm/cebKn+VU7Y3xTs53YJYHtPQp/lEHHf4nkDZHqz0A4lc3pWXpxNLbskD3Ctp3RItDwJgnYwIzrc47Z3gYG0j990usIiLiECTsoblKLISJQ2ljyFUDfjDKzmXmydpVXyvgO8TxatsXPJfxWcGGbVu0K0hUflrvuGdMypPvVyZ1SFz+VmvhthqZ0OvXjUUe6yK0846WzEU6uxJyih1LuBUWsvUO+cA5ZB3775Ru5A4AtxZXFD0W9TKTtAoXTcJplW+nG9nbDRNa4Z8M+bScGPh5QLYhbjGZs4Gm9TgqJjHRN52ENtX9P+KXs2ZfXrNWTR+Hg20c/wfJnj+74zrJcLDISi4eH1SK569TJG02ypEF0o/YV2vQCji7sLsDzJBJpCr16iDtmRZMt9Gkh6ws37kKH/g9hWD0YycZMV/mgn+tlKkoHHOiFCn60i2vpZIL+IgE1lNOyPLrB/Pa8WcY5+Tn+HrcUbVhp8pN6f3soFHe4eEMIZq8SBwPZBQOo17WFA6FXROl7V+qpf0nroa+33FivbWxgtefeVWL+0wqlJxS09/YkM+wMi+GEMPY2nVAulagR9IuCZ+vCE+bGC2Yi+1DCXb3BQZJtX5Q77E1b1WQFfnheQKWkmCBzyh7lTtR963PDM89lb/GaBPPsq5DvZVBOjiqwrN2Kwvi+/5KGLks4dey937xnVV519I/tWGhFxpJU3044CYTXrEEN2c7WjEltduLYVwbhuy7KncllX8w5UWh5SyweC7ivLy/mZK4y574gHT+aeRiLUOTR8tQjjj5q95FfM3ZehzCMFFsljFSmia996TalFYTZinlq29XcuLLswPEtABwSgXxOaG+tAUG5S8jT2u6Y2MMzrsI8V0cnKRwZaJH0AnhWRXy4wVJYUmvz3ATdt+jvyPbIio6IMwaK0EFihKahIAyxK0Q5zgR9RiApRPUXtwIALK9VCnj8Ra3zN88AL0JBJJvgCPbBVJDH4tRcv3tQfI0CGIx6BByxBnahC52KQ5x2cgHalX3DspcCv7IQSvGOnoI4UNWafnxsrAgUbquJmE2UopdCJYaPzoHMzDWeuX198YB3VfEXSguPhIZmTFpIsxDezsQe9T5PjK2q4X9Aa6jUvmVoNE+PlM/l9IHwZQHnWhI7pg1ukT1NwdeYCuy2S1QqkUw+Wb9DN5Ecqrmmb08RCtSJzB1F9aQfDIfUpGsczdu77ZhbLk1bJkpARJbjYWnjsSG/SFgMEyjowmHs9FdY+tPVpEP3e5aZSqytMNQshQMNvQrmq4q5UAL/O1IpWSX9WpDvquzC0Y6CIG4Kued2wmuBofyb7siMw0qNYxy9HQoVh7pGQ+1YSkTFTeVcHVa17HVDNjazNRsvY0ejkB8GOTsbWrx8E4WviWZlvwRoq3+cP60twD/KKUmNzd6bOby2mdDzn0PCcrawDN99WnWSrfa4qnHUy+jnG7D3vefW+2UEFi3DWl9vbuA6yzn+8bP/5XjIQwtg2eRAFzrtPCdvwrgrWA376OPHPwnRGjipmWDVEB7daj45okpA0sK8Zlj0Oqxm58EwcI2FyIQLtRuU5kozCiu7scNwMHHvs/f2p3AWdmATQW5pLJ3FhkBmFz2eTk3HrIGBHKX5A+auZwu13uCVNqR21wQ3kSmx2JOZ66t0DgbP+HA3IrGYH00TZqlMy3VeZr7ylDBcm8eN4bmy1iY6N6uCHQq5DHksHueSz5EzETMZ+eF5opmO/ub1R6nzMHXmKhsDpfUj0zG5YrrbmC+nhmk5ny29o+Wdq/+w93pVuzEVUJrTpPTVSDJVe3P25QlRj4+tjEMCrousaN9cDvVPnGnHlgQ7qu1/Aqmq35hC3V0P9ObQpZF9XPF1PHUVFrN35mJY8TG4ZfdTL3HjzDmMCVezF83h8cAisaE7TkVh7qCsYAr0FNqKNFMJCu4jN8J708SC0OrUBWGEkhSSqosQvuG344TmqxGckE587B6NurH3SDQQhSH3URj+bGgly6BvZV+gbCTK0FK3UKPSt5KhJ1kK/Sdd6HFErgQv9vhQOVysfZNLYlMxf8XsP1ADr6E3PBx/a3ySgF3nxHc94k61xp2GJGWQMPJyeSEzrEotedhhjGyK+WJjgS33oFYKAzsUHAUVIgToQL3bC57dJ5ZJ1wvD/AyWbTmwvVAAA')}
    MISSING_AT_BUILD = [k for k, v in BUILD_SHAS.items() if v == "MISSING"]
    for _k, (_name, _sha, _b64) in BLOBS.items():
        if _sha == "MISSING":
            continue
        _data = gzip.decompress(base64.b64decode(_b64))
        _got = hashlib.sha256(_data).hexdigest()
        if _got != _sha:
            raise RuntimeError(f"embedded {_name}: sha256 {_got[:12]} != build {_sha[:12]} (corrupt notebook)")
        (MODS / _name).write_bytes(_data)
        R["shas"][_k] = _got
    R["shas"]["notebook_builder"] = BUILD_SHAS["notebook_builder"]
    if MISSING_AT_BUILD:
        R["gates"]["modules"] = "MODULE_MISSING_AT_BUILD: " + ",".join(MISSING_AT_BUILD)
        save()
        raise RuntimeError("E24 notebook built with missing modules " + str(MISSING_AT_BUILD) + ": rebuild before running")
    HOOK_SITE = WORK / "e24_hook_site"
    HOOK_SITE.mkdir(parents=True, exist_ok=True)
    (HOOK_SITE / "sitecustomize.py").write_bytes((MODS / "e24_hook.py").read_bytes())
    R["shas"]["hook_site"] = hashlib.sha256((HOOK_SITE / "sitecustomize.py").read_bytes()).hexdigest()
    assert R["shas"]["hook_site"] == R["shas"]["hook"]
    UNIT = MODS / "e24_unit_41736_41737.py"
    sys.path.insert(0, str(MODS))
    if not FAKE:
        os.environ.pop("E24_TIME_SCALE", None)
    for _m in ("e24_lab", "e24_sglang_patches", "e14_loadpath", "e14_trace_summary"):
        sys.modules.pop(_m, None)
    import e24_lab as L
    import e24_sglang_patches as P
    import e14_loadpath as E14L
    if FAKE:
        _fd = os.environ.get("E24_FAKE_DIR")
        _cands = [Path(_fd)] if _fd else [p / "experiments" / "Experiment 24 - SGLang speed track (planned)" / "src" / "harness"
                                         for p in [Path.cwd(), *Path.cwd().parents]] + [p / "src" / "harness" for p in [Path.cwd(), *Path.cwd().parents]]
        FAKE_DIR = next((c for c in _cands if (c / "fake_sglang_server.py").is_file()), None)
        if FAKE_DIR is None:
            raise FileNotFoundError("E24_FAKE_SERVER=1 needs E24_FAKE_DIR (the dir with fake_sglang_server.py / fake_vllm_server.py)")
        R["setup"]["fake_dir"] = str(FAKE_DIR)
    M = INP / "models" / "foysalemonshanto"
    D = INP / "datasets" / "foysalemonshanto"
    NEED_VLLM = SM or PART in ("A", "ALL")
    INPUT_ERR = {}
    def _inp(key, fn):
        try:
            return fn()
        except Exception as e:
            INPUT_ERR[key] = f"{type(e).__name__}: {e}"[:400]
            err("inputs/" + key, e)
            return None
    FP8 = _inp("FP8", lambda: find("config.json", M / "qwen3-8-27b-fp8-repacked-v1").parent)
    INT4 = _inp("INT4", lambda: find("config.json", M / "qwen3-8-27b-int4-redhat").parent)
    DFLASH = _inp("DFLASH", lambda: find("config.json", M / "qwen3-8-27b-dflash2").parent)
    SGL_WH = _inp("SGL_WHEELHOUSE", lambda: find("requirements.lock", D / "sglang-0-5-21-cu13-wheelhouse").parent)
    VLLM_WH = _inp("VLLM_WHEELHOUSE", lambda: find("install_wheelhouse.py", D / "vllm-0-32-0-cu13-wheelhouse")) if NEED_VLLM else None
    ARC = _inp("ARC", lambda: find("arc-agi_evaluation_challenges.json"))
    R["inputs"] = {k: (str(v) if v else None) for k, v in dict(FP8=FP8, INT4=INT4, DFLASH=DFLASH, SGL_WHEELHOUSE=SGL_WH,
                                                               VLLM_WHEELHOUSE=VLLM_WH, ARC=ARC).items()}
    R["inputs"]["errors"] = INPUT_ERR
    R["inputs"]["sizes"] = {k: _dir_size(v) for k, v in dict(FP8=FP8, INT4=INT4, DFLASH=DFLASH).items() if v}
    _need = ["FP8", "INT4", "DFLASH", "SGL_WHEELHOUSE", "ARC"] + (["VLLM_WHEELHOUSE"] if NEED_VLLM else [])
    _miss = [k for k in _need if not R["inputs"].get(k)]
    if _miss:
        ABORT["why"] = f"NO_GO_INPUT_MISSING: {_miss} (attach the inputs; a wrong model variation drops the mount silently)"
        R["gates"]["inputs"] = ABORT["why"]
    else:
        R["gates"]["inputs"] = "GO"
    for _k, _v in dict(fp8=FP8, int4=INT4, dflash2=DFLASH).items():
        if _v:
            try:
                R["identity"][_k] = L.identity(_v)
            except Exception as e:
                err("identity/" + _k, e)
            _tj = Path(_v) / "tokenizer.json"
            R["identity"].setdefault("tokenizer_sha256", {})[_k] = hashlib.sha256(_tj.read_bytes()).hexdigest()[:16] if _tj.is_file() else None
    try:
        _dc = json.loads((DFLASH / "config.json").read_text()) if DFLASH else {}
        R["identity"]["dflash2_config"] = {"architectures": _dc.get("architectures"), "num_target_layers": _dc.get("num_target_layers"),
                                           "target_layer_ids": (_dc.get("dflash_config") or {}).get("target_layer_ids"),
                                           "block_size": (_dc.get("dflash_config") or {}).get("block_size"),
                                           "taps_ok": (_dc.get("dflash_config") or {}).get("target_layer_ids") == [5, 19, 33, 47, 61]
                                           and _dc.get("num_target_layers") == 64}
        _ic = json.loads((INT4 / "config.json").read_text()) if INT4 else {}
        VOCAB = int((_ic.get("text_config") or {}).get("vocab_size") or _ic.get("vocab_size") or 248320)
        _gc = json.loads((FP8 / "generation_config.json").read_text()) if FP8 and (FP8 / "generation_config.json").is_file() else {}
        EOS_IDS = sorted({int(x) for x in (_gc.get("eos_token_id") if isinstance(_gc.get("eos_token_id"), list) else [_gc.get("eos_token_id")]) if x is not None}) or [248046, 248044]
    except Exception as e:
        err("identity/configs", e)
        VOCAB, EOS_IDS = 248320, [248046, 248044]
    R["identity"]["vocab_size"], R["identity"]["eos_ids"] = VOCAB, EOS_IDS
    L.start_watchdog(HARD_STOP, lambda: (ABORT.update(why="HARD_STOP reached: watchdog killed live servers"), save()))
    log("E24", VARIANT, "| budget", round(BUDGET / 60, 1), "min | fake", FAKE, "ts", TS, "|", json.dumps(R["inputs"])[:900])
    save()


In [ ]:
if not RERUN:
    SGL_PORT, VLLM_PORT = 30000, 8200
    SPEC_FLAGS = ("--speculative-algorithm", "--speculative-draft-model-path", "--speculative-draft-model-quantization",
                  "--speculative-num-draft-tokens", "--speculative-num-steps", "--speculative-eagle-topk")
    TGT_DROP = SPEC_FLAGS + ("--min-free-slots-delay",)
    NCCL_DBG = {"NCCL_DEBUG": "INFO", "NCCL_DEBUG_SUBSYS": "INIT,GRAPH,ENV"}
    PATCH_ALL5 = ("43567", "43559", "43029", "41736", "41737")
    PATCH_ALL7 = PATCH_ALL5 + ("44401", "41872")
    def edit_flags(flags, drop=(), set_=None, add=()):
        """EMS e14_edit_flags (cell [8] L10-25): every flag in drop | set_ is removed with its values (up to the next --),
        then set_ is appended ({flag: value | [values] | None = bare flag}), then add."""
        out, i, set_ = [], 0, dict(set_ or {})
        names = set(drop) | set(set_)
        while i < len(flags):
            if flags[i] in names:
                i += 1
                while i < len(flags) and not str(flags[i]).startswith("--"):
                    i += 1
                continue
            out.append(str(flags[i]))
            i += 1
        for k, v in set_.items():
            out += [k] + ([str(x) for x in v] if isinstance(v, (list, tuple)) else ([str(v)] if v is not None else []))
        return out + [str(a) for a in add]
    def _streams(n, slots=4):
        """n client streams: cap n + 1, decode graphs 1..n+1 (with --min-free-slots-delay 1 a shorter graph list crashes
        every TP rank, E14A), slots x (n + 1) Mamba slots (extra_buffer_lazy + overlap = 4 per request; 0.5.21 caps
        max_running at max_mamba_cache_size // 4, the source-derived minimum)."""
        return {"--max-running-requests": n + 1, "--cuda-graph-bs-decode": list(range(1, n + 2)),
                "--max-mamba-cache-size": slots * (n + 1)}
    def prod_flags(model, draft, port=None):
        """E15B R3K2 production argv (S16, DFlash W6) = the EMS / E15B engine cell e11_flags() W1 base (cell [7] L165-189,
        E15B: mrr 17, Mamba 68) + the E14 submit-arm override (L195-223: graphs 1..17, width 6, delay 1, max-bs dropped)."""
        f = ["--model-path", str(model), "--tp-size", "4", "--kv-cache-dtype", "fp8_e4m3", "--context-length", "155648",
             "--max-running-requests", "17", "--chunked-prefill-size", "4096", "--mamba-ssm-dtype", "float16", "--enable-metrics",
             "--disable-flashinfer-autotune", "--random-seed", "20261005", "--host", "127.0.0.1",
             "--port", str(SGL_PORT if port is None else port), "--log-level", "info", "--cuda-graph-max-bs-decode", "16", "--watchdog-timeout", "1800", "--decode-log-interval", "40",
             "--speculative-algorithm", "DFLASH", "--speculative-draft-model-path", str(draft),
             "--speculative-draft-model-quantization", "unquant", "--speculative-num-draft-tokens", "8",
             "--disable-prefill-cuda-graph", "--mamba-radix-cache-strategy", "extra_buffer_lazy", "--max-mamba-cache-size", "68",
             "--mem-fraction-static", "0.83", "--incremental-streaming-output", "--stream-interval", "1",
             "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
             "--default-chat-template-kwargs", json.dumps({"preserve_thinking": True}), "--schedule-policy", "lpm",
             "--enable-cache-report"]
        return edit_flags(f, drop=("--cuda-graph-max-bs-decode",),
                          set_={"--cuda-graph-bs-decode": list(range(1, 18)), "--speculative-num-draft-tokens": 6,
                                "--min-free-slots-delay": 1})
    def sgl(model, draft, streams=16, width=6, spec=True, set_=None, drop=()):
        """An SGLang arm = production + _streams(n) (+ width) + the arm's edits; target-only drops the six speculative
        flags and --min-free-slots-delay (DESIGN §2 rules)."""
        s = dict(_streams(streams))
        if spec and width != 6:
            s["--speculative-num-draft-tokens"] = width
        s.update(set_ or {})
        return edit_flags(prod_flags(model, draft), drop=tuple(drop) + (() if spec else TGT_DROP), set_=s)
    def stock_flags(model, port=None, *, tp=4, seqs=16, spec=0, kv="fp8_e4m3", backend="FLASHINFER", prefix=False,
                    mem=0.90, batched=4096, max_len=131072, extra=()):
        """E23 stock_flags (build_e23_notebook.py L300-315) with the oracle values: seqs 16, no speculation, prefix off."""
        f = [str(model), "--tensor-parallel-size", str(tp), "--kv-cache-dtype", kv, "--max-model-len", str(max_len),
             "--max-num-seqs", str(seqs), "--max-num-batched-tokens", str(batched), "--gpu-memory-utilization", str(mem),
             "--disable-custom-all-reduce", "--seed", "20261005", "--host", "127.0.0.1",
             "--port", str(VLLM_PORT if port is None else port), "--served-model-name", "q27", "--attention-backend", backend, "--trust-remote-code", "--language-model-only"]
        f += ["--enable-prefix-caching"] if prefix else ["--no-enable-prefix-caching"]
        if spec:
            f += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": int(spec)})]
        return f + list(extra)
    BASE_DESIGN = ("--model-path <INT4> --tp-size 4 --kv-cache-dtype fp8_e4m3 --context-length 155648 --chunked-prefill-size 4096 "
                   "--mamba-ssm-dtype float16 --enable-metrics --disable-flashinfer-autotune --random-seed 20261005 --host 127.0.0.1 "
                   "--port <P> --log-level info --watchdog-timeout 1800 --decode-log-interval 40 --speculative-algorithm DFLASH "
                   "--speculative-draft-model-path <DFLASH> --speculative-draft-model-quantization unquant "
                   "--disable-prefill-cuda-graph --mamba-radix-cache-strategy extra_buffer_lazy --mem-fraction-static 0.83 "
                   "--incremental-streaming-output --stream-interval 1 --reasoning-parser qwen3 --tool-call-parser qwen3_coder "
                   "--default-chat-template-kwargs <CTK> --schedule-policy lpm --enable-cache-report "
                   "--speculative-num-draft-tokens 6 --min-free-slots-delay 1 --max-running-requests 21 --cuda-graph-bs-decode "
                   + " ".join(str(b) for b in range(1, 22)) + " --max-mamba-cache-size 84").split()
    def flag_pairs(argv):
        out, cur = [], None
        for a in argv:
            a = str(a)
            if a.startswith("--"):
                cur = [a]
                out.append(cur)
            elif cur is not None:
                cur.append(a)
            else:
                out.append(["<positional>", a])
        return sorted(tuple(x) for x in out)
    def base_check(model, draft):
        base = sgl(model, draft, streams=20)
        sub = {"<INT4>": str(model), "<DFLASH>": str(draft), "<P>": str(SGL_PORT), "<CTK>": json.dumps({"preserve_thinking": True})}
        want = [sub.get(t, t) for t in BASE_DESIGN]
        return {"ok": flag_pairs(base) == flag_pairs(want), "exact_order": base == want, "n_tokens": len(base)}
    def small(interval, slots=4):
        """E17 e17_small: the upstream small-slot recipe (4 Mamba slots, 1 running)."""
        return {"--max-mamba-cache-size": slots, "--max-running-requests": 1, "--mamba-track-interval": interval,
                "--chunked-prefill-size": 2048, "--cuda-graph-bs-decode": [1]}
    NGRAM_SET = {"--speculative-algorithm": "NGRAM", "--speculative-num-draft-tokens": 4, "--speculative-ngram-max-bfs-breadth": 1,
                 "--mamba-radix-cache-strategy": "extra_buffer", "--max-mamba-cache-size": 32, "--max-running-requests": 4,
                 "--cuda-graph-bs-decode": [1, 2, 3, 4], "--random-seed": 1, "--grammar-backend": "xgrammar"}
    NGRAM_DROP = ("--speculative-draft-model-path", "--speculative-draft-model-quantization", "--min-free-slots-delay")
    GDF_SET = {"--mamba-radix-cache-strategy": "extra_buffer", "--max-mamba-cache-size": 32, "--max-running-requests": 4,
               "--cuda-graph-bs-decode": [1, 2, 3, 4], "--random-seed": 1, "--grammar-backend": "xgrammar"}
    S2_CHALLENGERS = ("s2_w4s20", "s2_w6s16", "s2_w4s16", "s2_w5s20", "s2_w7s20")
    S3_ARMS = ("s3_ctrl", "s3_cta2", "s3_cta4", "s3_cta8", "s3_ring", "s3_tree", "s3_simple", "s3_ll", "s3_nthreads512")
    S3_ENV = {"s3_ctrl": {}, "s3_cta2": {"NCCL_MIN_CTAS": "2", "NCCL_MAX_CTAS": "2"}, "s3_cta4": {"NCCL_MIN_CTAS": "4", "NCCL_MAX_CTAS": "4"},
              "s3_cta8": {"NCCL_MIN_CTAS": "8", "NCCL_MAX_CTAS": "8"}, "s3_ring": {"NCCL_ALGO": "allreduce:ring"},
              "s3_tree": {"NCCL_ALGO": "allreduce:tree"}, "s3_simple": {"NCCL_PROTO": "Simple"}, "s3_ll": {"NCCL_PROTO": "LL"},
              "s3_nthreads512": {"NCCL_NTHREADS": "512"}}
    def catalog(int4, dflash, fp8, variant):
        """Every arm a variant can launch (DESIGN rev 2 §4): engine, server flags (after the launch prefix), arm env, cache
        policy (§3.3: fresh / ("reuse", arm) / ("copy", arm) / "shared"), expected patch set (§3.4), hook options."""
        A = {}
        full_b = variant in ("FULL_B", "FULL_ALL")
        warm_d6 = "shared" if full_b and variant == "FULL_B" else ("copy", "s1_d6")
        def add(arm, sec, engine, flags, env=None, cache="fresh", patches=(), hook=None, **kw):
            A[arm] = dict(arm=arm, section=sec, engine=engine, flags=[str(x) for x in flags], env=dict(env or {}), cache=cache,
                          patches=tuple(patches), hook=dict(hook or {}), **kw)
        def S(**k):
            return sgl(int4, dflash, **k)
        probe = {"probe": 1}
        add("s1_a", "1", "vllm", stock_flags(fp8), hook=probe)
        add("s1_b", "1", "vllm", stock_flags(int4), hook=probe)
        add("s1_d6", "1", "sglang", S(), hook=probe)
        add("s1_d6fr", "1", "sglang", S(), env={"SGLANG_SIMULATE_ACC_LEN": "1", "SGLANG_SIMULATE_ACC_TOKEN_MODE": "real-draft-token"},
            hook=probe, greedy_only=True)
        add("s1_d4", "1", "sglang", S(width=4), hook=probe)
        add("s1_c", "1", "sglang", S(spec=False), hook=probe)
        add("s1_c_warm", "1", "sglang", S(spec=False), cache=("reuse", "s1_c"), hook=probe)
        add("s1_c_nocache", "1", "sglang", S(spec=False, set_={"--disable-radix-cache": None}), cache=("reuse", "s1_c"), hook=probe)
        add("s1_c32", "1", "sglang", S(spec=False, set_={"--mamba-ssm-dtype": "float32"}), hook=probe)
        add("s1_d6_r2", "1", "sglang", S(), hook=probe)
        add("s1_c_r2", "1", "sglang", S(spec=False), hook=probe)
        sec2 = {"s2_w6s20": S(streams=20), "s2_w6s20_aa": S(streams=20), "s2_w4s20": S(streams=20, width=4),
                "s2_w6s16": S(streams=16), "s2_w4s16": S(streams=16, width=4), "s2_w5s20": S(streams=20, width=5),
                "s2_w7s20": S(streams=20, width=7)}
        for a, f in sec2.items():
            add(a, "2", "sglang", f, cache=warm_d6)
        for c in S2_CHALLENGERS:
            add("s2_best_long__" + c, "2", "sglang", sec2[c], cache=warm_d6, base_arm=c)
        for i in (1, 2, 3):
            add(f"aa_ctrl_r{i}", "2", "sglang", sec2["s2_w6s20"], cache=warm_d6, base_arm="s2_w6s20")
            for c in S2_CHALLENGERS:
                add(f"aa_best_r{i}__{c}", "2", "sglang", sec2[c], cache=warm_d6, base_arm=c)
        for a in S3_ARMS:
            add(a, "3", "sglang", S(streams=20), env=dict(NCCL_DBG, **S3_ENV[a]), cache=warm_d6)
            add("s3_conf__" + a, "3", "sglang", S(streams=20), env=dict(NCCL_DBG, **S3_ENV[a]), cache=warm_d6, base_arm=a)
        add("s3_ctrl_rerun", "3", "sglang", S(streams=20), env=dict(NCCL_DBG), cache=warm_d6, base_arm="s3_ctrl")
        cnt = {"counters": 1}
        add("s4_m84", "4", "sglang", S(streams=20), cache=warm_d6, hook=cnt)
        add("s4_m68", "4", "sglang", S(streams=16), cache=warm_d6, hook=cnt)
        add("s4_mem080", "4", "sglang", S(streams=20, set_={"--mem-fraction-static": "0.80"}), cache=warm_d6, hook=cnt)
        add("s4_mem086", "4", "sglang", S(streams=20, set_={"--mem-fraction-static": "0.86"}), cache=warm_d6, hook=cnt)
        add("s4_m88", "4", "sglang", S(streams=21), cache=warm_d6, hook=cnt)
        add("s4_m92", "4", "sglang", S(streams=22), cache=warm_d6, hook=cnt)
        add("s4_lowcap", "4", "sglang", S(streams=20, set_={"--max-total-tokens": 300000}), cache=warm_d6, hook=cnt)
        add("s5_c4096", "5", "sglang", S(streams=20), cache=warm_d6)
        add("s5_c2048", "5", "sglang", S(streams=20, set_={"--chunked-prefill-size": 2048}), cache=warm_d6)
        add("s5_c8192", "5", "sglang", S(streams=20, set_={"--chunked-prefill-size": 8192}), cache=warm_d6)
        add("s5_pdi1", "5", "sglang", S(streams=20, set_={"--prefill-decode-interval": 1}), cache=warm_d6)
        add("s5_pdi4", "5", "sglang", S(streams=20, set_={"--prefill-decode-interval": 4}), cache=warm_d6)
        add("s5_mixed", "5", "sglang", S(streams=20, set_={"--enable-mixed-chunk": None}), cache=warm_d6)
        c_c, c_d = ("copy", "s1_c"), ("copy", "s1_d6")
        ff = {"SGLANG_TEST_MAMBA_LAZY_ALLOC_FAIL": "1"}
        add("s6_unit", "6", "sglang", [], cache=c_d, kind="unit")
        add("s6_ffu", "6", "sglang", S(spec=False, set_=small(128)), env=ff, cache=c_c, hook=cnt)
        add("s6_ffp", "6", "sglang", S(spec=False, set_=small(128)), env=ff, cache=c_c, patches=("43567", "43559"), hook=cnt)
        add("s6_np", "6", "sglang", S(spec=False, set_=small(128)), cache=c_c, patches=("43567", "43559"), hook=cnt)
        add("s6_ffud", "6", "sglang", S(set_=small(64)), env=ff, cache=c_d, hook=cnt)
        add("s6_ffpd", "6", "sglang", S(set_=small(64)), env=ff, cache=c_d, patches=("43567", "43559"), hook=cnt)
        ng = edit_flags(prod_flags(int4, dflash), drop=NGRAM_DROP, set_=NGRAM_SET)
        add("s6_gu", "6", "sglang", ng, cache=c_c, hook=cnt)
        add("s6_gp", "6", "sglang", ng, cache=c_c, patches=("43029",), hook=cnt)
        gdf = edit_flags(prod_flags(int4, dflash), set_=GDF_SET)
        add("s6_gu_df", "6", "sglang", gdf, cache=c_d, hook=cnt)
        add("s6_gp_df", "6", "sglang", gdf, cache=c_d, patches=("43029",), hook=cnt)
        gl = S(streams=3, set_={"--random-seed": 1, "--grammar-backend": "xgrammar"})
        add("s6_gl_u", "6", "sglang", gl, cache=c_d, hook=cnt)
        add("s6_gl_p", "6", "sglang", gl, cache=c_d, patches=("43029",), hook=cnt)
        eager = S(set_={"--disable-decode-cuda-graph": None})
        inj_d = {"E24_INJECT_DRAFT_NAN": "1"}
        inj_s = {"E24_INJECT_SELECTOR_NAN": "1"}
        add("s6_sent_u", "6", "sglang", eager, env=inj_d, cache=c_d, patches=("41737",), hook=dict(cnt, eager=1))
        add("s6_sent_p", "6", "sglang", eager, env=inj_d, cache=c_d, patches=("41737", "41736"), hook=dict(cnt, eager=1))
        add("s6_nan_u", "6", "sglang", eager, env=inj_s, cache=c_d, hook=dict(cnt, eager=1))
        add("s6_nan_p", "6", "sglang", eager, env=inj_s, cache=c_d, patches=("41737",), hook=dict(cnt, eager=1))
        add("s6_combined", "6", "sglang", S(), cache=c_d, patches=PATCH_ALL5, hook=probe)
        return A
    PLANS = {
        "SMOKE": [("1", "s1_a", 19), ("1", "s1_b", 16), ("1", "s1_d6", 21), ("1", "s1_d6fr", 12), ("1", "s1_d4", 11),
                  ("1", "s1_c", 21), ("6", "s6_unit", 6), ("6", "s6_ffu", 8), ("6", "s6_ffp", 8), ("2", "s2_w6s20", 25),
                  ("2", "s2_w4s20", 7), ("3", "s3_cta4", 8)],
        "FULL_A": [("1", "s1_a", 42), ("1", "s1_b", 24), ("1", "s1_d6", 38), ("1", "s1_d6fr", 17), ("1", "s1_d4", 35),
                   ("1", "s1_c", 40), ("1", "s1_c_warm", 13), ("1", "s1_c_nocache", 19), ("1", "s1_c32", 38),
                   ("1", "s1_d6_r2", 19), ("6", "s6_unit", 6), ("6", "s6_ffu", 8), ("6", "s6_ffp", 8), ("6", "s6_np", 8),
                   ("6", "s6_ffud", 8), ("6", "s6_ffpd", 9), ("6", "s6_gu", 10), ("6", "s6_gp", 7),
                   ("6", "s6_gu_df?", 8), ("6", "s6_gp_df?", 8), ("6", "s6_sent_u", 8), ("6", "s6_sent_p", 8),
                   ("6", "s6_nan_u", 6), ("6", "s6_nan_p", 6), ("6", "s6_combined", 10), ("1", "s1_c_r2", 17),
                   ("6", "s6_gl_u?", 12), ("6", "s6_gl_p?", 6)],
        "FULL_B": [("2", "s2_w6s20", 57), ("2", "s2_w6s20_aa", 18), ("2", "s2_w4s20", 23), ("2", "s2_w6s16", 19),
                   ("2", "s2_w4s16", 19), ("2", "s2_w5s20|s2_w7s20", 19), ("2", "s2_best_long?", 24)]
                  + [("3", a, 10) for a in S3_ARMS] + [("3", "s3_conf__best1?", 20), ("3", "s3_conf__best2?", 20),
                                                         ("3", "s3_ctrl_rerun", 9)]
                  + [("4", "s4_m84", 35), ("4", "s4_m68", 26), ("4", "s4_mem080", 26), ("4", "s4_mem086", 29),
                     ("4", "s4_m88?", 26), ("4", "s4_m92?", 26)]
                  + [("5", "s5_c4096", 25), ("5", "s5_c2048", 25), ("5", "s5_c8192", 25), ("5", "s5_pdi1", 25), ("5", "s5_pdi4", 25)]
                  + [("2", "aa_ctrl_r%d" % i, 18) for i in (1, 2, 3)]
                  + [("4", "s4_lowcap?", 14), ("5", "s5_mixed?", 25)],
    }
    PLANS["FULL_ALL"] = PLANS["FULL_A"] + PLANS["FULL_B"]
    def plan_arms(variant):
        """Every arm id a variant can launch, both branches of every conditional (for plan_argvs_e24.json / the kit)."""
        out = []
        for sec, a, _ in PLANS[variant]:
            a = a.rstrip("?")
            if a == "s2_w5s20|s2_w7s20":
                out += ["s2_w5s20", "s2_w7s20"]
            elif a == "s2_best_long":
                out += ["s2_best_long__" + c for c in S2_CHALLENGERS]
            elif a.startswith("s3_conf__best"):
                out += [] if "s3_conf__s3_ctrl" in out else ["s3_conf__" + x for x in S3_ARMS]
            elif a.startswith("aa_ctrl_r"):
                out += [a] + [f"aa_best_r{a[-1]}__{c}" for c in S2_CHALLENGERS]
            else:
                out.append(a)
        return list(dict.fromkeys(out))
    import shlex
    if FAKE and os.environ.get("E24_PORT_BASE"):
        SGL_PORT, VLLM_PORT = int(os.environ["E24_PORT_BASE"]), int(os.environ["E24_PORT_BASE"]) + 1
    PLAN = PLANS[VARIANT]
    PLANNED = set(plan_arms(VARIANT))
    R["plan"] = [list(x) for x in PLAN]
    def run(name, cmd, timeout=3600, env=None):
        t = time.time()
        try:
            with open(LOGS / f"{name}.log", "w") as f:
                r = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], stdout=f, stderr=subprocess.STDOUT, timeout=timeout,
                                   env=env or dict(os.environ))
            rc = r.returncode
        except subprocess.TimeoutExpired:
            rc = -9
        txt = (LOGS / f"{name}.log").read_text(errors="replace")
        R["setup"][name] = {"rc": rc, "sec": round(time.time() - t, 1), "tail": txt[-1500:]}
        log(name, "rc", rc, f"{time.time() - t:.0f}s")
        save()
        return rc, txt
    SHIM = WORK / "e24_libcuda"
    SHIM.mkdir(exist_ok=True)
    for _d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/compat", "/usr/lib64"):
        _c = sorted(Path(_d).glob("libcuda.so.1")) if Path(_d).is_dir() else []
        if _c and not (SHIM / "libcuda.so").exists():
            (SHIM / "libcuda.so").symlink_to(_c[0])
    _KNOBS = ("SGLANG_TEST_", "SGLANG_SIMULATE_", "E24_INJECT_", "E24_HOOK_", "E24_SENTINEL")
    R["setup"]["inherited_knobs_dropped"] = sorted(k for k in os.environ if k.startswith(_KNOBS))
    R["setup"]["inherited_nccl"] = {k: v for k, v in os.environ.items() if k.startswith("NCCL_")}
    def clean_env(src):
        """The server env base: no inherited PYTHONPATH / CUDA_* / LD_PRELOAD (EMS e11_install) and never an inherited
        test / simulate / injection / hook knob (DESIGN §2: those exist only in the arms that set them)."""
        return {k: v for k, v in src.items() if k not in ("PYTHONPATH", "CUDA_HOME", "CUDA_PATH", "LD_PRELOAD")
                and not k.startswith(_KNOBS)}
    SGL_FIXED = {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",
                 "CUDA_DEVICE_ORDER": "PCI_BUS_ID", "PYTHONUNBUFFERED": "1", "TORCH_CUDA_ARCH_LIST": "8.9", "MAX_JOBS": "4",
                 "SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK": "0",
                 "SGLANG_LOGPROB_CHUNK_SIZE": "128"}
    VLLM_FIXED = {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "VLLM_NO_USAGE_STATS": "1",
                  "VLLM_ALLOW_LONG_MAX_MODEL_LEN": "1", "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1"}
    def sgl_install():
        """= the EMS / E15B engine cell e11_install() (cell [7] L73-156): a --without-pip venv from the wheelhouse lock
        (--no-deps, --only-binary), '+'-stripped wheel names rebuilt from SHA256SUMS (sha256-checked), the pip CUDA 13
        runtime as CUDA_HOME with the libcudart.so / lib64 / -lcuda shims. Raises on failure (the caller caches it)."""
        if FAKE:
            fs = str(FAKE_DIR / "fake_sglang_server.py")
            return {"vpy": sys.executable, "cmd": [sys.executable, fs], "env": dict(clean_env(os.environ), **SGL_FIXED),
                    "fake_server": fs, "fake": True}
        shims, t0 = {}, time.perf_counter()
        venv = TMP / "sglang-venv"
        vpy = venv / "bin" / "python"
        fl = TMP / "e24-wheels"
        fl.mkdir(parents=True, exist_ok=True)
        renamed = []
        for line in (SGL_WH / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            h, name = line.split("  ", 1)
            name = name.strip()
            if not name.endswith(".whl"):
                continue
            s = SGL_WH / name if (SGL_WH / name).exists() else SGL_WH / name.replace("+", "")
            if not s.exists():
                raise RuntimeError(f"wheel missing from the dataset: {name}")
            if s.name != name:
                d = hashlib.sha256()
                with open(s, "rb") as f:
                    for b in iter(lambda: f.read(16 * 1024 * 1024), b""):
                        d.update(b)
                if d.hexdigest() != h:
                    raise RuntimeError(f"sha256 mismatch for {s.name}")
                renamed.append(f"{s.name} -> {name}")
            if not (fl / name).exists():
                (fl / name).symlink_to(s)
        shims["renamed_wheels"] = renamed
        if not vpy.exists():
            subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True, timeout=300)
            r = subprocess.run([sys.executable, "-m", "pip", "--python", str(vpy), "install", "--no-index", "--find-links", str(fl),
                                "--requirement", str(SGL_WH / "requirements.lock"), "--no-deps", "--only-binary", ":all:",
                                "--no-compile", "--disable-pip-version-check", "--no-warn-conflicts", "-q"],
                               capture_output=True, text=True, timeout=1800)
            (LOGS / "install_sglang.log").write_text((r.stdout or "") + (r.stderr or ""))
            if r.returncode != 0:
                raise RuntimeError("offline SGLang install failed: " + (r.stderr or r.stdout)[-1500:])
        site = Path(subprocess.run([str(vpy), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                                   capture_output=True, text=True, check=True, timeout=120).stdout.strip())
        cuda_home = site / "nvidia" / "cu13"
        if not (cuda_home / "lib" / "libcudart.so").exists():
            (cuda_home / "lib" / "libcudart.so").symlink_to("libcudart.so.13")
            shims["libcudart.so"] = "created"
        if not (cuda_home / "lib64").exists():
            (cuda_home / "lib64").symlink_to("lib")
            shims["lib64"] = "created"
        libcuda = TMP / "e24-libcuda"
        have = [d for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs", "/usr/lib/x86_64-linux-gnu", "/usr/lib64")
                if Path(d, "libcuda.so").exists()]
        if have:
            shims["libcuda.so"] = f"exists in {have[0]}"
        else:
            real = None
            try:
                for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                    if "libcuda.so.1" in line and "=>" in line:
                        real = line.split("=>")[-1].strip()
                        break
            except Exception:
                pass
            for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
                if real is None and Path(d, "libcuda.so.1").exists():
                    real = str(Path(d, "libcuda.so.1"))
            if real:
                libcuda.mkdir(parents=True, exist_ok=True)
                if (libcuda / "libcuda.so").is_symlink() or (libcuda / "libcuda.so").exists():
                    (libcuda / "libcuda.so").unlink()
                (libcuda / "libcuda.so").symlink_to(Path(real).resolve())
                shims["libcuda.so"] = f"shim -> {real}"
            else:
                shims["libcuda.so"] = "libcuda.so.1 NOT FOUND"
        env = clean_env(os.environ)
        env.update({"PATH": os.pathsep.join([str(venv / "bin"), str(cuda_home / "bin"), os.environ.get("PATH", "")]),
                    "CUDA_HOME": str(cuda_home), "CUDA_PATH": str(cuda_home),
                    "LIBRARY_PATH": os.pathsep.join(x for x in (str(libcuda), str(SHIM), str(cuda_home / "lib"),
                                                                os.environ.get("LIBRARY_PATH", "")) if x),
                    "LD_LIBRARY_PATH": os.pathsep.join(x for x in (str(cuda_home / "lib"), os.environ.get("LD_LIBRARY_PATH", "")) if x)})
        env.update(SGL_FIXED)
        ver = subprocess.run([str(vpy), "-c", "import importlib.metadata as m; print(m.version('sglang'))"], capture_output=True,
                             text=True, timeout=120, env=env).stdout.strip()
        return {"vpy": str(vpy), "cmd": [str(vpy), "-m", "sglang.launch_server"], "env": env, "version": ver, "site": str(site),
                "cuda_home": str(cuda_home), "shims": shims, "sec": round(time.perf_counter() - t0, 1)}
    def vllm_install():
        """= E23 install_stock (the dataset's install_wheelhouse.py -> VENV_PYTHON / CUDA_HOME), failure cached by the caller."""
        if FAKE:
            fs = str(FAKE_DIR / "fake_vllm_server.py")
            return {"vpy": sys.executable, "cmd": [sys.executable, fs, "serve"], "env": dict(clean_env(os.environ), **VLLM_FIXED),
                    "fake_server": fs, "fake": True}
        rc, txt = run("install_vllm", f"{shlex.quote(sys.executable)} {shlex.quote(str(VLLM_WH))} {shlex.quote(str(TMP / 'vllm-venv'))}",
                      timeout=2400)
        kv = dict(l.split("=", 1) for l in txt.splitlines() if l.startswith(("VENV_PYTHON=", "CUDA_HOME=")))
        if rc or "CUDA_HOME" not in kv or "VENV_PYTHON" not in kv:
            raise RuntimeError(f"vLLM 0.32 install failed rc={rc} (logs/install_vllm.log)")
        env = clean_env(os.environ)
        env.update(CUDA_HOME=kv["CUDA_HOME"], PATH=f"{TMP}/vllm-venv/bin:{kv['CUDA_HOME']}/bin:{env.get('PATH', '')}",
                   LIBRARY_PATH=f"{SHIM}:{env.get('LIBRARY_PATH', '')}", **VLLM_FIXED)
        ver = subprocess.run([kv["VENV_PYTHON"], "-c", "import importlib.metadata as m; print(m.version('vllm'))"],
                             capture_output=True, text=True, env=env, timeout=300).stdout.strip()
        return {"vpy": kv["VENV_PYTHON"], "cmd": [f"{TMP}/vllm-venv/bin/vllm", "serve"], "env": env, "version": ver}
    if FAKE:
        os.environ["E14_LOADPATH"] = "0"
    INT4_RUN, DFLASH_RUN = INT4, DFLASH
    STAGING = None
    if not ABORT["why"]:
        try:
            STAGING = E14L.Staging([INT4, DFLASH], workers=24, stall_sec=600.0, deadline=time.perf_counter() + 40 * 60, log=log,
                                   dest_candidates=tuple(x for x in os.environ.get("E14_DEST", "/dev/shm,/tmp").split(",") if x))
            STAGING.start()
        except Exception as e:
            err("staging/start", e)
    ENGINES = {"sglang": {"failed": "not installed"}, "vllm": {"failed": "not installed"}}
    if not ABORT["why"]:
        try:
            ENGINES["sglang"] = sgl_install()
            R["setup"]["sglang_install"] = {k: v for k, v in ENGINES["sglang"].items() if k not in ("env",)}
        except Exception as e:
            ENGINES["sglang"] = {"failed": f"{type(e).__name__}: {e}"[:600]}
            R["setup"]["sglang_install"] = ENGINES["sglang"]
            err("install/sglang", e)
        save()
    if STAGING is not None:
        STAGING.join(timeout=max(0.0, 45 * 60 - (time.time() - T_START)))
        res = STAGING.result
        if res is None:
            R["setup"]["staging"] = {"still_running": True, "note": "launch from the /kaggle/input paths (fail open, as EMS)"}
        else:
            _paths, _rep = res
            INT4_RUN = Path(_paths.get(str(INT4), str(INT4)))
            DFLASH_RUN = Path(_paths.get(str(DFLASH), str(DFLASH)))
            R["setup"]["staging"] = json.loads(json.dumps(_rep, default=str))
            for _k, _v in (("int4_shard_sha256", INT4), ("dflash2_shard_sha256", DFLASH)):
                R["identity"][_k] = ((_rep.get("copies") or {}).get(str(_v)) or {}).get("sha256") or "NOT_STAGED"
        R["setup"]["launch_paths"] = {"int4": str(INT4_RUN), "dflash2": str(DFLASH_RUN)}
        save()
    if NEED_VLLM and not ABORT["why"]:
        if "s1_a" in PLANNED and not FAKE and FP8:
            def _fp8_prefetch():
                res = L.prefetch([FP8])
                with _SAVE_LOCK:
                    R["setup"]["fp8_prefetch"] = res
            threading.Thread(target=_fp8_prefetch, daemon=True).start()
        try:
            ENGINES["vllm"] = vllm_install()
            R["setup"]["vllm_install"] = {k: v for k, v in ENGINES["vllm"].items() if k not in ("env",)}
        except Exception as e:
            ENGINES["vllm"] = {"failed": f"{type(e).__name__}: {e}"[:600]}
            R["setup"]["vllm_install"] = ENGINES["vllm"]
            err("install/vllm", e)
    elif not NEED_VLLM:
        ENGINES["vllm"] = {"failed": "no vLLM arm in this variant"}
    CAT = catalog(INT4_RUN, DFLASH_RUN, FP8, VARIANT)
    R["setup"]["base_check"] = base_check(INT4_RUN, DFLASH_RUN)
    assert R["setup"]["base_check"]["ok"], R["setup"]["base_check"]
    PSITE, PATCH_BROKEN = None, None
    try:
        if FAKE:
            PSITE = os.environ.get("E24_FAKE_SGLANG_SITE") or None
        elif not ENGINES["sglang"].get("failed"):
            PSITE = P.site_dir(ENGINES["sglang"]["vpy"])
        R["setup"]["patch_site"] = PSITE or ("SKIPPED_FAKE" if FAKE else None)
        if PSITE:
            _chk = P.check(PSITE, names=list(PATCH_ALL7))
            R["setup"]["patch_state_start"] = _chk
            if not P.state_ok(_chk, expect_applied=()):
                R["setup"]["patch_revert_start"] = P.revert(PSITE, names=[n for n, s in (_chk.get("names") or {}).items() if s != "not_applied"])
                if not P.state_ok(P.check(PSITE, names=list(PATCH_ALL7)), expect_applied=()):
                    PATCH_BROKEN = "the installed sglang is not the pristine 0.5.21 at start and the revert failed"
    except Exception as e:
        PATCH_BROKEN = f"patch check failed: {type(e).__name__}: {e}"[:300]
        err("patches/start", e)
    R["setup"]["patch_broken"] = PATCH_BROKEN
    log("engines:", {k: (v.get("failed") or v.get("version") or "ok") for k, v in ENGINES.items()}, "| staged:",
        R["setup"].get("launch_paths"), "| patch site:", R["setup"].get("patch_site"))
    save()


In [ ]:
if not RERUN:
    import types
    def _bytes_to_unicode():
        bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
        cs, n = bs[:], 0
        for b in range(256):
            if b not in bs:
                bs.append(b)
                cs.append(256 + n)
                n += 1
        return dict(zip(bs, (chr(c) for c in cs)))
    class FakeTok:
        """FAKE only, when the local python has no `tokenizers`: a deterministic byte-level tokenizer built from the
        tokenizer.json it is given (added tokens = single special ids; every other UTF-8 byte -> its byte-level vocab id).
        On the mock tree's byte-level BPE (no merges) this gives the ids HF tokenizers would give. Never used on Kaggle."""
        def __init__(self, path):
            j = json.loads(Path(path).read_text(encoding="utf-8"))
            self.special = {t["content"]: int(t["id"]) for t in (j.get("added_tokens") or [])}
            voc = (j.get("model") or {}).get("vocab") or {}
            b2u = _bytes_to_unicode()
            self.byte_id = [int(voc.get(b2u[b], b)) if isinstance(voc, dict) else b for b in range(256)]
            alt = "|".join(re.escape(s) for s in sorted(self.special, key=len, reverse=True))
            self._re = re.compile("(" + alt + ")") if alt else None
        def token_to_id(self, t):
            return self.special.get(t)
        def encode(self, text, add_special_tokens=False):
            ids = []
            for p in (self._re.split(text) if self._re else [text]):
                if p in self.special:
                    ids.append(self.special[p])
                elif p:
                    ids.extend(self.byte_id[b] for b in p.encode("utf-8"))
            return types.SimpleNamespace(ids=ids)
    PROMPTS, MAX_NEW, ORACLE, C16, TRACE_P, M4_P, EMS, SHARED4, P2K = {}, {}, [], [], {}, [], {}, [], []
    BANDS = {"8k": 8192, "40k": 40000, "90k": 90000, "120k": 116000}
    NSTREAM = 24
    NTOK = 512 if not SM else 256
    TRACE_NEW = 30000 if not SM else 4000
    M4_NEW = 15000 if not SM else 4000
    TOK_INFO = {}
    if not ABORT["why"]:
        try:
            try:
                from tokenizers import Tokenizer as _HFTok
                TOK, TOK4 = _HFTok.from_file(str(FP8 / "tokenizer.json")), _HFTok.from_file(str(INT4 / "tokenizer.json"))
                TOK_INFO["kind"] = "tokenizers"
            except ImportError:
                if not FAKE:
                    raise
                TOK, TOK4 = FakeTok(FP8 / "tokenizer.json"), FakeTok(INT4 / "tokenizer.json")
                TOK_INFO["kind"] = "FAKE_byte_level (no `tokenizers` in this python)"
            _PIECES = {}
            def enc(text):
                """FP8 ids; the INT4 tokenizer must give the same ids on every piece (so on every prompt built from them)."""
                a = TOK.encode(text, add_special_tokens=False).ids
                if text not in _PIECES:
                    _PIECES[text] = a == TOK4.encode(text, add_special_tokens=False).ids
                return list(a)
            _tasks = json.load(open(ARC))
            _ids = sorted(_tasks)
            def _grid(g):
                return "\n".join("".join(str(v) for v in row) for row in g)
            def _task_text(k):
                t = _tasks[k]
                parts = [f"## Task {k}"]
                for i, ex in enumerate(t["train"]):
                    parts.append(f"Example {i + 1} input:\n{_grid(ex['input'])}\nExample {i + 1} output:\n{_grid(ex['output'])}")
                for ex in t["test"]:
                    parts.append(f"Test input:\n{_grid(ex['input'])}")
                return "\n\n".join(parts)
            _HEAD = enc("<|im_start|>user\nFind the transformation rule of each ARC task below and solve the test inputs.\n\n")
            _TAIL = enc("\n<|im_end|>\n<|im_start|>assistant\n<think>\n")
            for _t in ("<|im_start|>", "<|im_end|>", "<think>"):
                _id = TOK.token_to_id(_t)
                assert _id is not None and _id in (_HEAD + _TAIL), (_t, _id)
            _TASK_IDS = {k: enc(_task_text(k) + "\n\n") for k in _ids}
            def prompt(band, variant):
                """E23 CELL 3: exactly `band` tokens of DISTINCT ARC tasks (stream `variant` starts at its own task offset)."""
                body, i = [], (variant * 7) % len(_ids)
                need = band - len(_HEAD) - len(_TAIL)
                while len(body) < need:
                    body += _TASK_IDS[_ids[i % len(_ids)]]
                    i += 1
                return _HEAD + body[:need] + _TAIL
            PROMPTS = {b: [prompt(n, v) for v in range(NSTREAM)] for b, n in BANDS.items()}
            MAX_NEW = {b: MAX_LEN - n - 512 for b, n in BANDS.items()}
            P2K = [prompt(2048, v) for v in range(16)]
            _ov = {"8k": (0, 1, 2, 3), "40k": (4, 5), "90k": (6, 7), "120k": (8,)} if not SM else {"8k": (0, 1, 2), "40k": (4,)}
            ORACLE = [(f"{b}#{v}", PROMPTS[b][v]) for b, vs in _ov.items() for v in vs]
            C16 = [(f"8k#{v}", PROMPTS["8k"][v]) for v in (0, 1, 2, 3) + tuple(range(9, 21))]
            TRACE_P = {"t8k_a": PROMPTS["8k"][21], "t8k_b": PROMPTS["8k"][22]}
            if not SM:
                TRACE_P["t90k"] = PROMPTS["90k"][23]
            M4_P = [p for _, p in C16]
            PREFLIGHT = (PROMPTS["8k"][21], PROMPTS["8k"][20][-4096:])
            _ASK = "Find the transformation rule of this ARC-AGI-2 task and describe it, then give the test output grids."
            def _ems_text(k):
                t = _tasks[k]
                rows = lambda g: "\n".join(" ".join(str(v) for v in r) for r in g)
                s = "".join(f"\nExample {i + 1}\nInput:\n{rows(p['input'])}\nOutput:\n{rows(p['output'])}\n" for i, p in enumerate(t["train"]))
                return s + "".join(f"\nTest input {j + 1}:\n{rows(q['input'])}\n" for j, q in enumerate(t["test"]))
            def _chat(user, system=None):
                ids = enc("<|im_start|>system\n" + system + "<|im_end|>\n") if system else []
                return ids + enc("<|im_start|>user\n" + user + "<|im_end|>\n<|im_start|>assistant\n<think>\n")
            _stream_tasks = _ids[:20]
            _bg = [k for k in _ids if k not in _stream_tasks]
            def _block(limit, start=0):
                txt, n, i = [], 0, start
                while n < limit and i < start + len(_bg) * 3:
                    t = _ems_text(_bg[i % len(_bg)])
                    txt.append(t)
                    n += len(enc(t))
                    i += 1
                return "Reference tasks (not to be solved):\n" + "\n".join(txt)
            _b36, _b85 = _block(36000), _block(85000)
            EMS = {"short": [_chat(_ASK + "\n" + _ems_text(k)) for k in _stream_tasks],
                   "long": [_chat(_ASK + "\n" + _ems_text(k), _b36) for k in _stream_tasks],
                   "xl": [_chat(_ASK + "\n" + _ems_text(k), _b85) for k in _stream_tasks]}
            SHARED4 = [_chat(_ASK + "\n" + _ems_text(_stream_tasks[(4 * g + s) % len(_stream_tasks)]), _block(85000, start=7 * (g + 1)))
                       for g in range(4) for s in range(4)]
            S6P = {"W44": [44] * 128, "W42": [42] * 128, "P255": [42] * 128 + [43] * 127, "P511": [42] * 128 + [43] * 383,
                   "P252": [42] * 128 + [43] * 124}
            G_PHRASE = enc("The answer is yes. ")
            G_REGEX = "The answer is yes\\. "
            G_FILLER = enc(("The history of the printing press begins in the fifteenth century, when movable type transformed "
                            "how books were made and shared across Europe. ") * 12)
            G_FOLLOW = enc(" In summary, the most important consequence of this invention was")
            G_TRACK = 256
            TOK_INFO.update(pieces=len(_PIECES), pieces_equal=sum(_PIECES.values()), fp8_int4_ids_identical=all(_PIECES.values()))
            if not TOK_INFO["fp8_int4_ids_identical"]:
                ABORT["why"] = "NO_GO_TOKENIZER_MISMATCH: the INT4 tokenizer gives different ids than FP8 on " \
                               f"{len(_PIECES) - TOK_INFO['pieces_equal']} prompt pieces"
            R["gates"]["tokenizer"] = "GO" if TOK_INFO["fp8_int4_ids_identical"] else "NO_GO_TOKENIZER_MISMATCH"
            _sha = lambda p: hashlib.sha256(json.dumps(p).encode()).hexdigest()[:12]
            R["prompts"] = {"tokenizer": TOK_INFO,
                            "bands": {b: {"n": len(p), "len": len(p[0]), "sha": _sha(p[:4]),
                                          "distinct_first_task": len({tuple(x[len(_HEAD):len(_HEAD) + 50]) for x in p})}
                                      for b, p in PROMPTS.items()},
                            "oracle": [[k, len(p)] for k, p in ORACLE], "c16_sha": _sha([p for _, p in C16]),
                            "traces": {k: len(p) for k, p in TRACE_P.items()}, "trace_new": TRACE_NEW, "m4_new": M4_NEW,
                            "ems": {k: {"n": len(v), "p50": sorted(len(x) for x in v)[len(v) // 2]} for k, v in EMS.items()},
                            "shared4": {"n": len(SHARED4), "lens": [len(x) for x in SHARED4][:16]},
                            "s6": {k: len(v) for k, v in S6P.items()}, "grammar": {"phrase": len(G_PHRASE), "filler": len(G_FILLER),
                                                                                  "follow": len(G_FOLLOW)}}
        except Exception as e:
            ABORT["why"] = ABORT["why"] or f"ERROR_PROMPTS: {type(e).__name__}: {e}"[:300]
            err("prompts", e)
    log("prompts", json.dumps(R.get("prompts", {}).get("bands", {}))[:600], "| tokenizer", TOK_INFO)
    save()


In [ ]:
if not RERUN:
    WIN = max(2.0, (60.0 if not SM else 30.0) * TS)
    SEV = ("ERROR_", "NO_GO_", "INVALID_", "INCONCLUSIVE_")
    DEAD_GATES = {"NO_GO_LAUNCH", "NO_GO_SERVER_DIED", "NO_GO_LOGPROB_PREFLIGHT", "NO_GO_TOOL_PROBE"}
    GOOD = {"STEADY", "CAPACITY_LIMITED", "MEASURED", "ADMITTED_ALL", "GO", "EXACT", "EXACT_TO_N", "NEAR_TIE", "PASS", "ISOLATED",
            "OK", "PLATEAU", "AGGREGATE_STEADY"}
    PREFLIGHT_DONE = {"done": False, "ok": None, "arm": None}
    SHARED = {"first": True}
    ERR_SIGNS = ("Scheduler hit an exception", "Can not alloc mamba cache", "Mamba release hit lock_ref=0", "lock receipt anchored on node",
                 "alloc_req_slots runs out of memory", "illegal memory access", "CUDA error", "no kernel image", "invalid device function")
    def ts(sec, floor=1.0):
        return max(floor, float(sec) * TS)
    def call(fn, *a, **kw):
        """Call a lab / patcher function with only the keywords its signature accepts. A dropped keyword is recorded in
        R["api_drift"] (parallel work packages: drift must show in the results, never silently change a measurement)."""
        try:
            sig = inspect.signature(fn)
            if not any(p.kind == p.VAR_KEYWORD for p in sig.parameters.values()):
                bad = [k for k in kw if k not in sig.parameters]
                if bad:
                    d = R["api_drift"].setdefault(getattr(fn, "__qualname__", str(fn)), [])
                    d += [k for k in bad if k not in d]
                    kw = {k: v for k, v in kw.items() if k not in bad}
        except (TypeError, ValueError):
            pass
        return fn(*a, **kw)
    def small(obj, limit=20000):
        """obj itself when its JSON is at most `limit` bytes, else a marker (never a truncated JSON text)."""
        try:
            n = len(json.dumps(obj, default=str))
        except Exception as e:
            return {"unserializable": repr(e)[:200]}
        return obj if n <= limit else {"truncated": True, "bytes": n}
    def rawdir(arm):
        d = WORK / arm
        d.mkdir(parents=True, exist_ok=True)
        return d
    def jdump(path, obj):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + ".tmp")
        tmp.write_text(json.dumps(obj, default=str))
        tmp.replace(path)
    def http(port, path, body=None, timeout=60, method=None):
        """(status, text) of a local request; HTTP errors return their status, transport errors status None."""
        data = json.dumps(body).encode() if body is not None else None
        req = _U.Request(f"http://127.0.0.1:{port}{path}", data=data, method=method or ("POST" if data is not None else "GET"),
                         headers={"Content-Type": "application/json"})
        try:
            with _U.urlopen(req, timeout=timeout) as r:
                return r.status, r.read().decode("utf-8", "replace")
        except _U.HTTPError as e:
            try:
                return e.code, e.read().decode("utf-8", "replace")
            except Exception:
                return e.code, ""
        except Exception as e:
            return None, f"{type(e).__name__}: {e}"[:300]
    def flag_value(flags, name):
        return flags[flags.index(name) + 1] if name in flags and flags.index(name) + 1 < len(flags) else None
    def worst(statuses):
        """The most severe gate: ERROR_ > NO_GO_ > INVALID_ > INCONCLUSIVE_ > GO (SKIPPED_FAKE is neutral, FAKE only)."""
        best, rank = "GO", 0
        for s in statuses:
            if not s or s == "GO" or str(s).startswith("SKIPPED_FAKE"):
                continue
            r = next((len(SEV) - i for i, p in enumerate(SEV) if str(s).startswith(p)), 1)
            if r > rank:
                best, rank = str(s), r
        return best
    def mgate(st):
        """A measurement status as a gate: valid -> GO, FAIL_x -> NO_GO_x, anything else unknown -> INCONCLUSIVE_x."""
        if st is None:
            return "INCONCLUSIVE_MISSING"
        s = str(st)
        if s in GOOD or s.split("(")[0] in GOOD:
            return "GO"
        if s.startswith("FAIL_"):
            return "NO_GO_" + s[5:]
        if s.startswith(SEV):
            return s
        if s == "ERROR":
            return "ERROR_MEASUREMENT"
        return "INCONCLUSIVE_" + s
    def step(sec, arm, est_min, fn, dep=None):
        """One planned step: SKIPPED_ABORT / SKIPPED_DEPENDENCY / SKIPPED_BUDGET (unless est_min + 5 min remain, scaled),
        any exception -> ERROR_<Type> + R["errors"]; the status is always set explicitly (no DONE default)."""
        rec = R["sections"].setdefault(sec, {}).setdefault(arm, {"status": "NOT_RUN"})
        R["order"].append(f"{sec}/{arm}")
        rec["est_min"] = est_min
        if ABORT["why"]:
            rec.update(status="SKIPPED_ABORT", why=ABORT["why"])
            save()
            return rec
        why = dep() if callable(dep) else dep
        if why:
            rec.update(status="SKIPPED_DEPENDENCY", why=why)
            save()
            return rec
        if left() < (est_min * 60 + 300) * TS:
            rec.update(status="SKIPPED_BUDGET", left_min=round(left() / 60, 1))
            save()
            return rec
        t = time.time()
        log(">>>", sec, arm, f"(est {est_min} min, left {left() / 60:.0f} min)")
        try:
            out = fn() or {}
            rec.update(out)
            if rec.get("status") in (None, "NOT_RUN"):
                rec["status"] = "ERROR_NO_STATUS"
        except Exception as e:
            rec.update(status="ERROR_" + type(e).__name__, error=f"{type(e).__name__}: {e}"[:800], tb=traceback.format_exc()[-2500:])
            err(f"{sec}/{arm}", e)
        rec["minutes"] = round((time.time() - t) / 60, 2)
        if ABORT["why"] and "HARD_STOP" in ABORT["why"]:
            g = rec.get("gates") or {}
            new = worst([v for v in g.values() if not str(v).startswith("ERROR_") and v not in DEAD_GATES] + ["INCONCLUSIVE_HARD_STOP"])
            if (g or rec["status"] == "GO" or str(rec["status"]).startswith("ERROR_")) and new != rec["status"]:
                rec["status_before_hard_stop"], rec["status"] = rec["status"], new
        try:
            jdump(ARMS_OUT / f"{arm}.json", rec)
        except Exception as e:
            err(f"{sec}/{arm}/summary", e)
        try:
            raw_snapshot(sec, arm)
        except Exception as e:
            err(f"{sec}/{arm}/raw_snapshot", e)
        log("<<<", sec, arm, rec.get("status"), f"{rec['minutes']:.1f} min")
        save()
        return rec
    LAUNCH_FAILS = {"sglang": [], "vllm": []}
    LAUNCH_RISKY = re.compile(r"^(s6_g|s4_mem086|s4_lowcap|s4_m88|s4_m92|s3_(?!ctrl))")
    RAW_ARMS = OUT / "raw_arms"
    def launch_streak(eng, arm, gate, why=None, argv=None):
        """Quota guard (review: real platform #4): startup TIMEOUTS (900 s each) of three DISTINCT argvs in a row, on arms that
        vary no risky launch knob, mean the engine itself does not start; later arms of that engine become SKIPPED_DEPENDENCY.
        A fast exit costs seconds and is the arm's own config, and one config failing three times (s6_ffu / s6_ffp / s6_np
        share an argv) is one config (review 2 #5). Any launch that comes up resets the count."""
        if ABORT["why"] or gate != "NO_GO_LAUNCH" or LAUNCH_RISKY.match(arm) or "timeout" not in str(why or ""):
            return
        LAUNCH_FAILS[eng].append((arm, json.dumps(argv, default=str)))
        if len({k for _, k in LAUNCH_FAILS[eng]}) >= 3:
            ENGINES[eng]["stopped"] = "startup timeouts of three distinct argvs: " + ", ".join(a for a, _ in LAUNCH_FAILS[eng])
            err("launch_streak/" + eng, None, note=ENGINES[eng]["stopped"])
    def raw_snapshot(sec, arm):
        """Result-loss guard (review: real platform #6): a section-1 / section-6 arm's JSON raw files (rows, teacher forcing,
        M4, hook records; never profiles) go to /kaggle/working right after the arm, so a kernel death keeps them; CELL6
        folds them into e24_raw.tar.gz and removes these."""
        d = WORK / arm
        files = [p for p in d.rglob("*") if p.is_file() and p.suffix in (".json", ".jsonl") and "profile" not in p.parts] \
            if sec in ("1", "6") and d.is_dir() else []
        if files:
            RAW_ARMS.mkdir(parents=True, exist_ok=True)
            with tarfile.open(RAW_ARMS / f"{arm}.tar.gz.tmp", "w:gz") as tf:
                for p in files:
                    tf.add(p, arcname=str(p.relative_to(WORK)))
            (RAW_ARMS / f"{arm}.tar.gz.tmp").replace(RAW_ARMS / f"{arm}.tar.gz")
    def dep_engine(eng):
        def f():
            if ENGINES[eng].get("failed"):
                return f"{eng} engine unavailable: {ENGINES[eng]['failed']}"[:300]
            if ENGINES[eng].get("stopped"):
                return f"{eng} launches stopped: {ENGINES[eng]['stopped']}"[:300]
            if eng == "sglang" and PATCH_BROKEN:
                return "PATCH_BROKEN: " + PATCH_BROKEN
            return None
        return f
    def cache_root(arm, spec):
        """DESIGN §3.3: section 1 fresh roots (c_warm / c_nocache reuse s1_c's root); section 6 + SMOKE speed arms run on a
        COPY of the matching section-1 root; FULL_B builds one shared cache with its first launch."""
        pol = spec.get("cache") or "fresh"
        base = WORK / "cache"
        if pol == "fresh":
            root = base / arm
            shutil.rmtree(root, ignore_errors=True)
            return root, {"policy": "fresh", "root": str(root)}
        if pol == "shared":
            first = SHARED["first"]
            SHARED["first"] = False
            return base / "shared", {"policy": "shared", "root": str(base / "shared"), "first_launch": first}
        kind, src = pol
        if kind == "reuse":
            return base / src, {"policy": "reuse", "source": src, "root": str(base / src), "source_exists": (base / src).is_dir()}
        root, s = base / arm, base / src
        shutil.rmtree(root, ignore_errors=True)
        info = {"policy": "copy", "source": src, "root": str(root), "source_exists": s.is_dir()}
        if s.is_dir():
            try:
                shutil.copytree(s, root, symlinks=True, ignore_dangling_symlinks=True)
                info["bytes"] = _dir_size(root).get("bytes")
            except Exception as e:
                info["copy_error"] = f"{type(e).__name__}: {e}"[:300]
                err(f"cache_copy/{arm}", e)
        return root, info
    def targets(spec):
        """The arm's expected hook targets (E24_HOOK_TARGETS, e24_hook TARGETS names): a target missing on any TP rank ->
        HOOK_MISSING. vLLM arms: none required (the GPUModelRunner probe is informational)."""
        if spec["engine"] != "sglang" or spec.get("kind") == "unit":
            return []
        f, h, env = spec["flags"], spec.get("hook") or {}, spec.get("env") or {}
        dflash = flag_value(f, "--speculative-algorithm") == "DFLASH"
        eager = bool(h.get("eager")) and "--disable-decode-cuda-graph" in f
        t = ["Scheduler"]
        if h.get("counters"):
            t += ["ScheduleBatch", "SchedulerBatchResultProcessor", "MambaComponent"] + (["DFlashWorkerV2"] if dflash else [])
        if eager and env.get("E24_INJECT_DRAFT_NAN") == "1":
            t += ["DFlashWorkerV2", "_grouped_conv"]
        if eager and env.get("E24_INJECT_SELECTOR_NAN") == "1":
            t += ["DFlashWorkerV2", "CandidateSelector"]
        return list(dict.fromkeys(t))
    def arm_env(spec, cenv, hookdir):
        E = ENGINES[spec["engine"]]
        env = dict(E["env"])
        env.update(cenv)
        if spec["engine"] == "sglang":
            env["SGLANG_TORCH_PROFILER_DIR"] = str(rawdir(spec["arm"]) / "profile")
        env.update(spec.get("env") or {})
        hookdir.mkdir(parents=True, exist_ok=True)
        env.update(PYTHONPATH=str(HOOK_SITE), E24_HOOK_DIR=str(hookdir), E24_HOOK_TARGETS=",".join(targets(spec)),
                   E24_ARM=spec["arm"], E24_RAW_DIR=str(rawdir(spec["arm"])))
        h = spec.get("hook") or {}
        if h.get("counters"):
            env["E24_HOOK_COUNTERS"] = "1"
        if h.get("probe"):
            env["E24_HOOK_PROBE"] = "1"
        if h.get("eager") and "--disable-decode-cuda-graph" in spec["flags"]:
            env["E24_HOOK_EAGER"] = "1"
        return env
    def env_static(env, spec):
        """DESIGN §2: SGLANG_TEST_* / SGLANG_SIMULATE_* / E24_INJECT_* exist only in the arms that set them."""
        declared = set(spec.get("env") or {})
        bad = sorted(k for k in env if k.startswith(("SGLANG_TEST_", "SGLANG_SIMULATE_", "E24_INJECT_")) and k not in declared)
        fixed = {k: env.get(k) for k in ("SGLANG_LOGPROB_CHUNK_SIZE", "SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK", "HF_HUB_OFFLINE",
                                         "TRANSFORMERS_OFFLINE")}
        ok = not bad and (spec["engine"] != "sglang" or (fixed["SGLANG_LOGPROB_CHUNK_SIZE"] == "128"
                                                         and fixed["SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK"] == "0"))
        return {"unexpected": bad, "fixed": fixed}, ("GO" if ok else "INVALID_ENV_UNEXPECTED")
    def flag_check(spec, env):
        """ENFORCED (DESIGN §3.1): every --flag an exact 0.5.21 option string + prepare_server_args parse; FAKE: the fake's
        --e24-flag-check (same option list). resolve_once() standalone runs in SMOKE only and is ADVISORY: launch_server.py:29
        resolves the same args before anything loads, so a real conflict fails the launch within seconds anyway, while a
        standalone failure from the environment must not block an arm (FULL also saves ~15 s per distinct argv)."""
        eng = spec["engine"]
        E = ENGINES[eng]
        kw = {"env": env, "timeout": int(ts(300, 30))}
        if FAKE:
            kw["fake_server"] = E["fake_server"]
        elif eng == "sglang":
            kw.update(resolve=MODE == "SMOKE", model_path=flag_value(spec["flags"], "--model-path"))
        fc = call(L.flag_check, eng, E["vpy"], list(spec["flags"]), **kw) or {}
        if not fc.get("ran"):
            return fc, "ERROR_FLAG_CHECK_NOT_RUN"
        if fc.get("status") == "NO_GO_FLAG_RESOLUTION":
            fc["resolve_advisory"] = "resolve_once failed standalone (recorded, not blocking): the launch re-resolves"
            return fc, "GO"
        return fc, ("GO" if fc.get("ok") else (fc.get("status") if str(fc.get("status")).startswith("NO_GO") else "NO_GO_FLAG_UNSUPPORTED"))
    def ensure_patches(expect):
        """Bring the installed sglang to exactly `expect` (revert what is extra, apply the expected set; apply() refuses
        43559 without 43567), then check all 7 known names against the expectation (DESIGN §3.4)."""
        global PATCH_BROKEN
        if PSITE is None:
            return {"state": "SKIPPED_FAKE" if FAKE else "NO_SITE"}, ("SKIPPED_FAKE" if FAKE else "NO_GO_PATCH_SITE")
        rec = {}
        chk = P.check(PSITE, names=list(PATCH_ALL7))
        cur = sorted(n for n, s in (chk.get("names") or {}).items() if s != "not_applied")
        rec["before"] = chk.get("names")
        if set(cur) != set(expect):
            if cur:
                rec["revert"] = P.revert(PSITE, names=cur)
            if expect:
                rec["apply"] = P.apply(PSITE, names=list(expect))
        chk2 = P.check(PSITE, names=list(PATCH_ALL7))
        rec.update(after=chk2.get("names"), files=chk2.get("files"))
        ok = bool(P.state_ok(chk2, expect_applied=tuple(expect)))
        if not ok and not expect:
            PATCH_BROKEN = "revert to the pristine 0.5.21 failed: no unpatched arm may run on a patched build"
        return rec, ("GO" if ok else "NO_GO_PATCH_STATE")
    def compact_facts(f):
        f = dict(f or {})
        for k in ("error_lines", "oom_lines", "fp8_scale_lines"):
            if isinstance(f.get(k), list):
                f[k] = f[k][-12:]
        sa = f.pop("server_args", None)
        if isinstance(sa, dict):
            f["server_args_keys"] = len(sa)
            f["server_args_sel"] = {k: sa.get(k) for k in ("max_running_requests", "max_mamba_cache_size", "mamba_radix_cache_strategy",
                                                           "speculative_algorithm", "speculative_num_draft_tokens", "chunked_prefill_size",
                                                           "mem_fraction_static", "mamba_ssm_dtype", "prefill_decode_interval",
                                                           "enable_mixed_chunk", "disable_radix_cache", "attention_backend",
                                                           "linear_attn_backend", "context_length", "max_total_tokens") if k in sa}
        return f
    def compact_info(si):
        if not isinstance(si, dict):
            return si
        keep = {k: si.get(k) for k in ("version", "max_total_num_tokens", "max_req_input_len", "launch_command", "prefill_decode_interval",
                                       "enable_mixed_chunk", "chunked_prefill_size", "max_running_requests") if k in si}
        st = (si.get("internal_states") or [{}])[0] if isinstance(si.get("internal_states"), list) else {}
        keep["internal"] = {k: st.get(k) for k in ("effective_max_running_requests_per_dp", "memory_usage", "decode_cuda_graph_bs")
                            if isinstance(st, dict) and k in st}
        return keep
    def hook_files(hdir):
        """The e24_hook outputs of one launch: counters_<pid>.json, trace_<pid>.jsonl events, probe_<pid>.json."""
        hdir = Path(hdir)
        out = {"counters": [], "trace": [], "probe": []}
        for p in sorted(hdir.glob("*")) if hdir.is_dir() else []:
            try:
                if p.name.startswith("counters_") and p.suffix == ".json":
                    out["counters"].append(json.loads(p.read_text()))
                elif p.name.startswith("probe_") and p.suffix == ".json":
                    out["probe"].append(json.loads(p.read_text()))
                elif p.suffix == ".jsonl":
                    out["trace"] += [json.loads(l) for l in p.read_text(encoding="utf-8", errors="replace").splitlines() if l.strip()]
            except Exception:
                continue
        return out
    TP = 4
    def rank_files(hdir, tp=TP):
        """The TP scheduler ranks' counters files ONLY (hook rev 2b: top-level tp_rank + role_tag). The launcher, the
        detokenizer, API servers and trackers import target modules but run no rank: never counted as a rank."""
        by, ignored, dup = {}, 0, 0
        for c in hook_files(hdir)["counters"]:
            r = c.get("tp_rank")
            if c.get("role_tag") != "scheduler" or not isinstance(r, int) or isinstance(r, bool) or not 0 <= r < tp:
                ignored += 1
                continue
            if r in by:
                dup += 1
                if (c.get("t") or 0) < (by[r].get("t") or 0):
                    continue
            by[r] = c
        return by, {"ranks": sorted(by), "ignored_files": ignored, "duplicate_ranks": dup}
    def hook_selftest(hdir, spec, wait=60.0):
        """DESIGN §4.6: the arm's targets live on EXACTLY the TP ranks (one scheduler counters file per rank 0..tp-1 whose
        self_test is ok and lists every expected target); fewer ranks is fail-closed. FAKE servers have no sglang classes:
        a FAKE miss is INCONCLUSIVE_HOOK_FAKE, never GO."""
        need = targets(spec)
        if not need:
            return {"required": []}, "GO"
        end = time.time() + ts(wait, 5)
        while True:
            by, info = rank_files(hdir)
            okr = sorted(r for r, c in by.items() if (c.get("self_test") or {}).get("ok") and set(need) <= set(c.get("patched") or []))
            if len(okr) == TP or time.time() > end:
                break
            time.sleep(1.0)
        rep = dict(info, required=need, ranks_ok=okr, ranks_wanted=TP,
                   missing=sorted({m for c in by.values() for m in ((c.get("self_test") or {}).get("missing") or [])}),
                   failed=[(c.get("self_test") or {}).get("failed") for c in by.values() if (c.get("self_test") or {}).get("failed")][:4])
        if len(okr) == TP and len(by) == TP:
            return rep, "GO"
        return rep, ("INCONCLUSIVE_HOOK_FAKE" if FAKE else "INVALID_HOOK_MISSING")
    def hook_summary(hdir, spec):
        """Env records (roles, guarded knobs vs the arm's declaration), per-rank counters (min / max: identical on the 4
        ranks?), M5 probe records."""
        try:
            recs = call(L.hook_records, str(hdir)) or []
            cnt = call(L.hook_counters, str(hdir)) or {}
        except Exception as e:
            err("hook_summary/" + spec["arm"], e)
            recs, cnt = [], {}
        knobs = sorted({k for r in recs for k in list(r.get("env") or {}) + list(r) if isinstance(k, str)
                        and k.startswith(("SGLANG_TEST_", "SGLANG_SIMULATE_", "E24_INJECT_"))})
        roles = {}
        for r in recs:
            k = str((r.get("role") or {}).get("kind") if isinstance(r.get("role"), dict) else r.get("role"))
            roles[k] = roles.get(k, 0) + 1
        by, rinfo = rank_files(hdir)
        rc = [c.get("counters") or {} for c in by.values()]
        keys = sorted({k for c in rc for k, v in c.items() if isinstance(v, (int, float))})
        mn = {k: min(c.get(k, 0) for c in rc) for k in keys}
        mx = {k: max(c.get(k, 0) for c in rc) for k in keys}
        probes = []
        for pr in hook_files(hdir)["probe"]:
            probes.append({k: pr.get(k) for k in ("tp_rank", "where", "target", "draft", "mamba", "server_args", "modules")})
        return {"env_records": len(recs), "roles": roles, "knobs_seen": knobs,
                "knobs_undeclared": [k for k in knobs if k not in (spec.get("env") or {})], "counter_files": cnt.get("files"),
                "rank_files": rinfo, "counters_max": mx, "counters_identical": (mn == mx) if mx else None,
                "self_test_ok": {str(r): (c.get("self_test") or {}).get("ok") for r, c in by.items()},
                "probe": small(probes, 30000) if probes else []}
    def gpu_gate(arm, rec):
        rel = L.wait_gpu_release(int(ts(180, 5)), fake=FAKE, gpus="0,1,2,3")
        if rel is False:
            try:
                rec["orphans_killed"] = call(L.kill_orphans, fake=FAKE)
            except Exception as e:
                err(arm + "/kill_orphans", e)
            rel = L.wait_gpu_release(int(ts(60, 5)), fake=FAKE, gpus="0,1,2,3")
        rec["gpu_released"] = rel
        if rel is False:
            rec["gates"]["gpu_release"] = "ERROR_GPU_RELEASE"
            ABORT["why"] = f"GPUs still held after {arm} (release gate)"
    def with_server(arm, body, spec=None, wait=900.0, expect_errors=False):
        """One fresh server for one arm (DESIGN §3): env assert -> flag check (enforced) -> patch state (enforced) -> cache
        env (+ dir stats before) -> hook dir -> start -> wait (900 s) with the 600 s startup gate on EVERY launch -> facts /
        server_info / hook self-test -> body -> kill -> GPU release gate -> hook outputs -> cache verdict (section 1)."""
        spec = dict(spec or CAT[arm])
        spec["arm"] = arm
        eng = spec["engine"]
        rec = {"engine": eng, "gates": {}, "measured": {}}
        raw = rawdir(arm)
        hookdir = raw / "hook"
        shutil.rmtree(hookdir, ignore_errors=True)
        root, rec["cache"] = cache_root(arm, spec)
        cenv = call(L.cache_env, eng, str(root)) or {}
        env = arm_env(spec, cenv, hookdir)
        argv = list(ENGINES[eng]["cmd"]) + list(spec["flags"])
        R["argv"][arm] = argv
        rec.update(argv=argv, env_arm=spec.get("env"), patches_expected=list(spec.get("patches") or ()), hook_targets=targets(spec))
        rec["env_static"], rec["gates"]["env_static"] = env_static(env, spec)
        iso = spec["section"] == "1"
        paths = list(L.default_cache_dirs()) if iso else []
        before = call(L.dir_stats, paths) if iso else None
        rec["flag_check"], rec["gates"]["flags"] = flag_check(spec, env)
        if rec["gates"]["flags"] != "GO":
            rec["status"] = worst(rec["gates"].values())
            return rec
        if eng == "sglang":
            rec["patch_state"], rec["gates"]["patches"] = ensure_patches(spec.get("patches") or ())
            if rec["gates"]["patches"] not in ("GO", "SKIPPED_FAKE"):
                rec["status"] = worst(rec["gates"].values())
                return rec
        port = SGL_PORT if eng == "sglang" else VLLM_PORT
        srv = call(L.Server, arm, argv, env, port, str(LOGS), gpus="0,1,2,3", engine=eng)
        try:
            srv.start()
            ok, why = srv.wait_ready(min(ts(wait, 30), max(60.0, left() - 120)))
            rec.update(ready=ok, ready_why=why, ready_sec=srv.ready_sec)
            if not ok:
                rec["log_tail"] = srv.log()[-4000:]
                try:
                    fx = srv.facts() or {}
                except Exception:
                    fx = {}
                rec["facts"] = compact_facts(fx)
                rec["gates"]["launch"] = ("NO_GO_SM89_KERNEL" if fx.get("cuda_invalid_kernel") else
                                          "NO_GO_OOM" if fx.get("oom_lines") else "NO_GO_LAUNCH")
                launch_streak(eng, arm, rec["gates"]["launch"], why, [argv, sorted((spec.get("env") or {}).items())])
                return rec
            LAUNCH_FAILS[eng] = []
            rec["gates"]["launch"] = "GO" if (srv.ready_sec or 0) <= ts(600) else "NO_GO_STARTUP_600"
            if rec["gates"]["launch"] != "GO":
                rec["diagnostic"] = "startup gate failed: measurements kept (startup time does not change numerics), adoption blocked"
            if rec["cache"].get("policy") == "shared" and not rec["cache"].get("first_launch") and (srv.ready_sec or 0) > ts(300):
                rec["cache_miss"] = "CACHE_MISS"
            fx = srv.facts() or {}
            rec["facts"] = compact_facts(fx)
            if fx.get("cuda_invalid_kernel"):
                rec["gates"]["sm89"] = "NO_GO_SM89_KERNEL"
            if eng == "sglang":
                try:
                    rec["server_info"] = compact_info(srv.server_info())
                except Exception as e:
                    rec["server_info"] = {"error": repr(e)[:200]}
                    err(arm + "/server_info", e)
            rec["hook_selftest"], rec["gates"]["hook"] = hook_selftest(hookdir, spec)
            rec["gpu_mem_after_ready"] = L.gpu_mem()
            rec["model"] = getattr(srv, "model_name", None)
            try:
                out = body(srv) or {}
                for k, v in (out.pop("gates", None) or {}).items():
                    rec["gates"][k] = v
                rec.update(out)
            except Exception as e:
                rec["gates"]["body"] = "ERROR_BODY_" + type(e).__name__
                rec["body_error"] = f"{type(e).__name__}: {e}"[:800]
                err(arm + "/body", e)
            try:
                fx2 = srv.facts() or {}
                rec["facts_end"] = {k: fx2.get(k) for k in ("error_lines", "oom_lines", "allocator_oom_warnings") if fx2.get(k)}
            except Exception as e:
                err(arm + "/facts_end", e)
            text = srv.log()
            pre_kill = text.partition("SIGTERM received")[0]
            rec["error_sign_hits"] = {s: pre_kill.count(s) for s in ERR_SIGNS if s in pre_kill}
            if rec["error_sign_hits"] and not expect_errors:
                rec["gates"]["engine_errors"] = "NO_GO_ENGINE_ERROR"
            if rec.get("sampled") and "falling back to greedy argmax verification" in text:
                rec["gates"]["greedy_fallback"] = "NO_GO_GREEDY_FALLBACK"
            if not srv.alive():
                rec["gates"].setdefault("alive", "ERROR_SERVER_DIED" if not expect_errors else "INCONCLUSIVE_SERVER_DIED")
                rec["log_tail"] = text[-4000:]
                if rec["gates"].get("engine_errors") == "NO_GO_ENGINE_ERROR":
                    rec["engine_consequence_gates"] = {k: v for k, v in rec["gates"].items() if str(v).startswith("ERROR_")}
                    for k in rec["engine_consequence_gates"]:
                        rec["gates"][k] = "NO_GO_ENGINE_ERROR"
            rec["gpu_mem_end"], rec["host_rss_mb"] = L.gpu_mem(), srv.rss_mb()
            return rec
        finally:
            try:
                srv.kill()
            except Exception as e:
                err(arm + "/kill", e)
            gpu_gate(arm, rec)
            try:
                rec["hook"] = hook_summary(hookdir, spec)
                if rec["hook"].get("knobs_undeclared"):
                    rec["gates"]["env_records"] = "INVALID_ENV_UNEXPECTED"
            except Exception as e:
                err(arm + "/hook_summary", e)
            if iso:
                try:
                    after = call(L.dir_stats, sorted(set(paths) | set(L.default_cache_dirs())))
                    cv = call(L.cache_verdict, eng, str(root), str(hookdir), before, after,
                              dflash=flag_value(spec["flags"], "--speculative-algorithm") == "DFLASH") or {}
                    rec["cache_verdict"] = cv
                    rec["gates"]["cache_isolation"] = "GO" if cv.get("status") == "ISOLATED" else "INVALID_CACHE_ISOLATION"
                except Exception as e:
                    rec["gates"]["cache_isolation"] = "INVALID_CACHE_ISOLATION"
                    err(arm + "/cache_verdict", e)
            rec["status"] = worst(rec["gates"].values())
    def flush(eng, port):
        """/flush_cache on an idle server; valid = flushed AND idle afterwards (E17 rule)."""
        if eng != "sglang":
            return {"valid": None, "why": "vLLM oracle arms run with prefix caching off"}
        f = call(L.flush_cache, eng, port, wait=ts(60, 3)) or {}
        if not f.get("ok"):
            time.sleep(ts(2.0, 0.2))
            f = call(L.flush_cache, eng, port, wait=ts(30, 3)) or {}
        f["idle_after"] = call(L.wait_idle, eng, port, wait=ts(30, 3))
        f["valid"] = bool(f.get("ok") and f["idle_after"])
        return f
    def between_loads(eng, port):
        """DESIGN §4.4 / §4.5 (review #51 / #60): wait_idle + /flush_cache (valid only when idle) + a baseline snapshot."""
        f = flush(eng, port)
        return {"flush": f, "snapshot": L.snapshot(eng, port) if eng == "sglang" else None}
    def tmo():
        return int(min(3600, max(120, left() / 4)))
    def row_summary(rows):
        rows = rows or []
        return {"n": len(rows), "errors": [r.get("error") for r in rows if r.get("error")][:4],
                "garbage": [None if r.get("error") else L.degenerate(r.get("text")) for r in rows], "finish": [r.get("finish_reason") for r in rows],
                "cached_tokens": [r.get("cached_tokens") for r in rows], "lens": [len(r.get("tokens") or []) for r in rows]}
    def keyed(rows, keys, ns):
        for k, r in zip(keys, rows or []):
            if isinstance(r, dict):
                r["key"] = ns + k
        return rows or []
    def meas(o, name, fn):
        """One measurement inside an arm: an exception is recorded (gate ERROR_<name>) and never loses the others."""
        t = time.time()
        try:
            o["measured"][name] = fn()
        except Exception as e:
            o["measured"][name] = {"error": f"{type(e).__name__}: {e}"[:500]}
            o["gates"][name] = "ERROR_" + name.upper()
            err(f"measure/{name}", e)
        if isinstance(o["measured"][name], dict):
            o["measured"][name]["sec"] = round(time.time() - t, 1)
        save()
        return o["measured"][name]


In [ ]:
if not RERUN:
    ROWS, TFD, SELF, M4D, TRACES = {}, {}, {}, {}, {}
    C16F = {}
    S2 = {"pool": None, "ctrl_rows": None, "aa_rows": None}
    S5 = {"control": {}}
    SENT = os.environ.get("E24_SENTINEL_PREFIX", "e24-sentinel-")
    TOL, AA_MAX = 5e-3, 1e-3
    BINS8 = ((0, 2000), (2000, 8000), (8000, 16000), (16000, 30000))
    BINS90 = ((90000, 100000), (100000, 110000), (110000, 120000))
    def skip_cond(sec, arm, why):
        rec = R["sections"].setdefault(sec, {}).setdefault(arm, {})
        rec.update(status="SKIPPED_CONDITION", why=why)
        R["order"].append(f"{sec}/{arm}")
        save()
        return rec
    S1_MEAS = {
        "s1_a": ("traces m1 m2 m3", "traces m1 self m3"), "s1_b": ("m1 m2 m3", "m1 self m3"),
        "s1_d6": ("pre m1 self p2 m2 m3 m4", "pre m1 self p2 m2 m3 m4"), "s1_d6fr": ("m1 self m3", "m1 self"),
        "s1_d4": ("m1 self p2 m2 m3 m4", "m1 self p2"), "s1_c": ("m1 self p2 m2 m3 m4", "m1 self p2 m2 m3 m4"),
        "s1_c_warm": ("m1 self m3", "m1"), "s1_c_nocache": ("m1 self p2 m3", "m1"), "s1_c32": ("m1 self p2 m2 m3 m4", "m1"),
        "s1_d6_r2": ("m1 self p2 m2 m3", "m1"), "s1_c_r2": ("m1 self p2 m3", "m1")}
    def m_rows(eng, srv, arm, tag, items, n_tok, ns, concurrent=False):
        keys = [k for k, _ in items]
        rows = call(L.greedy, eng, srv.port, [p for _, p in items], n_tok=n_tok, top=2, timeout=tmo(), model=srv.model_name,
                    concurrent=concurrent, keys=keys)
        rows = keyed(rows, keys, ns)
        ROWS.setdefault(arm, {})[tag] = rows
        jdump(rawdir(arm) / f"rows_{tag}.json", rows)
        return row_summary(rows)
    def m_traces(eng, srv, arm):
        """M3 references (s1_a, vLLM FP8): production sampling (T 1, top_p 0.95, top_k 20), fixed seeds, ignore_eos, the
        natural EOS position recorded; generated concurrently (one batch of 2-3)."""
        keys = list(TRACE_P)
        gens = call(L.generate_logged_many, eng, srv.port, [TRACE_P[k] for k in keys], max_new=TRACE_NEW, temperature=1.0,
                    top_p=0.95, top_k=20, seeds=[20261005 + i for i in range(len(keys))], ignore_eos=True, eos_ids=EOS_IDS, top=1,
                    timeout=int(min(7200, max(300, left() - 600))), model=srv.model_name) or []
        for k, g in zip(keys, gens):
            g = g or {}
            TRACES[k] = {"prompt": TRACE_P[k], "cont": list(g.get("tokens") or []), "natural_eos_pos": g.get("natural_eos_pos"),
                         "error": g.get("error"), "seed": 20261005 + keys.index(k)}
        jdump(rawdir(arm) / "traces.json", TRACES)
        return {k: {"n": len(v["cont"]), "want": TRACE_NEW, "natural_eos_pos": v["natural_eos_pos"], "error": v["error"]}
                for k, v in TRACES.items()}
    def m_score(eng, srv, arm):
        """M3: every arm teacher-forces the FP8 traces, one request at a time, top-2 (SGLang: logprob chunk 128)."""
        if not TRACES:
            return {"status": "INCONCLUSIVE_NO_TRACES"}
        out = {}
        for k, tr in TRACES.items():
            if not tr["cont"]:
                out[k] = {"status": "INCONCLUSIVE_NO_TRACE"}
                continue
            tf = call(L.teacher_forced, eng, srv.port, tr["prompt"], tr["cont"], top=2, timeout=tmo(), model=srv.model_name) or {}
            TFD.setdefault(arm, {})[k] = tf
            jdump(rawdir(arm) / f"tf_{k}.json", tf)
            out[k] = {"n": tf.get("n"), "status": tf.get("status"), "error": tf.get("error")}
            if not srv.alive():
                out[k]["server_died"] = True
                break
        return out
    def m_pre(eng, srv, arm):
        """The first SGLang arm scores one 4,096-row extend before M3 (rev 2 blocker #1: logprob OOM)."""
        tf = call(L.teacher_forced, eng, srv.port, PREFLIGHT[0], PREFLIGHT[1], top=2, timeout=tmo(), model=srv.model_name) or {}
        ok = srv.alive() and not tf.get("error") and tf.get("status") != "INCONCLUSIVE_TF_ALIGN"
        PREFLIGHT_DONE.update(done=True, ok=ok, arm=arm)
        fl = flush(eng, srv.port) if ok else None
        return {"rows": len(PREFLIGHT[1]), "n": tf.get("n"), "status": tf.get("status"), "error": tf.get("error"), "alive_after": srv.alive(),
                "flush_after": fl, "gate": "GO" if ok else "NO_GO_LOGPROB_PREFLIGHT"}
    def m_self(eng, srv, arm):
        """Prefill re-score of each serial M1 row along its own greedy path (top-2 at every position): with both arms'
        self scores, a FLIP at k is attributed decode/verify-path specific (prefills agree) or engine-level."""
        out = 0
        for (key, p), r in zip(ORACLE, (ROWS.get(arm) or {}).get("m1") or []):
            if r.get("tokens"):
                tf = call(L.teacher_forced, eng, srv.port, p, r["tokens"], top=2, timeout=tmo(), model=srv.model_name) or {}
                SELF.setdefault(arm, {})["s:" + key] = tf
                out += 1
        jdump(rawdir(arm) / "self_tf.json", SELF.get(arm) or {})
        return {"rows": out}
    def m_m2(eng, srv, arm, sampled):
        """M2: the c16 set twice, each batch after a flush, so c16b is a pure repeat of c16a (the A/A) and never a radix re-run
        of it; a restore inside the A/A would turn restore errors into A/A allowance (review: decision logic #4)."""
        fa = flush(eng, srv.port)
        a = m_rows(eng, srv, arm, "c16a", C16, 256, "c:", concurrent=True)
        fb = flush(eng, srv.port)
        out = {"c16a": a, "c16b": m_rows(eng, srv, arm, "c16b", C16, 256, "c:", concurrent=True),
               "flush_valid": [fa.get("valid"), fb.get("valid")]}
        C16F[arm] = out["flush_valid"]
        if sampled:
            gens = call(L.generate_logged_many, eng, srv.port, [p for _, p in C16], max_new=2048, temperature=1.0, top_p=0.95,
                        top_k=20, seeds=[7000 + i for i in range(16)], ignore_eos=False, eos_ids=EOS_IDS, top=1, timeout=tmo(),
                        model=srv.model_name) or []
            jdump(rawdir(arm) / "t1_batch.json", gens)
            out["t1_smoke_garbage"] = {"label": "smoke garbage check (no pair verdict)", "n": len(gens),
                                       "garbage": [L.degenerate(g.get("text")) for g in gens if isinstance(g, dict)],
                                       "natural_eos_pos": [g.get("natural_eos_pos") for g in gens if isinstance(g, dict)],
                                       "lens": [len(g.get("tokens") or []) for g in gens if isinstance(g, dict)]}
        return out
    def m_m4(eng, srv, arm):
        """M4: production-sampled c16 (fixed seeds, ignore_eos, output logprobs), then the same server re-scores every
        trace by prefill (after a flush: prefill path only)."""
        gens = call(L.generate_logged_many, eng, srv.port, M4_P, max_new=M4_NEW, temperature=1.0, top_p=0.95, top_k=20,
                    seeds=[30000 + i for i in range(len(M4_P))], ignore_eos=True, eos_ids=EOS_IDS, top=1,
                    timeout=int(min(7200, max(300, left() - 600))), model=srv.model_name) or []
        fl = flush(eng, srv.port)
        tfs = []
        for p, g in zip(M4_P, gens):
            toks = list((g or {}).get("tokens") or [])
            tfs.append(call(L.teacher_forced, eng, srv.port, p, toks, top=2, timeout=tmo(), model=srv.model_name) if toks
                       else {"error": "no generated tokens"})
        M4D[arm] = {"gen": gens, "tf": tfs}
        jdump(rawdir(arm) / "m4.json", M4D[arm])
        own = call(L.decode_vs_prefill, gens, tfs)
        return {"flush": fl, "streams": len(gens), "tokens": [len((g or {}).get("tokens") or []) for g in gens], "own": own}
    def s1(arm):
        """A section-1 arm: fresh isolated cache, its measurement list, M5 rank probe (hook), M6 tool probe (SGLang)."""
        spec = CAT[arm]
        eng = spec["engine"]
        def body(srv):
            o = {"gates": {}, "measured": {}}
            plan = [m for m in (S1_MEAS[arm][1 if SM else 0] or "").split() if m != "pre"]
            if eng == "vllm" and not SM and "self" in plan:
                plan.remove("self")
            heavy = [m for m in plan if m in ("m3", "m4")]
            if eng == "sglang" and heavy and not PREFLIGHT_DONE["done"]:
                plan.insert(plan.index(heavy[0]), "pre")
            try:
                o["probes"] = call(L.chat_probes, eng, srv.port, model=srv.model_name)
            except Exception as e:
                o["probes"] = {"error": repr(e)[:200]}
                err(arm + "/chat_probes", e)
            for m in plan:
                if m in ("m3", "m4") and eng == "sglang" and PREFLIGHT_DONE["done"] and not PREFLIGHT_DONE["ok"]:
                    o["measured"][{"m3": "m3_score", "m4": "m4"}[m]] = {"status": "NOT_MEASURED", "why": "the logprob pre-flight "
                                                                        f"failed on {PREFLIGHT_DONE['arm']}: no >= 4,096-row extend"}
                    continue
                if m == "pre":
                    r = meas(o, "pre", lambda: m_pre(eng, srv, arm))
                    o["gates"]["preflight"] = (r or {}).get("gate", "ERROR_PREFLIGHT")
                elif m == "traces":
                    meas(o, "m3_traces", lambda: m_traces(eng, srv, arm))
                    o["sampled"] = True
                elif m == "m1":
                    meas(o, "m1", lambda: m_rows(eng, srv, arm, "m1", ORACLE, NTOK, "s:"))
                elif m == "self":
                    meas(o, "self_tf", lambda: m_self(eng, srv, arm))
                elif m == "p2":
                    meas(o, "m1_pass2", lambda: m_rows(eng, srv, arm, "p2", ORACLE, NTOK, "s:"))
                elif m == "m2":
                    meas(o, "m2", lambda: m_m2(eng, srv, arm, sampled=not spec.get("greedy_only")))
                    o["sampled"] = o.get("sampled") or not spec.get("greedy_only")
                elif m == "m3":
                    meas(o, "m3_score", lambda: m_score(eng, srv, arm))
                elif m == "m4":
                    meas(o, "m4", lambda: m_m4(eng, srv, arm))
                    o["sampled"] = True
                if not srv.alive():
                    o["gates"]["alive"] = "ERROR_SERVER_DIED"
                    break
            if eng == "sglang" and srv.alive():
                o["m6_tool"] = call(L.tool_probe, eng, srv.port, model=srv.model_name)
                o["gates"]["m6_tool"] = "GO" if o["m6_tool"].get("ok") else "NO_GO_TOOL_PROBE"
            else:
                o["m6_tool"] = {"status": "NOT_APPLICABLE", "why": "vLLM oracle argv has no tool-call parser"}
            for k in ("m1", "m1_pass2"):
                g = [x for x in ((o["measured"].get(k) or {}).get("garbage") or []) if x]
                if len(g) >= 2:
                    o["gates"]["garbage_" + k] = "NO_GO_REPEATED_GARBAGE"
            return o
        def f():
            rec = with_server(arm, body)
            rec["m5_probe"] = (rec.get("hook") or {}).get("probe") or "INCONCLUSIVE_NO_PROBE_RECORD"
            try:
                R["comparisons"]["1"] = s1_compare_all(final=False)
            except Exception as e:
                err("comparisons/1", e)
            return rec
        return f
    def _cls_counts(c):
        cnt = dict(c.get("counts") or c.get("class_counts") or {})
        if not cnt:
            for r in c.get("rows") or []:
                k = r.get("class") or r.get("status")
                cnt[k] = cnt.get(k, 0) + 1
        return cnt
    def hm(c, kind="same_weights"):
        """High-margin count of a compare() result: FLIP + STATE_FLIP (same weights) or STATE_FLIP (cross weights)."""
        c = c or {}
        if kind == "cross_weights":
            return int(c.get("state_flips") if c.get("state_flips") is not None else _cls_counts(c).get("STATE_FLIP", 0))
        if isinstance(c.get("high"), int):
            return c["high"]
        cnt = _cls_counts(c)
        return int(cnt.get("STATE_FLIP", 0)) + int(cnt.get("FLIP", 0))
    def _rows(arm, *tags):
        out = []
        for t in tags:
            out += list((ROWS.get(arm) or {}).get(t) or [])
        return out
    def _tp(a, b):
        """T_pair = max(0.5, q99.9 of the pair's |delta(top1 - top2)| on identical prefixes), from M3; per trace, the max."""
        vals = []
        for k in TRACES:
            x, y = (TFD.get(a) or {}).get(k), (TFD.get(b) or {}).get(k)
            if x and y and not x.get("error") and not y.get("error"):
                try:
                    vals.append(float(call(L.tpair, x, y)))
                except Exception as e:
                    err(f"tpair/{a}~{b}/{k}", e)
        return (max([0.5] + vals), "m3" if vals else "default_0.5")
    def _attr(ref, cand, c):
        """FLIP attribution by both arms' prefill re-scores at the first differing token."""
        out = []
        for r in c.get("rows") or []:
            if (r.get("class") or r.get("status")) not in ("FLIP", "STATE_FLIP"):
                continue
            key, k = r.get("key"), r.get("first_diff")
            pf = []
            for arm in (ref, cand):
                tf = (SELF.get(arm) or {}).get(key) or {}
                pos = tf.get("positions") or []
                pf.append(pos[k][1] if isinstance(k, int) and k < len(pos) and pos[k] else None)
            out.append({"key": key, "k": k, "class": r.get("class"), "prefill_top1": pf,
                        "attribution": ("partial" if None in pf else ("decode_verify_path" if pf[0] == pf[1] else "engine_level"))})
        return out
    S1_PAIRS = [("s1_a", "s1_b", "cross_weights", "weights + KV scale"), ("s1_b", "s1_c32", "same_weights", "engine (both fp32 SSM)"),
                ("s1_c32", "s1_c", "same_weights", "SSM dtype"), ("s1_c", "s1_d6", "same_weights", "speculation W6"),
                ("s1_c", "s1_d4", "same_weights", "speculation W4"), ("s1_c", "s1_d6fr", "same_weights", "target-verify path vs plain decode"),
                ("s1_c", "s1_c_warm", "same_weights", "cold vs warm cache"),
                ("s1_c", "s1_c_nocache", "same_weights", "radix / Mamba checkpoint reuse"),
                ("s1_b", "s1_c", "same_weights", "mixed engine + SSM (README literal b vs c, report only)")]
    S1_AA_BOOT = [("s1_d6", "s1_d6_r2"), ("s1_c", "s1_c_r2")]
    S1_CHAIN = [("s1_a", "s1_b"), ("s1_b", "s1_c32"), ("s1_c32", "s1_c"), ("s1_c", "s1_d6"), ("s1_c", "s1_d4")]
    SPEC_PAIRS = {("s1_c", "s1_d6"), ("s1_c", "s1_d4"), ("s1_c", "s1_d6fr")}
    S1_TARGET_ONLY = {"s1_c32", "s1_c", "s1_c_warm", "s1_c_nocache", "s1_c_r2"}
    S1_REPORT_ONLY = {("s1_b", "s1_c")}
    M4_BASE = {"s1_d6": "s1_c", "s1_d4": "s1_c", "s1_c": "s1_c32"}
    def _same_restore(a_rows, b_rows):
        """A/A rows only where both runs resumed from the same cached prefix (cached_tokens equal): the ARC prompts share a
        64-token header, so "cold" rows are partial restores whose hit pattern depends on admission order; a row that hit in
        one run and not in the other compares two different paths, not A/A noise (second validation round, FAKE scenario
        restore_corruption). Returns (a_rows, b_rows, n_excluded)."""
        cb = {r.get("key"): r.get("cached_tokens") or 0 for r in b_rows or []}
        keep = {r.get("key") for r in a_rows or [] if r.get("key") in cb and (r.get("cached_tokens") or 0) == cb[r.get("key")]}
        return ([r for r in a_rows or [] if r.get("key") in keep], [r for r in b_rows or [] if r.get("key") in keep],
                len({r.get("key") for r in a_rows or []} & set(cb)) - len(keep))
    def _n_eval(c):
        """Rows a compare() result judged (aligned keys minus INCONCLUSIVE rows): the denominator of an A/A rate."""
        c = c or {}
        return max(0, int(c.get("n_pairs") or 0) - int(_cls_counts(c).get("INCONCLUSIVE", 0)))
    def _rate(cmps, state=False):
        """Max per-row rate of FLIP + STATE_FLIP (state: STATE_FLIP only) over usable A/A comparisons; None without one."""
        v = [(hm(c, "cross_weights") if state else hm(c)) / _n_eval(c) for c in cmps or () if isinstance(c, dict) and _n_eval(c) > 0]
        return max(v) if v else None
    S1_ALPHA = 0.05
    def _fisher_greater(x, n1, y, n2):
        """One-sided Fisher exact test: P(X >= x) for X ~ Hypergeometric(n1 + n2 rows, x + y high-margin rows, n1 drawn),
        the chance that a pair with no real difference shows x of its n1 judged rows high-margin when y of n2 pure A/A rows
        did. None without rows on either side."""
        if not n1 or not n2 or x is None or y is None:
            return None
        k_all, m = x + y, n1 + n2
        return sum(math.comb(k_all, k) * math.comb(m - k_all, n1 - k) for k in range(x, min(k_all, n1) + 1)) / math.comb(m, n1)
    def _aa_pool(aa, *arms):
        """(FLIP + STATE_FLIP rows, judged rows) over the distinct pure A/A comparisons of the arms; (None, None) without one."""
        seen = {id(c): c for a in arms for c in (aa.get(a) or ()) if isinstance(c, dict) and _n_eval(c) > 0}
        return (sum(hm(c) for c in seen.values()), sum(_n_eval(c) for c in seen.values())) if seen else (None, None)
    def _allow(rates, n):
        """A/A allowance scaled to the rows a pair judged: ceil(rate x n) (review #12: a 16-row A/A against a 25-row pair is
        not the same count); None when neither arm has a usable A/A."""
        r = [x for x in rates if x is not None]
        return int(math.ceil(max(r) * n - 1e-9)) if r else None
    def _pair(ref_rows, cand_rows, kind, t_pair, rates, state_rates):
        """compare() with the scaled allowance: one pass for the judged rows, one with aa_high = the allowance (cross weights:
        the STATE_FLIP allowance). Returns (result, allowance FLIP + STATE_FLIP, allowance STATE_FLIP, judged rows)."""
        c0 = call(L.compare, ref_rows, cand_rows, kind=kind, t_pair=t_pair, aa_high=None) or {}
        n = _n_eval(c0)
        ah, ast = _allow(rates, n), _allow(state_rates, n)
        use = ast if kind == "cross_weights" else ah
        c = (call(L.compare, ref_rows, cand_rows, kind=kind, t_pair=t_pair, aa_high=use) or {}) if use is not None else c0
        return c, ah, ast, n
    def _usable(arm):
        st = str((R["sections"].get("1", {}).get(arm) or {}).get("status") or "")
        gates = (R["sections"].get("1", {}).get(arm) or {}).get("gates") or {}
        return bool(ROWS.get(arm)) and gates.get("cache_isolation") != "INVALID_CACHE_ISOLATION" and not st.startswith("SKIPPED")
    def s1_compare_all(final=False):
        """DESIGN §3.2 + §4.1: A/A per arm from pure repeats (c16 <-> c16 after flushes, r2 boots) as per-row rates; every pair
        aligned by key with its calibrated T_pair and the A/A allowance scaled to its judged rows; the within-arm test pairs
        (pass 1 <-> pass 2 restore, serial vs c16) and cold <-> warm with their own verdicts; FLIP attribution, M3 per trace,
        M4 with the length-matched baseline, the decisive-pair chain a -> b -> c32 -> c -> d6 / d4 (unevaluated links are
        never clean), and the arm verdicts written into the arm statuses. Recomputed in the report cell with every A/A."""
        out = {"final": final, "aa": {}, "within": {}, "pairs": {}, "m4": {}, "chain": [], "decisive": None}
        aa = {}
        for arm in ROWS:
            w = {}
            if _rows(arm, "c16a") and _rows(arm, "c16b"):
                xa, xb, w["c16_aa_rows_restore_mismatch"] = _same_restore(_rows(arm, "c16a"), _rows(arm, "c16b"))
                w["c16_aa"] = call(L.compare, xa, xb, kind="same_weights", t_pair=0.5, aa_high=None)
                fl = C16F.get(arm) or []
                w["c16_aa_pure"] = bool(fl) and all(x in (True, None) for x in fl)
            out["within"][arm] = w
            aa[arm] = [w["c16_aa"]] if w.get("c16_aa_pure") and isinstance(w.get("c16_aa"), dict) else []
        for a, b in S1_AA_BOOT:
            if _usable(a) and _usable(b):
                tags = ("m1", "c16a") if all((C16F.get(x) or [True])[0] in (True, None) for x in (a, b)) else ("m1",)
                xa, xb, nx = _same_restore(_rows(a, *tags), _rows(b, *tags))
                c = call(L.compare, xa, xb, kind="same_weights", t_pair=0.5, aa_high=None)
                if isinstance(c, dict):
                    c["rows_restore_mismatch"] = nx
                out["aa"][f"{a}~{b}"] = c
                aa.setdefault(a, []).append(c)
                aa.setdefault(b, []).append(c)
        out["aa_rate"] = {k: _rate(v) for k, v in aa.items()}
        out["aa_state_rate"] = {k: _rate(v, state=True) for k, v in aa.items()}
        out["aa_high"] = {k: L.aa_high_of(v) for k, v in aa.items()}
        out["aa_state"] = {k: (max(hm(c, "cross_weights") for c in v if not str(c.get("status", "")).startswith("INCONCLUSIVE"))
                               if any(not str(c.get("status", "")).startswith("INCONCLUSIVE") for c in v) else None) for k, v in aa.items()}
        for arm, w in out["within"].items():
            rr, rs = [out["aa_rate"].get(arm)], [out["aa_state_rate"].get(arm)]
            if _rows(arm, "m1") and _rows(arm, "p2"):
                c, ah, ast, n = _pair(_rows(arm, "m1"), _rows(arm, "p2"), "same_weights", 0.5, rr, rs)
                cached = [r.get("cached_tokens") for r in _rows(arm, "p2")]
                hy, ny = _aa_pool(aa, arm)
                w.update(pass1_pass2=c, pass1_pass2_allow={"high": ah, "state": ast, "rows": n}, pass2_cached_tokens=cached,
                         pass1_pass2_p=_fisher_greater(hm(c), n, hy, ny),
                         pass2_restore="NOT_EXERCISED" if not any(isinstance(x, int) and x > 0 for x in cached) else "EXERCISED")
            if _rows(arm, "m1") and _rows(arm, "c16a"):
                ser = [dict(r, key=str(r.get("key", "")).replace("s:", "")) for r in _rows(arm, "m1") if "8k#" in str(r.get("key"))]
                c16 = [dict(r, key=str(r.get("key", "")).replace("c:", "")) for r in _rows(arm, "c16a")]
                c16 = [r for r in c16 if r["key"] in {x["key"] for x in ser}]
                c, ah, ast, n = _pair(ser, c16, "same_weights", 0.5, rr, rs)
                w.update(serial_vs_c16=c, serial_vs_c16_allow={"high": ah, "state": ast, "rows": n})
        for ref, cand, kind, label in S1_PAIRS:
            key = f"{ref}~{cand}"
            if not (_usable(ref) and _usable(cand)):
                out["pairs"][key] = {"label": label, "kind": kind, "status": "INCONCLUSIVE_ARM_MISSING_OR_INVALID",
                                     "ref_status": (R["sections"]["1"].get(ref) or {}).get("status"),
                                     "cand_status": (R["sections"]["1"].get(cand) or {}).get("status")}
                continue
            tp, tsrc = _tp(ref, cand)
            rs = [out["aa_state_rate"].get(ref), out["aa_state_rate"].get(cand)]
            rr = rs if kind == "cross_weights" else [out["aa_rate"].get(ref), out["aa_rate"].get(cand)]
            c, ah, ast, n = _pair(_rows(ref, "m1", "c16a"), _rows(cand, "m1", "c16a"), kind, tp, rr, rs)
            aah = ast if kind == "cross_weights" else ah
            hy, ny = _aa_pool(aa, ref, cand)
            pv = _fisher_greater(hm(c, kind), n, hy, ny) if kind == "same_weights" else None
            p = {"label": label, "kind": kind, "t_pair": tp, "t_pair_source": tsrc, "aa_high": aah, "aa_state_allow": ast,
                 "rows_judged": n, "hm": hm(c, kind), "state_flips": c.get("state_flips"), "garbage": c.get("garbage"), "p_value": pv,
                 "inconclusive_rows": _cls_counts(c).get("INCONCLUSIVE", 0),
                 "status": c.get("status"), "counts": _cls_counts(c), "positions_examined": c.get("positions_examined"),
                 "attribution": _attr(ref, cand, c),
                 "diagnostic": [x for x in (ref, cand) if ((R["sections"]["1"].get(x) or {}).get("gates") or {}).get("launch") == "NO_GO_STARTUP_600"]}
            p["rows"] = [{k: r.get(k) for k in ("key", "first_diff", "class", "m_sum", "swap", "garbage", "status") if k in r}
                         for r in (c.get("rows") or [])][:40]
            m3 = {}
            for k in TRACES:
                x, y = (TFD.get(ref) or {}).get(k), (TFD.get(cand) or {}).get(k)
                if x and y:
                    m3[k] = call(L.compare_tf, x, y, bins=BINS90 if k == "t90k" else BINS8,
                                 offset=len(TRACES[k]["prompt"]) if k == "t90k" else 0)
            p["m3"] = m3 or {"status": "INCONCLUSIVE_NO_M3"}
            if (ref, cand) in SPEC_PAIRS:
                p["m3_label"] = "NOT_APPLICABLE_PREFILL_ONLY for DFlash state (server-equivalence sanity); M4 is the DFlash test"
            out["pairs"][key] = p
        m4own = {}
        for arm, d in M4D.items():
            try:
                m4own[arm] = call(L.decode_vs_prefill, d["gen"], d["tf"])
            except Exception as e:
                m4own[arm] = {"error": repr(e)[:200]}
                err(f"m4/{arm}", e)
        for arm, own in m4own.items():
            base = M4_BASE.get(arm)
            res = {"own": own, "base_arm": base}
            if base and base in m4own and not (m4own[base] or {}).get("error"):
                try:
                    res["vs_base"] = call(L.decode_vs_prefill, M4D[arm]["gen"], M4D[arm]["tf"], base=m4own[base])
                except Exception as e:
                    res["vs_base"] = {"error": repr(e)[:200]}
                    err(f"m4/{arm}/vs_base", e)
            else:
                res["vs_base"] = {"status": "INCONCLUSIVE_NO_BASELINE", "why": f"baseline {base} has no M4" if base else "no baseline (gross rule only)"}
            effect = arm == "s1_c"
            res["vs_base_role"] = "effect (reported, not a verdict)" if effect else "STATE_DRIFT baseline"
            res["state_drift"] = any(str((x or {}).get("status")) == "STATE_DRIFT" for x in ((own,) if effect else (own, res["vs_base"])))
            out["m4"][arm] = res
        for ref, cand in S1_CHAIN:
            p = out["pairs"].get(f"{ref}~{cand}") or {}
            norm, note = None, None
            g = p.get("garbage") or 0
            if "hm" not in p:
                note = "pair missing (an arm missing or invalid)"
            elif str(p.get("status") or "").startswith("INCONCLUSIVE") or not p.get("positions_examined"):
                note = "pair not evaluated (INCONCLUSIVE or no positions examined): never a clean link"
            elif p.get("aa_high") is None and not g:
                note = "no usable A/A: not normalisable"
            else:
                base = p["hm"] - (p["aa_high"] + (0 if p.get("kind") == "cross_weights" else 1)) if p.get("aa_high") is not None else None
                norm = max(x for x in (base, g or None) if x is not None)
                if norm <= 0 and p.get("inconclusive_rows"):
                    norm, note = None, f"{p['inconclusive_rows']} rows unjudged: a clean link is not proven"
            sig = norm is not None and norm > 0 and (p.get("kind") == "cross_weights" or g > 0 or
                                                      (p.get("p_value") is not None and p["p_value"] < S1_ALPHA))
            out["chain"].append({"pair": f"{ref}~{cand}", "kind": p.get("kind"), "hm": p.get("hm"), "aa_high": p.get("aa_high"),
                                 "normalised": norm, "p_value": p.get("p_value"), "significant": sig, "status": p.get("status"), "note": note})
        dec = next((x for x in out["chain"] if x["significant"]), None)
        sug = next((x for x in out["chain"] if x["normalised"] is not None and x["normalised"] > 0 and not x["significant"]), None)
        lead = dec or sug
        upstream_open = [x["pair"] for x in out["chain"][:out["chain"].index(lead)] if x["normalised"] is None] if lead else \
            [x["pair"] for x in out["chain"] if x["normalised"] is None]
        tail = "_UPSTREAM_UNCERTAIN" if upstream_open else ""
        out["decisive"] = {"pair": dec["pair"] if dec else None, "link": dec, "suggestive_pair": sug["pair"] if sug else None,
                           "suggestive_link": sug, "alpha": S1_ALPHA, "upstream_not_normalisable": upstream_open,
                           "verdict": ("DECISIVE_PAIR_FOUND" + tail if dec else "DECISIVE_PAIR_SUGGESTIVE" + tail if sug else
                                       "NO_HIGH_MARGIN_LINK" if not upstream_open else "INCONCLUSIVE_CHAIN_INCOMPLETE")}
        arm_v, notes = {}, {}
        for key, p in out["pairs"].items():
            ref, cand = key.split("~")
            if p.get("kind") != "same_weights" or (ref, cand) in S1_REPORT_ONLY or str(p.get("status") or "").startswith("INCONCLUSIVE"):
                continue
            if (p.get("state_flips") or 0) > (p.get("aa_state_allow") or 0):
                arm_v.setdefault(cand, []).append("NO_GO_STATE_FLIP vs " + ref)
            if cand in S1_TARGET_ONLY and p.get("status") == "FAIL_HIGH_MARGIN":
                if (p.get("garbage") or 0) > 0 or (p.get("p_value") is not None and p["p_value"] < S1_ALPHA):
                    arm_v.setdefault(cand, []).append("NO_GO_HIGH_MARGIN_DRIFT vs " + ref)
                else:
                    notes.setdefault(cand, []).append(f"SUGGESTIVE_HIGH_MARGIN_DRIFT vs {ref} (p={p.get('p_value')})")
        for arm, w in out["within"].items():
            for k in ("pass1_pass2", "serial_vs_c16"):
                c = w.get(k)
                if not isinstance(c, dict) or str(c.get("status") or "").startswith("INCONCLUSIVE") or \
                        (k == "pass1_pass2" and w.get("pass2_restore") != "EXERCISED"):
                    continue
                if (c.get("state_flips") or 0) > ((w.get(k + "_allow") or {}).get("state") or 0):
                    arm_v.setdefault(arm, []).append(f"NO_GO_STATE_FLIP ({k})")
                if k == "pass1_pass2" and c.get("status") == "FAIL_HIGH_MARGIN":
                    pv = w.get("pass1_pass2_p")
                    if (c.get("garbage") or 0) > 0 or (pv is not None and pv < S1_ALPHA):
                        arm_v.setdefault(arm, []).append("NO_GO_RESTORE_DRIFT (pass1_pass2)")
                    else:
                        notes.setdefault(arm, []).append(f"SUGGESTIVE_RESTORE_DRIFT (pass1_pass2, p={pv})")
            if w.get("c16_aa_pure") and ((w.get("c16_aa") or {}).get("state_flips") or 0) > 0:
                arm_v.setdefault(arm, []).append("NO_GO_STATE_FLIP (c16 A/A)")
        for key, c in out["aa"].items():
            if isinstance(c, dict) and (c.get("state_flips") or 0) > 0 and not str(c.get("status") or "").startswith("INCONCLUSIVE"):
                for a in key.split("~"):
                    arm_v.setdefault(a, []).append(f"NO_GO_STATE_FLIP (boot A/A {key})")
        for arm, r in out["m4"].items():
            if r.get("state_drift"):
                arm_v.setdefault(arm, []).append("NO_GO_STATE_DRIFT")
        out["arm_notes"] = notes
        out["arm_verdicts"] = arm_v
        if not SM and PART in ("A", "ALL"):
            cc = {a: (R["sections"]["1"].get(a) or {}).get("status") for a in ("s1_c_warm", "s1_c_nocache")}
            out["cache_controls"] = dict(cc, verdict="OK" if all(_usable(a) for a in cc) else "NO_GO_MISSING_COLD_WARM_CONTROLS")
        s1_apply_verdicts(out)
        return out
    def s1_apply_verdicts(cmp):
        """The section-1 arm verdicts reach the arm status (review #5: printing them was not enough). Recomputed on every
        call from the measured status (status_measured), so A/A data that arrives later can add or lift a verdict."""
        for arm, rec in (R["sections"].get("1") or {}).items():
            st = str((rec or {}).get("status") or "") if isinstance(rec, dict) else ""
            if st in ("", "NOT_RUN") or st.startswith("SKIPPED"):
                continue
            base = rec.setdefault("status_measured", rec["status"])
            rec["s1_verdicts"] = list((cmp.get("arm_verdicts") or {}).get(arm) or [])
            rec["status"] = worst([base] + [v.split(" ")[0] for v in rec["s1_verdicts"]])
    def _pool(srv):
        try:
            return int((srv.facts() or {}).get("kv_tokens") or 0) or None
        except Exception as e:
            err("pool/" + srv.name, e)
            return None
    def equal_n(band, streams, own_pool):
        """DESIGN §4.2: N = floor((pool - margin) / (band + growth + 2.9k)) from the CONTROL's pool, the same for every
        compared arm; an arm whose own pool cannot hold N records NOT_EQUAL_N (and runs at what it can hold)."""
        growth = int(30 * (ts(60) + WIN + ts(60)))
        pool = S2["pool"] or own_pool
        info = {"pool_source": "control" if S2["pool"] else "own", "pool": pool, "growth": growth}
        if not pool:
            return streams, dict(info, label="POOL_UNKNOWN")
        n = int(call(L.equal_n, pool, BANDS[band], growth) or 0)
        nn = max(1, min(streams, n))
        own = int(call(L.equal_n, own_pool, BANDS[band], growth) or 0) if own_pool else nn
        if own < nn:
            return max(1, own), dict(info, n_equal=nn, label="NOT_EQUAL_N", own_n=own)
        return nn, dict(info, n_equal=nn)
    def window(srv, arm, band, n, prompts=None, label=None, max_new=None, warm_ids=None):
        """flush + idle + snapshot, the optional shared-prefix warm request (AFTER the flush), then one steady window."""
        bl = between_loads("sglang", srv.port)
        if warm_ids is not None:
            bl["warm_http"] = warm(srv, warm_ids)
        w = call(L.steady, "sglang", srv.port, prompts or PROMPTS[band], n, secs=WIN, max_new=max_new or MAX_NEW.get(band, 16384),
                 temperature=1.0, model=srv.model_name, server=srv, min_depth=1000, prefill_tok_s=1000.0,
                 raw_dir=str(rawdir(arm) / f"win_{label or band}")) or {}
        w["between"] = bl
        w["n_offered"] = n
        return w
    def warm(srv, ids):
        """EMS steady_window warm request: one 1-token request so a shared prefix is in the radix cache first."""
        code, txt = http(srv.port, "/generate", {"input_ids": list(ids), "sampling_params": {"max_new_tokens": 1, "temperature": 0}},
                         timeout=tmo())
        return code
    def parity(srv, arm, role):
        """8k x 3 greedy rows (256 tokens): the control saves them, the A/A arm gives the noise count, challengers compare."""
        rows = keyed(call(L.greedy, "sglang", srv.port, PROMPTS["8k"][:3], n_tok=256, top=2, timeout=tmo(), model=srv.model_name,
                          keys=[f"8k#{i}" for i in range(3)]), [f"8k#{i}" for i in range(3)], "p:")
        jdump(rawdir(arm) / "parity_rows.json", rows)
        if role == "control":
            S2["ctrl_rows"] = rows
            return {"saved": len(rows), "status": "MEASURED"}
        if not S2["ctrl_rows"]:
            return {"status": "INCONCLUSIVE_NO_CONTROL_ROWS"}
        if role == "aa":
            S2["aa_rows"] = rows
        aah = hm(call(L.compare, S2["ctrl_rows"], S2["aa_rows"], kind="same_weights", t_pair=0.5, aa_high=None)) if S2["aa_rows"] else None
        c = call(L.compare, S2["ctrl_rows"], rows, kind="same_weights", t_pair=0.5, aa_high=aah) or {}
        return {"status": c.get("status"), "hm": hm(c), "aa_high": aah, "counts": _cls_counts(c)}
    def profile_load(srv, arm):
        """A separate 30-step profile under its own 8k load (never overlapping a measured window): breakdown only."""
        pdir = rawdir(arm) / "profile"
        shutil.rmtree(pdir, ignore_errors=True)
        return call(L.profiled_load, "sglang", srv.port, PROMPTS["8k"], 20, str(pdir), num_steps=30, max_new=MAX_NEW["8k"],
                    model=srv.model_name, prefill_tok_s=1000.0)
    def parity_gate(p):
        """Section-2 parity as a gate (§3.2 rule): FAIL -> NO_GO_PARITY; EXACT_TO_N / NEAR_TIE (control: rows saved) -> GO;
        BOUNDARY is GO only within a usable A/A allowance; anything else (no control rows, INCONCLUSIVE, BOUNDARY without an
        A/A, missing) -> INCONCLUSIVE_PARITY_* (review #1: never GO by default)."""
        st = str((p or {}).get("status") or "MISSING")
        if st.startswith("FAIL"):
            return "NO_GO_PARITY"
        if st in ("MEASURED", "EXACT", "EXACT_TO_N", "NEAR_TIE") or (st == "BOUNDARY" and (p or {}).get("aa_high") is not None):
            return "GO"
        return "INCONCLUSIVE_PARITY_" + (re.sub(r"^INCONCLUSIVE_?", "", st) or "UNJUDGED")
    def s2_arm(arm, bands, ems=(), role="challenger"):
        spec = CAT[arm]
        streams = int(flag_value(spec["flags"], "--max-running-requests")) - 1
        def body(srv):
            o = {"gates": {}, "measured": {}, "windows": {}, "sampled": True, "streams": streams}
            pool = _pool(srv)
            if role == "control" and pool:
                S2["pool"] = pool
            for b in bands:
                n, info = equal_n(b, streams, pool)
                w = o["windows"][b] = window(srv, arm, b, n)
                w["equal_n"] = info
            for e in ems:
                prompts = EMS[e]
                lv = streams if e != "xl" else max(1, min(streams, (pool or 700000) // 100000))
                o["windows"]["ems_" + e] = window(srv, arm, "8k", lv, prompts=prompts, label="ems_" + e, max_new=16384,
                                                  warm_ids=prompts[0] if e != "short" else None)
            if SM and role == "control":
                o["measured"]["profile"] = profile_load(srv, arm)
                bl = between_loads("sglang", srv.port)
                o["measured"]["census_90k"] = census_load(srv, arm, "90k")
                o["measured"]["census_90k"]["between"] = bl
                o["measured"]["interference_D2"] = interference_load(srv, arm, "D2", d0=None)
            o["parity_between"] = between_loads("sglang", srv.port)
            o["parity"] = parity(srv, arm, role)
            o["gates"]["parity"] = parity_gate(o["parity"])
            o["gates"]["windows"] = worst([mgate(w.get("status")) for w in o["windows"].values()])
            return o
        return lambda: with_server(arm, body)
    def _wnum(w, k):
        v = (w or {}).get(k)
        return v.get("mean") if isinstance(v, dict) else v
    def _s2_win_ok(w, strict=True):
        """A section-2 window usable in a comparison: a GO measurement status, a tok/s, the common N (no NOT_EQUAL_N), and
        (strict, every gain claim) STEADY, i.e. the offered N actually running: a CAPACITY_LIMITED window ran fewer streams
        (review 2 audit #2). The long-band hold check is lenient: running fewer streams IS the XL collapse it looks for."""
        return isinstance(w, dict) and mgate(w.get("status")) == "GO" and bool(_wnum(w, "tok_s")) and \
            (w.get("equal_n") or {}).get("label") != "NOT_EQUAL_N" and (not strict or w.get("status") == "STEADY")
    def _s2_gates_ok(rec, ref=False):
        """Every gate GO except the windows summary (each compared band is checked on its own). A reference (the control, an
        A/A boot) may have failed only the startup gate (startup time does not change numerics); a challenger may not
        (NO_GO_STARTUP_600 blocks adoption, §3.6)."""
        st = str((rec or {}).get("status") or "")
        if not isinstance(rec, dict) or st.startswith(("SKIPPED", "ERROR")) or st == "INCONCLUSIVE_HARD_STOP":
            return False
        g = {k: v for k, v in (rec.get("gates") or {}).items() if k != "windows" and not (ref and k == "launch" and v == "NO_GO_STARTUP_600")}
        return bool(g) and worst(g.values()) == "GO"
    def _s2_mean_ratio(rec, ref, bands=("8k", "40k")):
        """Mean tok/s ratio of an arm's windows over a reference's: (ratio, None), or (None, why) unless every band is usable
        on both sides."""
        ws, rs = (rec or {}).get("windows") or {}, (ref or {}).get("windows") or {}
        bad = [b for b in bands if not _s2_win_ok(ws.get(b)) or not _s2_win_ok(rs.get(b))]
        if bad:
            return None, "window not usable at " + ",".join(bad)
        r = [_wnum(ws[b], "tok_s") / _wnum(rs[b], "tok_s") for b in bands]
        return round(sum(r) / len(r), 4), None
    def s2_compare():
        """Section 2 (§4.2): every arm vs the control at each band (equal N): tok/s ratio, tokens / step, step ms, running p50.
        The A/A spread comes from control boots whose compared windows are valid on BOTH sides. A challenger is eligible only
        with every gate GO (startup included) and usable 8k / 40k windows on it and on the control; better = mean ratio - 1
        beyond the spread (§4.2 STOP: "net tok/s not better (beyond the A/A spread)"). best = the top better challenger that
        also holds 90k / 116k in s2_best_long (no XL collapse beyond the spread) and is reproduced in every evaluable A/A-round
        repeat (review: decision logic #1 / #2); best_candidate (8k / 40k only) picks the arm s2_best_long runs."""
        sec = R["sections"].get("2") or {}
        crec = sec.get("s2_w6s20") or {}
        ctrl = crec.get("windows") or {}
        ctrl_ok = _s2_gates_ok(crec, ref=True)
        sp = []
        for a in ["s2_w6s20_aa"] + sorted(x for x in sec if x.startswith("aa_ctrl_r")):
            aa = (sec.get(a) or {}).get("windows") or {}
            for b in ("8k", "40k"):
                if _s2_win_ok(ctrl.get(b)) and _s2_win_ok(aa.get(b)):
                    x, y = _wnum(ctrl[b], "tok_s"), _wnum(aa[b], "tok_s")
                    sp.append(abs(x - y) / x)
        spread = max(sp) if sp else None
        out = {"aa_spread": spread, "aa_points": len(sp), "control_ok": ctrl_ok, "arms": {}}
        for arm, rec in sec.items():
            if arm == "s2_w6s20" or not isinstance(rec, dict) or not rec.get("windows"):
                continue
            rows = {}
            for b, w in rec["windows"].items():
                c = ctrl.get(b)
                t1, t0 = _wnum(w, "tok_s"), _wnum(c, "tok_s")
                rows[b] = {"tok_s": t1, "ctrl_tok_s": t0, "ratio": round(t1 / t0, 4) if t1 and t0 else None,
                           "tokens_per_step": _wnum(w, "tokens_per_step"), "step_ms": _wnum(w, "step_ms"),
                           "running_p50": (((w.get("decode_lines") or {}).get("running")) or {}).get("p50"),
                           "client_step_ms_mean": (w.get("client_step_ms") or {}).get("mean"), "status": w.get("status"), "ctrl_status": (c or {}).get("status"),
                           "n": w.get("n_offered"), "ctrl_n": (c or {}).get("n_offered"), "equal_n": (w.get("equal_n") or {}).get("label")}
            r = [rows[b]["ratio"] for b in ("8k", "40k") if b in rows and rows[b]["ratio"]]
            mr, why = _s2_mean_ratio(rec, crec)
            no = (([] if ctrl_ok else ["control gates not GO"]) + ([] if _s2_gates_ok(rec) else ["arm gates not GO: " + _bad_gate_list(rec)])
                  + ([why] if why else []) + ([] if spread is not None else ["no usable A/A spread"]))
            out["arms"][arm] = {"bands": rows, "mean_ratio_8k_40k": mr, "raw_mean_ratio_8k_40k": round(sum(r) / len(r), 4) if len(r) == 2 else None,
                                "parity": (rec.get("parity") or {}).get("status"), "threshold": spread, "eligible": not no,
                                "not_eligible": no, "better": (mr - 1 > spread) if not no else None}
        w4r, w4why = _s2_mean_ratio(sec.get("s2_w4s20"), crec)
        if w4r and spread is not None:
            gain, thr = 1 / w4r - 1, max(0.05, 2 * spread)
            out["branch"] = {"w6_over_w4": round(gain, 4), "threshold": thr, "run": "s2_w7s20" if gain >= thr else "s2_w5s20"}
        else:
            out["branch"] = {"run": None, "why": "rule not evaluable: " + (w4why or "no usable A/A spread")}
        cand = [(a, v["mean_ratio_8k_40k"]) for a, v in out["arms"].items() if a in S2_CHALLENGERS and v.get("eligible")]
        cand.sort(key=lambda x: -x[1])
        out["ranking"] = cand
        top = next((a for a, _ in cand if out["arms"][a].get("better")), None)
        out["best_candidate"], out["best_or_top"] = top, top or (cand[0][0] if cand else None)
        out["best"], out["best_check"] = None, None
        if top:
            bl = sec.get("s2_best_long__" + top) or {}
            lr = {}
            for b in ("90k", "120k"):
                w, c = (bl.get("windows") or {}).get(b), ctrl.get(b)
                ok = _s2_win_ok(w, strict=False) and _s2_win_ok(c, strict=False)
                lr[b] = {"ratio": round(_wnum(w, "tok_s") / _wnum(c, "tok_s"), 4) if ok else None, "status": (w or {}).get("status"),
                         "ctrl_status": (c or {}).get("status"), "running_p50": (((w or {}).get("decode_lines") or {}).get("running") or {}).get("p50"),
                         "ctrl_running_p50": (((c or {}).get("decode_lines") or {}).get("running") or {}).get("p50")}
            if not bl.get("windows"):
                lv = "INCONCLUSIVE_LONG_BANDS_NOT_RUN"
            elif not _s2_gates_ok(bl) or any(v["ratio"] is None for v in lr.values()):
                lv = "INCONCLUSIVE_LONG_BANDS"
            else:
                lv = "STOP_XL_COLLAPSE" if any(v["ratio"] < 1 - spread for v in lr.values()) else "HOLDS"
            reps = {}
            for a in sorted(x for x in sec if x.startswith("aa_best_r") and x.endswith("__" + top)):
                i = a[len("aa_best_r"):].split("__")[0]
                ref, rep = sec.get(f"aa_ctrl_r{i}"), sec[a]
                use = f"aa_ctrl_r{i}" if _s2_gates_ok(ref, ref=True) and _s2_mean_ratio(ref, crec)[0] is not None else "s2_w6s20"
                rr, rwhy = _s2_mean_ratio(rep, ref if use != "s2_w6s20" else crec)
                st = str(rep.get("status") or "")
                ran = not (st.startswith(("SKIPPED", "ERROR")) or st == "INCONCLUSIVE_HARD_STOP" or not rep.get("windows"))
                bad = [f"{k}={v}" for k, v in (rep.get("gates") or {}).items() if k != "windows" and str(v).startswith(("NO_GO", "INVALID"))]
                v = ("NOT_REPRODUCED_GATE" if ran and bad else "UNEVALUABLE" if not ran or rr is None or not _s2_gates_ok(rep) else
                     "REPRODUCED" if rr - 1 > spread else "NOT_REPRODUCED")
                reps[a] = {"ratio": rr, "vs": use, "why": rwhy or (", ".join(bad) or None), "verdict": v}
            nr = [a for a, v in reps.items() if v["verdict"].startswith("NOT_REPRODUCED")]
            ok_r = [a for a, v in reps.items() if v["verdict"] == "REPRODUCED"]
            out["best"] = top if lv == "HOLDS" and ok_r and not nr else None
            out["best_check"] = {"candidate": top, "long": lr, "long_verdict": lv, "repeats": reps,
                                 "verdict": "CONFIRMED" if out["best"] else ("NOT_REPRODUCED" if nr and lv == "HOLDS" else
                                                                             "UNCONFIRMED_NO_REPEAT" if lv == "HOLDS" else lv)}
        return out
    def _bad_gate_list(rec):
        return ", ".join(f"{k}={v}" for k, v in ((rec or {}).get("gates") or {}).items() if v not in ("GO", None, "SKIPPED_FAKE"))[:200] or \
            str((rec or {}).get("status"))
    def s3_arm(arm, conf=False, rerun=False):
        spec = CAT[arm]
        nccl = {k: v for k, v in spec["env"].items() if k.startswith("NCCL_") and k not in ("NCCL_DEBUG", "NCCL_DEBUG_SUBSYS")}
        def body(srv):
            o = {"gates": {}, "measured": {}, "windows": {}, "sampled": True}
            pool = _pool(srv)
            if conf or rerun:
                bands = ("40k",) if rerun else ("40k", "90k")
                for b in bands:
                    n, info = equal_n(b, 20, pool)
                    o["windows"][b] = window(srv, arm, b, n)
                    o["windows"][b]["equal_n"] = info
                if rerun:
                    between_loads("sglang", srv.port)
                    t = time.time()
                    code = warm(srv, PROMPTS["40k"][23])
                    o["measured"]["cold_prefill_40k_sec"] = round(time.time() - t, 2) if code == 200 else None
            else:
                o["windows"]["8k"] = window(srv, arm, "8k", 20)
                between_loads("sglang", srv.port)
                o["measured"]["profile"] = profile_load(srv, arm)
            o["nccl"] = call(L.nccl_info, srv.log())
            trace, tsrc = (o["measured"].get("profile") or {}).get("trace"), arm
            if not trace and arm.startswith("s3_conf__"):
                tsrc = arm.split("__", 1)[1]
                trace = ((((R["sections"].get("3") or {}).get(tsrc) or {}).get("measured") or {}).get("profile") or {}).get("trace")
            o["env_proof"] = call(L.nccl_env_proof, spec["env"], o["nccl"], trace=trace) or {}
            o["env_proof"]["trace_from"] = tsrc if trace else None
            ps = o["env_proof"].get("status")
            o["gates"]["env_proof"] = "GO" if (ps == "APPLIED" or not nccl) else (ps if str(ps).startswith(SEV) else "INCONCLUSIVE_" + str(ps))
            o["gates"]["windows"] = worst([mgate(w.get("status")) for w in o["windows"].values()])
            return o
        return lambda: with_server(arm, body)
    def s3_compare():
        sec = R["sections"].get("3") or {}
        rows = {}
        for arm in S3_ARMS:
            rec = sec.get(arm) or {}
            w = (rec.get("windows") or {}).get("8k") or {}
            cs = w.get("client_step_ms") or {}
            tps = _wnum(w, "tokens_per_step")
            rows[arm] = {"status": rec.get("status"), "window": w.get("status"), "env_proof": (rec.get("gates") or {}).get("env_proof"),
                         "client_step_ms_mean": cs.get("mean") if isinstance(cs, dict) else None,
                         "client_step_ms_p95": cs.get("p95") if isinstance(cs, dict) else None, "tok_s": _wnum(w, "tok_s"),
                         "tok_s_per_token_per_step": round(_wnum(w, "tok_s") / tps, 2) if _wnum(w, "tok_s") and tps else None,
                         "running_p50": (((w.get("decode_lines") or {}).get("running")) or {}).get("p50")}
        cr = sec.get("s3_ctrl") or {}
        cms = rows.get("s3_ctrl", {}).get("client_step_ms_mean") if _s2_gates_ok(cr, ref=True) and mgate(rows.get("s3_ctrl", {}).get("window")) == "GO" else None
        valid = [(a, r["client_step_ms_mean"]) for a, r in rows.items() if a != "s3_ctrl" and r["client_step_ms_mean"]
                 and _s2_gates_ok(sec.get(a)) and mgate(r["window"]) == "GO" and r["env_proof"] == "GO" and cms and r["client_step_ms_mean"] < cms]
        valid.sort(key=lambda x: x[1])
        ctrl40r, ctrl90r = sec.get("s3_ctrl_rerun") or {}, (R["sections"].get("2") or {}).get("s2_w6s20") or {}
        ctrl40, ctrl90 = (ctrl40r.get("windows") or {}).get("40k") or {}, (ctrl90r.get("windows") or {}).get("90k") or {}
        spread = (R["comparisons"].get("2") or {}).get("aa_spread")
        conf = {}
        for a, rec in sec.items():
            if a.startswith("s3_conf__") and isinstance(rec, dict):
                ws = rec.get("windows") or {}
                bands = {b: {"tok_s": _wnum(ws.get(b), "tok_s"), "ctrl_tok_s": _wnum(c, "tok_s"), "status": (ws.get(b) or {}).get("status"),
                             "ctrl_status": c.get("status"), "n": (ws.get(b) or {}).get("n_offered"), "ctrl_n": c.get("n_offered"),
                             "ratio": round(_wnum(ws.get(b), "tok_s") / _wnum(c, "tok_s"), 4)
                             if _s2_win_ok(ws.get(b)) and _s2_win_ok(c) and _s2_gates_ok(cref, ref=True) else None,
                             "control": "s3_ctrl_rerun (same session)" if b == "40k" else "s2_w6s20 90k window"}
                         for b, c, cref in (("40k", ctrl40, ctrl40r), ("90k", ctrl90, ctrl90r))}
                rs = [v["ratio"] for v in bands.values()]
                conf[a] = {"bands": bands, "verdict": ("INCONCLUSIVE_ARM_" + str(rec.get("status")) if not _s2_gates_ok(rec) else
                                                       "INCONCLUSIVE_WINDOWS" if None in rs else "INCONCLUSIVE_NO_AA_SPREAD" if spread is None else
                                                       "STOP_SLOWER" if min(rs) < 1 - spread else
                                                       "GAIN_CONFIRMED" if min(rs) > 1 + spread else "NO_CLEAR_GAIN")}
        return {"arms": rows, "control_step_ms_mean": cms, "ranking_by_mean_step_ms": valid, "best_two": [a for a, _ in valid[:2]],
                "confirmations": conf, "aa_spread_from_section2": spread,
                "cold_prefill_40k_sec": ((sec.get("s3_ctrl_rerun") or {}).get("measured") or {}).get("cold_prefill_40k_sec"),
                "note": "gains are claimed only from the 40k / 90k confirmation windows + the control re-run (same session)"}
    CENSUS = {"90k": ("90k", 24), "116k": ("120k", 24), "shared": (None, 16)}
    def max_running(srv):
        """The resolved running cap for census(): /server_info effective_max_running_requests_per_dp, else its
        max_running_requests, else the capacity log line (WP2 contract: without it a capped load has no plateau)."""
        try:
            si = srv.server_info() or {}
            st = (si.get("internal_states") or [{}])[0] if isinstance(si.get("internal_states"), list) else {}
            for v in ((st or {}).get("effective_max_running_requests_per_dp"), si.get("max_running_requests")):
                if isinstance(v, int) and v > 0:
                    return v
        except Exception as e:
            err("max_running/" + srv.name, e)
        return (srv.facts() or {}).get("max_running")
    def census_load(srv, arm, load):
        """DESIGN §4.4: offer the full logical count (never a client cap); plateau = prefill counter flat AND no new Prefill
        batch line AND queue unchanged AND a blocking reason, held >= max(60, 1.5 x band / 1300) s; max_wait = offered x
        band / 1300 + 120 s (rev 2 blockers #2 / #4)."""
        band_key, offered = CENSUS[load]
        prompts = PROMPTS[band_key] if band_key else SHARED4
        band = BANDS[band_key] if band_key else max(len(p) for p in SHARED4)
        if not band_key:
            for g in range(4):
                warm(srv, SHARED4[4 * g])
        mr = max_running(srv)
        off = len(srv.log() or "")
        c = call(L.census, "sglang", srv.port, prompts, offered, secs=WIN, plateau_sec=ts(max(60.0, 1.5 * band / 1300.0)),
                 max_wait=ts(offered * band / 1300.0 + 120.0), server=srv, model=srv.model_name, band=band, max_running=mr,
                 raw_dir=str(rawdir(arm) / f"census_{load}"), tag=load) or {}
        c["offered"], c["band"], c["max_running_passed"] = offered, band, mr
        try:
            pls = L.prefill_lines(srv.log() or "", off)
            adm, new = c.get("admitted_client") or 0, sum(x.get("new_token") or 0 for x in pls)
            c["prefill_accounting"] = {"prefill_lines": len(pls), "new_tokens": new, "admitted": adm,
                                       "cached_tokens_logged": sum(x.get("cached_token") or 0 for x in pls),
                                       "new_per_admitted_over_band": round(new / (adm * band), 4) if adm and band else None}
        except Exception as e:
            err(f"census/{arm}/{load}/prefill_accounting", e)
        return c
    def _blocking(c):
        return str((c or {}).get("blocking") or (c or {}).get("blocking_reason") or "").lower()
    def s4_arm(arm, loads):
        def body(srv):
            o = {"gates": {}, "measured": {}, "sampled": True,
                 "label": "hook-on: the counter hook is live (natural #43567 exposure); tok/s at plateau labelled hook-on"}
            if arm == "s4_mem086":
                rows = call(L.greedy, "sglang", srv.port, PROMPTS["8k"][:20], n_tok=32, top=2, timeout=tmo(), model=srv.model_name,
                            concurrent=True) or []
                oom = bool(L.log_errors(srv.log()).get("oom_lines"))
                o["probe_8k"] = {"alive": srv.alive(), "oom": oom, "errors": [r.get("error") for r in rows if r.get("error")][:3]}
                if oom or not srv.alive():
                    o["gates"]["probe_8k"] = "NO_GO_OOM" if oom else "NO_GO_SERVER_DIED"
                    if oom:
                        o["expected_negative"] = "mem-fraction 0.86 OOM: an expected-negative result, not ERROR"
                    return o
            for ld in loads:
                bl = between_loads("sglang", srv.port)
                c = meas(o, "census_" + ld, lambda ld=ld: census_load(srv, arm, ld))
                if isinstance(c, dict):
                    c["between"] = bl
                st = str((c or {}).get("status"))
                o["gates"]["census_" + ld] = mgate(st)
                if st.startswith("FAIL_") or not srv.alive():
                    o["gates"]["census_" + ld] = mgate(st) if st.startswith("FAIL_") else "NO_GO_SERVER_DIED"
                    break
                pa = (c or {}).get("prefill_accounting") or {}
                if ld == "shared" and isinstance(pa.get("new_per_admitted_over_band"), (int, float)) and pa["new_per_admitted_over_band"] > 0.5:
                    o["gates"]["census_shared_prefix"] = "NO_GO_PREFIX_RECOMPUTE"
            if arm == "s4_lowcap":
                c90 = o["measured"].get("census_90k") or {}
                cap = ((c90.get("gauges_end") or {}).get("max_total_num_tokens") or (c90.get("gauges_mean") or {}).get("max_total_num_tokens"))
                o["lowcap_check"] = {"max_total_num_tokens": cap, "blocking": _blocking(c90)}
                o["gates"]["lowcap"] = ("GO" if isinstance(cap, (int, float)) and cap <= 300000 * 1.02 and "kv" in _blocking(c90)
                                        else "INCONCLUSIVE_LOWCAP_NOT_REPORTED")
            return o
        def f():
            rec = with_server(arm, body)
            cnt = (rec.get("hook") or {}).get("counters_max") or {}
            b = cnt.get("optional_alloc_attempts")
            rec["natural_exposure"] = {"optional_alloc_failed": cnt.get("optional_alloc_failed"),
                                       "stale_donation_candidate": cnt.get("stale_donation_candidate"),
                                       "per_1000_boundaries": (round(1000.0 * (cnt.get("optional_alloc_failed") or 0) / b, 3) if b else None),
                                       "boundaries": b}
            return rec
        return f
    LOADS5 = {"D0": (20, 0, False), "D1": (19, 1, True), "D2": (19, 1, False), "D3": (16, 4, False)}
    def s5_windows():
        """One fixed wall window per load, identical in every arm: preset from the planned prefill work at a conservative
        1,100 tok/s (EMS: 1,615 -> 1,280 tok/s by 86k context) + 30 s, so slower arms still finish inside it and read as
        slower, not INCONCLUSIVE (D0 / D1: 60 s); the control's observed latest injected first token is recorded against it."""
        return {"D0": ts(60), "D1": ts(60), "D2": ts(90000 / 1100 + 30), "D3": ts(4 * 90000 / 1100 + 30)}
    def interference_load(srv, arm, load, d0):
        nres, ninj, warm_ = LOADS5[load]
        W = s5_windows()
        res = PROMPTS["8k"][:nres]
        inj = [PROMPTS["90k"][23]] if warm_ else [PROMPTS["90k"][20 + i] for i in range(ninj)]
        bl = between_loads("sglang", srv.port)
        r = call(L.interference, "sglang", srv.port, res, inj if ninj else [], nres, warm_prompt=inj[0] if warm_ else None,
                 secs_after=ts(30), server=srv, model=srv.model_name, window_sec=W[load], d0_median_step=d0,
                 raw_dir=str(rawdir(arm) / f"load_{load}")) or {}
        r.update(window_preset_sec=W[load], between=bl)
        return r
    def _d0_step(r):
        """D0's median client step (ms): every later load of the arm counts stalls against it."""
        v = (r or {}).get("d0_median_step_ms")
        if v is None:
            v = ((r or {}).get("pre_injection_step_ms") or {}).get("p50")
        return v if isinstance(v, (int, float)) else None
    def s5_arm(arm):
        def body(srv):
            o = {"gates": {}, "measured": {}, "sampled": True, "windows_preset": s5_windows()}
            d0 = None
            for ld in LOADS5:
                r = meas(o, ld, lambda ld=ld: interference_load(srv, arm, ld, d0))
                if ld == "D0":
                    d0 = _d0_step(r)
                st = (r or {}).get("status")
                g = mgate(st)
                if (r or {}).get("starvation"):
                    g = "NO_GO_STARVATION"
                o["gates"]["load_" + ld] = g
            if arm == "s5_c4096":
                S5["control"] = {ld: max([x for x in ((o["measured"].get(ld) or {}).get("prefill_sec") or []) if x is not None] or [None],
                                         key=lambda x: -1 if x is None else x) for ld in LOADS5}
            o["control_latest_completion_sec"] = S5["control"]
            return o
        return lambda: with_server(arm, body)
    def s5_compare():
        sec = R["sections"].get("5") or {}
        ctrl = (sec.get("s5_c4096") or {}).get("measured") or {}
        out = {"windows_preset": s5_windows(), "control_latest_completion": S5["control"], "arms": {}, "stop": {}}
        order = sorted(sec, key=lambda a: a != "s5_c4096")
        for arm in order:
            rec = sec[arm]
            m = (rec or {}).get("measured") or {}
            out["arms"][arm] = {ld: {"decode_tok_s": (m.get(ld) or {}).get("tok_s_window"),
                                     "ctrl_decode_tok_s": (ctrl.get(ld) or {}).get("tok_s_window"),
                                     "starvation": (m.get(ld) or {}).get("starvation"), "max_gap_s": (m.get(ld) or {}).get("max_gap_s"),
                                     "stall_s": (m.get(ld) or {}).get("stall_s_max"),
                                     "prefill_s": max([x for x in ((m.get(ld) or {}).get("prefill_sec") or []) if x is not None] or [0]) or None,
                                     "decode_tokens_prefill_phase": (m.get(ld) or {}).get("resident_tokens_prefill_phase"),
                                     "decode_flat_s": (m.get(ld) or {}).get("decode_flat_sec"),
                                     "tpot_p95": ((m.get(ld) or {}).get("tpot_ms") or {}).get("p95"), "status": (m.get(ld) or {}).get("status"),
                                     "ctrl_status": (ctrl.get(ld) or {}).get("status")}
                                for ld in LOADS5}
            L5 = out["arms"][arm]
            starv = [l for l, v in L5.items() if v["starvation"]]
            bad = [l for l in ("D1", "D2", "D3") if mgate(L5[l]["status"]) != "GO" or mgate(L5[l]["ctrl_status"]) != "GO"
                   or L5[l]["starvation"] is None or not L5[l]["decode_tok_s"] or not L5[l]["ctrl_decode_tok_s"]]
            r_ = [L5[l]["decode_tok_s"] / L5[l]["ctrl_decode_tok_s"] for l in ("D1", "D2", "D3") if l not in bad]
            mean_r = round(sum(r_) / len(r_), 4) if r_ else None
            g5 = (rec or {}).get("gates") or {}
            startup = g5.get("launch") == "NO_GO_STARTUP_600"
            other = {k: v for k, v in g5.items() if not (k.startswith("load_") and v == "NO_GO_STARVATION")
                     and not (k == "launch" and v == "NO_GO_STARTUP_600")}
            ost = worst(other.values()) if other else ("GO" if str((rec or {}).get("status")) in ("GO", "NO_GO_STARTUP_600")
                                                       else str((rec or {}).get("status")))
            cv = (out["stop"].get("s5_c4096") or {}).get("verdict")
            if starv:
                v = "STOP_STARVATION"
            elif ost.startswith("NO_GO"):
                v = "STOP_" + ost
            elif arm == "s5_c4096":
                v = "CONTROL" if ost == "GO" and not bad else "INCONCLUSIVE_CONTROL"
            elif cv not in ("CONTROL", "STOP_STARVATION"):
                v = "INCONCLUSIVE_CONTROL_" + str(cv)
            elif mean_r is not None and mean_r < 0.97:
                v = "STOP_WORSE_MIXED" + ("_PARTIAL" if bad else "")
            elif bad:
                v = "INCONCLUSIVE_LOADS_" + "_".join(bad)
            elif ost != "GO":
                v = "INCONCLUSIVE_ARM_" + ost
            else:
                v = "NOT_WORSE"
            out["stop"][arm] = {"starvation_loads": starv, "mixed_ratio_d1_d3": mean_r if not bad else None,
                                "mixed_ratio_measured_loads": mean_r, "unusable_loads": bad, "verdict": v,
                                "adoption_blocked": "NO_GO_STARTUP_600" if startup else None}
        return out
    def gen_raw(port, body, timeout=None):
        code, txt = http(port, "/generate", body, timeout=timeout or tmo())
        try:
            r = json.loads(txt) if code == 200 else {"error": f"HTTP {code}: {txt[:300]}"}
        except Exception as e:
            r = {"error": f"{type(e).__name__}: {txt[:200]}"}
        return r[0] if isinstance(r, list) and r else r
    def s6_req(port, ids, max_new=1, score=False, ignore_eos=True, rid=None):
        """score=True: the full-vocab next-token distribution (lab next_token_dist, E17 method); else a greedy run (rid)."""
        if score:
            d = call(L.next_token_dist, port, list(ids), VOCAB, timeout=tmo()) or {}
            return {"rid": None, "cached": d.get("cached"), "lp": d.get("lp"), "error": d.get("error"), "sum_raw": d.get("sum_raw")}
        body = {"rid": rid or f"e24-{random.randrange(10 ** 12)}", "input_ids": list(ids),
                "sampling_params": {"temperature": 0.0, "max_new_tokens": int(max_new), "ignore_eos": ignore_eos}}
        r = gen_raw(port, body)
        meta = r.get("meta_info") or {}
        fr = meta.get("finish_reason")
        return {"rid": body["rid"], "cached": meta.get("cached_tokens"), "n_out": len(r.get("output_ids") or []),
                "finish": fr.get("type") if isinstance(fr, dict) else fr, "error": r.get("error")}
    def kl_rec(a, b):
        k = call(L.kl, a, b) if a and b else None
        d = [abs(x - y) for x, y in zip(a, b)] if a and b else []
        return {"kl": float(k) if k is not None else None, "max_abs_dlogp": max(d) if d else None,
                "top1_changed": (max(range(len(a)), key=a.__getitem__) != max(range(len(b)), key=b.__getitem__)) if a and b else None}
    def s6_t1(port, pname, seed_new, aa=3, cycles=3):
        """Upstream Class 1 (test_mamba_cache_kl.py, E17 e17_t1): reference = flush, warm W44 + W42, score P (aa times = the
        A/A); cycle = flush, warm, seed (P + seed_new greedy), score P (the reuse). Persisted per cycle."""
        Pp, pref = S6P[pname], [S6P["W44"], S6P["W42"]]
        tag = f"{pname}-{random.randrange(10 ** 9)}"
        out = {"prompt": pname, "len": len(Pp), "seed_new": seed_new, "safe_depth": max(len(p) for p in pref if Pp[:len(p)] == p),
               "ref_cached": [], "ref_flush_valid": [], "aa": [], "cycles": [], "tol": TOL}
        refs = []
        for _ in range(aa):
            f = flush("sglang", port)
            out["ref_flush_valid"].append(f["valid"])
            for p in pref:
                s6_req(port, p, 1)
            r = s6_req(port, Pp, 1, score=True)
            refs.append(r.get("lp"))
            out["ref_cached"].append(r.get("cached"))
        good = [x for x in refs if x]
        out["aa"] = [kl_rec(good[0], x) for x in good[1:]] if len(good) > 1 else []
        kls = [x["kl"] for x in out["aa"]]
        out["aa_kl_max"] = max(kls) if kls and None not in kls else float("nan")
        out["reference_valid"] = bool(len(good) == aa and all(out["ref_flush_valid"]) and all(c == out["safe_depth"] for c in out["ref_cached"])
                                      and math.isfinite(out["aa_kl_max"]) and out["aa_kl_max"] <= AA_MAX)
        for c in range(cycles):
            f = flush("sglang", port)
            for p in pref:
                s6_req(port, p, 1)
            srid, prid = f"e24-seed-{tag}-{c}", f"e24-probe-{tag}-{c}"
            s = s6_req(port, Pp, seed_new, rid=srid)
            r = s6_req(port, Pp, 1, score=True, rid=prid)
            k = kl_rec(good[0], r["lp"]) if good and r.get("lp") else {"kl": None, "top1_changed": None}
            out["cycles"].append({"cycle": c, "seed_rid": srid, "probe_rid": prid, "flush_valid": f["valid"], "seed_cached": s.get("cached"),
                                  "seed_finish": s.get("finish"), "probe_cached": r.get("cached"), "probe_kl": k["kl"],
                                  "top1_changed": k.get("top1_changed"), "error": s.get("error") or r.get("error")})
            jdump(WORK / "s6_cycles" / f"{tag}_{c}.json", out["cycles"][-1])
        out["_ref"] = good[0] if good else None
        return out
    def s6_events(hdir):
        """Per-rid hook evidence: optional-slot failures, stale donations (decode path confirmed at donation; spec path
        confirmed by a later spec_resolve crossed=True), lock releases, alloc asserts, grammar / injection events."""
        h = hook_files(hdir)
        ev = {"failed": set(), "confirmed": {}, "events": {}, "by_rid": {}}
        pend = {}
        for e in h["trace"]:
            k, rid = e.get("ev"), e.get("rid")
            ev["events"][k] = ev["events"].get(k, 0) + 1
            if rid:
                ev["by_rid"].setdefault(rid, {}).setdefault(k, 0)
                ev["by_rid"][rid][k] += 1
            if k in ("alloc_failed", "optional_alloc_failed") and rid:
                ev["failed"].add(rid)
            elif k == "donation" and e.get("stale_candidate") and rid:
                ln = e.get("returned") if isinstance(e.get("returned"), int) else e.get("cache_len")
                if ((e.get("marked") or {}).get("path")) == "decode":
                    ev["confirmed"].setdefault(rid, []).append(ln)
                else:
                    pend.setdefault(rid, []).append(ln)
            elif k == "spec_resolve" and rid and e.get("confirmed"):
                ev["confirmed"].setdefault(rid, []).extend(pend.pop(rid, []))
        by, ev["ranks"] = rank_files(hdir)
        ranks = [by[r] for r in sorted(by)]
        ev["counters"] = [c.get("counters") or {} for c in ranks]
        ev["alloc"] = [c.get("alloc") for c in ranks if c.get("alloc")]
        per = {}
        for c in ranks:
            for rid, d in (c.get("per_rid") or {}).items():
                for k, v in (d or {}).items():
                    per.setdefault(rid, {})[k] = max(per.get(rid, {}).get(k, 0), v or 0)
        ev["per_rid"] = per
        ev["failed"] |= {rid for rid, d in per.items() if d.get("optional_alloc_failed")}
        return ev
    S6_AGREE = {"t1": ("optional_alloc_failed", "stale_donation_candidate", "prefix_lock_released"),
                "g": ("grammar_steps", "grammar_truncated_steps", "false_crossing_possible", "false_crossing_returned"),
                "sent": ("draft_nan_injections", "neighbor_rows_nonfinite"), "nan": ("selector_nan_injections", "zero_q_rows", "first_candidate_picks")}
    def _ranks_agree(ev, names):
        """DESIGN §4.6 "with identical counters": the SPMD schedulers of the TP ranks take the same Mamba / grammar / sampling
        decisions, so a verdict counter seen on one rank only is a desync, never evidence (review 2 audit #4)."""
        cs = ev.get("counters") or []
        return len(cs) == TP and all(len({int((c or {}).get(n) or 0) for c in cs}) == 1 for n in names)
    def s6_events_settled(hdir, kind, tries=3):
        """s6_events after the hook's last flushes: re-read (2 s apart) until the verdict counters agree across the ranks."""
        ev = s6_events(hdir)
        for _ in range(tries - 1):
            if _ranks_agree(ev, S6_AGREE[kind]):
                break
            time.sleep(2.0)
            ev = s6_events(hdir)
        ev["ranks_agree"] = _ranks_agree(ev, S6_AGREE[kind])
        return ev
    def _csum(ev, name):
        vals = [int(c.get(name) or 0) for c in ev.get("counters") or []]
        return max(vals) if vals else None
    def s6_chain(t, cyc, ev):
        lens = [x for x in ev["confirmed"].get(cyc["seed_rid"], []) if isinstance(x, int)]
        link = isinstance(cyc["probe_cached"], int) and cyc["probe_cached"] > t["safe_depth"] and cyc["probe_cached"] in lens
        return {"failed": cyc["seed_rid"] in ev["failed"], "stale_confirmed": bool(lens), "stale_lens": lens, "restore_link": link,
                "hot": isinstance(cyc["probe_kl"], float) and cyc["probe_kl"] > TOL}
    def s6_verdict_unpatched(t1s, ev):
        """E17 e17_verdict: BUG_EXPOSED only when ONE valid cycle holds the whole chain (failure -> confirmed stale donation
        -> restore link -> KL above the fixed tolerance); evidence from different cycles never adds up."""
        if not t1s or not all(t["reference_valid"] for t in t1s):
            return "INCONCLUSIVE_INVALID_REFERENCE"
        ch = [s6_chain(t, c, ev) for t in t1s for c in t["cycles"] if c["flush_valid"]]
        if not ch:
            return "INCONCLUSIVE_INVALID_COLD_CONTROL"
        if any(x["failed"] and x["stale_confirmed"] and x["restore_link"] and x["hot"] for x in ch):
            return "BUG_EXPOSED" if ev.get("ranks_agree", True) else "INCONCLUSIVE_COUNTERS_DESYNC"
        return "INCONCLUSIVE_NOT_EXERCISED"
    def s6_verdict_patched(t1s, ev, safety, trigger="failed", need_safety=False):
        """Patched arm: CLEAN needs the trigger evidence in THIS arm (every valid cycle's seed rid failed its optional slot;
        the probe hit the safe depth), no stale donation, KL <= tol, no leak, and (FF-P / FF-P-D / N-P) the E17 safety cases
        with a valid allocator baseline; missing trigger -> INCONCLUSIVE_NOT_EXERCISED."""
        if not t1s or not all(t["reference_valid"] for t in t1s):
            return "INCONCLUSIVE_INVALID_REFERENCE"
        cyc = [(t, c) for t in t1s for c in t["cycles"] if c["flush_valid"]]
        if not cyc:
            return "INCONCLUSIVE_INVALID_COLD_CONTROL"
        stale = any(ev["confirmed"].get(c["seed_rid"]) for _, c in cyc) or (_csum(ev, "stale_donation_candidate") or 0) > 0
        hot = any(isinstance(c["probe_kl"], float) and c["probe_kl"] > TOL for _, c in cyc)
        if stale or hot or (safety or {}).get("stale") or ((safety or {}).get("leaks") and (safety or {}).get("status") == "OK"):
            return "FAIL"
        if any(c["probe_kl"] is None for _, c in cyc):
            return "INCONCLUSIVE_PROBE_MISSING"
        if need_safety and (safety or {}).get("status") != "OK":
            return "INCONCLUSIVE_SAFETY_" + str((safety or {}).get("status") or "NOT_RUN").replace("INCONCLUSIVE_", "")
        if trigger == "failed":
            ok = all(c["seed_rid"] in ev["failed"] and c["probe_cached"] == t["safe_depth"] for t, c in cyc)
        else:
            ok = (_csum(ev, trigger) or 0) >= 1
        if ok and not ev.get("ranks_agree", True):
            return "INCONCLUSIVE_COUNTERS_DESYNC"
        return "CLEAN" if ok else "INCONCLUSIVE_NOT_EXERCISED"
    def s6_safety(port, ref_t1, hdir):
        """E17 e17_safety: EOS, abort mid-decode, max_new, 5x reuse, cold after flush; after each the allocator's
        available slots vs the baseline (a leak = fewer) and a warm probe (deeper than safe with KL > tol = stale)."""
        Pp, pref, ref = S6P["P255"], [S6P["W44"], S6P["W42"]], ref_t1.get("_ref")
        def avail():
            time.sleep(3.0)
            vals = [a.get("available") for a in s6_events(hdir)["alloc"] if isinstance(a, dict) and a.get("available") is not None]
            return max(vals) if vals else None
        f0 = flush("sglang", port)
        base, rows = avail(), []
        def after(case, extra=None):
            for p in pref:
                s6_req(port, p, 1)
            r = s6_req(port, Pp, 1, score=True)
            k = kl_rec(ref, r["lp"]) if ref and r.get("lp") else {"kl": None}
            f = flush("sglang", port)
            a = avail()
            rows.append(dict({"case": case, "flush_valid": f["valid"], "available": a, "available_base": base,
                              "leak": bool(f["valid"]) and a is not None and base is not None and a < base,
                              "probe_cached": r.get("cached"), "probe_kl": k["kl"],
                              "stale": bool(r.get("cached") and r["cached"] > 128 and isinstance(k["kl"], float) and k["kl"] > TOL)},
                             **(extra or {})))
        r = s6_req(port, Pp, 64, ignore_eos=False)
        after("eos", {"finish": r.get("finish")})
        rid = f"e24-abort-{random.randrange(10 ** 9)}"
        th = threading.Thread(target=lambda: s6_req(port, Pp, 400, rid=rid), daemon=True)
        th.start()
        time.sleep(0.8)
        http(port, "/abort_request", {"rid": rid}, timeout=30)
        th.join(60)
        after("abort")
        r = s6_req(port, Pp, 48)
        after("max_new", {"n_out": r.get("n_out")})
        for _ in range(5):
            s6_req(port, Pp, 8)
        after("reuse5")
        flush("sglang", port)
        r = s6_req(port, Pp, 1, score=True)
        after("cold", {"cold_cached": r.get("cached")})
        out = {"rows": rows, "leaks": sum(1 for x in rows if x["leak"]), "stale": sum(1 for x in rows if x["stale"]),
               "available_base": base, "base_flush_valid": f0.get("valid"), "cold_cached_zero": rows[-1].get("cold_cached") == 0}
        out["status"] = ("INCONCLUSIVE_NO_ALLOCATOR_SNAPSHOT" if base is None or any(x["available"] is None for x in rows) else
                         "INCONCLUSIVE_FLUSH_INVALID" if not f0.get("valid") or not all(x["flush_valid"] for x in rows) else
                         "INCONCLUSIVE_BASELINE_INVALID" if any(x["available"] > base for x in rows) else
                         "INCONCLUSIVE_COLD_NOT_ZERO" if not out["cold_cached_zero"] else "OK")
        return out
    def _strip(t):
        return {k: v for k, v in t.items() if not k.startswith("_")}
    def s6_t1_arm(arm, patched, prompts=("P255",), seed_new=32, safety=False, natural=False):
        """FF-U / FF-P (target-only, interval 128), FF-U-D / FF-P-D (DFlash W6, interval 64, P255), N-P (natural)."""
        def body(srv):
            o = {"gates": {}, "measured": {}}
            hdir = rawdir(arm) / "hook"
            if natural:
                flush("sglang", srv.port)
                s6_req(srv.port, S6P["W42"], 1)
                o["measured"]["natural_seed"] = s6_req(srv.port, S6P["P255"], 32, rid="e24-natural-seed")
                o["measured"]["alive_after_seed"] = srv.alive()
                if not srv.alive():
                    hits = {s: srv.log().count(s) for s in ERR_SIGNS if s in srv.log()}
                    o["verdict"], o["log_error_hits"] = ("FAIL" if hits else "INCONCLUSIVE_SERVER_DIED"), hits
                    o["gates"]["natural"] = ("NO_GO_ASSERT_43559" if "Can not alloc mamba cache" in srv.log() else
                                             "NO_GO_PATCHED_ARM_FAILED" if hits else "ERROR_SERVER_DIED")
                    return o
            t1s = []
            for pn in prompts:
                t1s.append(s6_t1(srv.port, pn, seed_new))
                if not srv.alive():
                    break
            if safety and t1s and srv.alive():
                o["measured"]["safety"] = s6_safety(srv.port, t1s[0], hdir)
            time.sleep(3.0)
            ev = s6_events_settled(hdir, "t1")
            o["measured"]["t1"] = [_strip(t) for t in t1s]
            o["measured"]["rid_evidence"] = {"failed": sorted(ev["failed"])[:40], "confirmed": dict(list(ev["confirmed"].items())[:40]),
                                             "events": ev["events"], "counters": ev["counters"][:4]}
            for t in t1s:
                for c in t["cycles"]:
                    if c["flush_valid"]:
                        c["chain"] = s6_chain(t, c, ev)
            o["measured"]["t1"] = [_strip(t) for t in t1s]
            log_hits = {s: srv.log().count(s) for s in ERR_SIGNS if s in srv.log()}
            if patched:
                v = s6_verdict_patched(t1s, ev, o["measured"].get("safety"), trigger="prefix_lock_released" if natural else "failed",
                                       need_safety=safety)
                if log_hits:
                    v = "FAIL"
            else:
                v = s6_verdict_unpatched(t1s, ev)
            o["verdict"], o["log_error_hits"] = v, log_hits
            o["gates"]["verdict"] = {"CLEAN": "GO", "BUG_EXPOSED": "GO", "FAIL": "NO_GO_PATCHED_ARM_FAILED"}.get(v, v if v.startswith(SEV) else "INCONCLUSIVE_" + v)
            return o
        return lambda: with_server(arm, body, expect_errors=not patched)
    def ngram_launch_failed(rec):
        """§4.6 fallback trigger: the NGRAM arm ran but never reached its body (a refused flag, a failed or OOM launch, an
        invalid sm89 kernel), so #43029 has no verdict (review #12: not only NO_GO_LAUNCH*); a skipped arm is no trigger."""
        if not isinstance(rec, dict) or str(rec.get("status") or "").startswith("SKIPPED") or rec.get("verdict"):
            return False
        g = rec.get("gates") or {}
        return str(g.get("flags", "GO")) != "GO" or str(g.get("launch")) not in ("GO", "NO_GO_STARTUP_600")
    def s6_g_arm(arm, patched):
        """#43029 (upstream spec_grammar_at_43029.py port): a grammar-constrained NGRAM run whose last verify accepts past
        the terminator, swept across the 256 boundary; follow-up scored with the cache (hit) and after a flush (miss)."""
        def body(srv):
            o = {"gates": {}, "measured": {}}
            port = srv.port
            r0 = gen_raw(port, {"input_ids": G_FILLER[:64] + G_PHRASE * 4,
                                "sampling_params": {"max_new_tokens": 32, "regex": G_REGEX, "temperature": 0}})
            ids0 = r0.get("output_ids") or []
            if r0.get("error") or not ids0:
                o["gates"]["recipe"] = "INCONCLUSIVE_RECIPE_STOP_TOKEN"
                o["measured"]["stop_run"] = {"error": r0.get("error")}
                return o
            pattern = (G_PHRASE + [ids0[-1]]) * 6
            rows = []
            for plen in range(G_TRACK - 10, G_TRACK + 2):
                flush("sglang", port)
                for k in range(2):
                    gen_raw(port, {"text": f"Write a short story about planet number {2 * plen + k}. ", "sampling_params": {"max_new_tokens": 24, "temperature": 0}})
                prompt = G_FILLER[:max(0, plen - len(pattern))] + pattern
                con = gen_raw(port, {"rid": f"e24-g-{plen}", "input_ids": prompt,
                                     "sampling_params": {"max_new_tokens": 32, "regex": G_REGEX, "temperature": 0}})
                follow = prompt + list(con.get("output_ids") or []) + G_FOLLOW
                def score():
                    r = gen_raw(port, {"input_ids": follow, "sampling_params": {"max_new_tokens": 8, "temperature": 0},
                                       "return_logprob": True, "logprob_start_len": G_TRACK})
                    m = r.get("meta_info") or {}
                    return [x[0] for x in (m.get("input_token_logprobs") or []) if x and x[0] is not None], m.get("cached_tokens")
                hit, cached = score()
                flush("sglang", port)
                miss, _ = score()
                cm = con.get("meta_info") or {}
                rows.append({"prompt_len": plen, "finish": (cm.get("finish_reason") or {}).get("type") if isinstance(cm.get("finish_reason"), dict) else cm.get("finish_reason"),
                             "spec_verify_ct": cm.get("spec_verify_ct"), "cached": cached, "resumed": cached == G_TRACK,
                             "delta": round(sum(abs(a - b) for a, b in zip(hit, miss)), 4) if len(hit) == len(miss) and hit else None,
                             "len_ok": len(hit) == len(miss)})
                jdump(rawdir(arm) / "g_rows.json", rows)
                if not srv.alive():
                    break
            time.sleep(3.0)
            ev = s6_events_settled(rawdir(arm) / "hook", "g")
            c = {k: _csum(ev, k) for k in ("grammar_steps", "grammar_truncated_steps", "false_crossing_possible", "false_crossing_returned")}
            resumed = sum(1 for r in rows if r["resumed"])
            big = [r for r in rows if r["delta"] is not None and r["delta"] >= 3.0]
            o["measured"].update(rows=rows, counters=c, resumed_at_checkpoint=resumed, events=ev["events"])
            if patched:
                trig = (c["grammar_truncated_steps"] or 0) >= 1 and (c["false_crossing_possible"] or 0) >= 1 and resumed > 0
                if (c["false_crossing_returned"] or 0) > 0 or any(r["resumed"] for r in big):
                    v = "FAIL"
                elif any(r["resumed"] and r["delta"] is None for r in rows):
                    v = "INCONCLUSIVE_DELTA_MISSING"
                else:
                    v = "CLEAN" if trig else "INCONCLUSIVE_NOT_EXERCISED"
            else:
                chain = [r for r in big if r["resumed"] and (ev["per_rid"].get(f"e24-g-{r['prompt_len']}") or {}).get("false_crossing_returned")]
                v = "BUG_EXPOSED" if chain and (c["false_crossing_returned"] or 0) >= 1 else (
                    "INCONCLUSIVE_NOT_EXERCISED" if not (c["grammar_truncated_steps"] or 0) else "INCONCLUSIVE_NO_CHAIN")
            if v in ("CLEAN", "BUG_EXPOSED") and not ev.get("ranks_agree"):
                v = "INCONCLUSIVE_COUNTERS_DESYNC"
            o["verdict"] = v
            o["gates"]["verdict"] = {"CLEAN": "GO", "BUG_EXPOSED": "GO", "FAIL": "NO_GO_PATCHED_ARM_FAILED"}.get(v, v if v.startswith(SEV) else "INCONCLUSIVE_" + v)
            return o
        return lambda: with_server(arm, body, expect_errors=not patched)
    def greedy_rid_rows(port, prompts, rids, n_tok=256, conc=True):
        """Native /generate greedy rows WITH rids (T 0, top-2), in the lab's row shape, all requests concurrent."""
        rows = [None] * len(prompts)
        def one(i):
            t = time.time()
            r = gen_raw(port, {"rid": rids[i], "input_ids": list(prompts[i]), "return_logprob": True, "top_logprobs_num": 2,
                               "sampling_params": {"max_new_tokens": n_tok, "temperature": 0.0, "top_k": 1, "ignore_eos": True}})
            m = r.get("meta_info") or {}
            tops = m.get("output_top_logprobs") or []
            top2, marg = [], []
            for e in tops:
                e = sorted([x for x in (e or []) if x and x[0] is not None], key=lambda x: -x[0])
                top2.append([e[0][1], e[0][0], e[1][1], e[1][0]] if len(e) > 1 else None)
                marg.append(e[0][0] - e[1][0] if len(e) > 1 else None)
            rows[i] = {"key": rids[i], "tokens": list(r.get("output_ids") or []), "margins": marg, "top2": top2,
                       "lps": [x[0] for x in (m.get("output_token_logprobs") or [])], "text": r.get("text") or "", "error": r.get("error"),
                       "finish_reason": m.get("finish_reason"), "cached_tokens": m.get("cached_tokens"), "elapsed": round(time.time() - t, 2)}
        ths = [threading.Thread(target=one, args=(i,), daemon=True) for i in range(len(prompts))]
        for th in ths:
            th.start()
            if not conc:
                th.join()
        for th in ths:
            th.join(tmo())
        return [r or {"error": "NO_ROW", "tokens": []} for r in rows]
    def sentinel_load(port, conc, inject, tag):
        """The sentinel is submitted first and streams its first token before the other conc-1 requests are submitted
        (merge appends: it sits at batch index 0); inject=False uses a non-sentinel rid (the same load, no injection)."""
        srid = (SENT if inject else "e24-ctrl-") + f"{tag}-c{conc}"
        first, box = threading.Event(), {}
        def stream():
            body = {"rid": srid, "input_ids": list(P2K[0]), "stream": True, "return_logprob": True, "top_logprobs_num": 2,
                    "sampling_params": {"max_new_tokens": 256, "temperature": 0.0, "top_k": 1, "ignore_eos": True}}
            req = _U.Request(f"http://127.0.0.1:{port}/generate", data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
            ids = []
            try:
                with _U.urlopen(req, timeout=tmo()) as r:
                    for line in r:
                        line = line.decode("utf-8", "replace").strip()
                        if not line.startswith("data:") or line[5:].strip() == "[DONE]":
                            continue
                        d = json.loads(line[5:])
                        ids += list(d.get("output_ids") or [])
                        if ids:
                            first.set()
            except Exception as e:
                box["error"] = repr(e)[:200]
                err("sentinel_stream/" + srid, e)
            first.set()
            box["tokens"] = ids
        th = threading.Thread(target=stream, daemon=True)
        th.start()
        first.wait(tmo())
        rows = greedy_rid_rows(port, P2K[1:conc], [f"e24-nb-{tag}-c{conc}-{i}" for i in range(1, conc)]) if conc > 1 else []
        th.join(tmo())
        return {"sentinel_rid": srid, "sentinel_tokens": len(box.get("tokens") or []), "sentinel_error": box.get("error"), "rows": rows}
    def s6_inject_verdict(kind, patched, injected, nb, zq, fcp, neighbour_logged, cmp, alive, hits, agree=True):
        """#41736 sentinels / #41737 selector NaN. Unpatched: BUG_EXPOSED = the exact counter (non-finite neighbour rows /
        an all-zero q row) with the injection on every rank. Patched FAIL: that counter, a STATE_FLIP or garbage between the
        uninjected and injected neighbours, or an error sign in the log; a plain FLIP without an uninjected A/A is BOUNDARY,
        not FAIL (§3.2; review #9) -> INCONCLUSIVE_FLIP_NO_AA; a death without an error sign is not a verdict."""
        chain = (nb or 0) > 0 if kind == "sent" else (zq or 0) > 0
        if not patched:
            return ("BUG_EXPOSED" if agree else "INCONCLUSIVE_COUNTERS_DESYNC") if chain and injected else "INCONCLUSIVE_NOT_EXERCISED"
        if chain or hits or any(((c or {}).get("state_flips") or 0) or ((c or {}).get("garbage") or 0) for c in cmp.values()):
            return "FAIL"
        if not alive:
            return "INCONCLUSIVE_SERVER_DIED"
        if not injected or (kind == "sent" and not neighbour_logged) or (kind == "nan" and not (fcp or 0) > 0):
            return "INCONCLUSIVE_NOT_EXERCISED"
        return "INCONCLUSIVE_FLIP_NO_AA" if sum(hm(c) for c in cmp.values()) else "CLEAN" if agree else "INCONCLUSIVE_COUNTERS_DESYNC"
    def s6_inject_arm(arm, kind, patched):
        """#41736 sentinels (kind 'sent') / #41737 selector NaN (kind 'nan'): eager DFlash W6, T 0 everywhere, each
        concurrency with and without injection; neighbours compared with compare() (fails only at the calibrated margin)."""
        concs = (1, 8, 16) if kind == "sent" else (1, 8)
        def body(srv):
            o = {"gates": {}, "measured": {}, "loads": {}}
            caps = "Capture" in srv.log() and "CUDA graph begin" in srv.log()
            o["measured"]["decode_graphs_captured"] = caps
            if caps:
                o["gates"]["eager"] = "INVALID_DECODE_GRAPHS_CAPTURED"
            cmp, rows = {}, {}
            for c in concs:
                for inj in (False, True):
                    flush("sglang", srv.port)
                    ld = sentinel_load(srv.port, c, inj, f"{arm}")
                    jdump(rawdir(arm) / f"load_c{c}_{int(inj)}.json", ld)
                    rows[(c, inj)] = ld.pop("rows")
                    o["loads"][f"c{c}_{'inj' if inj else 'base'}"] = dict(ld, n_rows=len(rows[(c, inj)]),
                                                                          row_errors=[r.get("error") for r in rows[(c, inj)] if r.get("error")][:3])
                b, i = rows[(c, False)], rows[(c, True)]
                for r_b, r_i in zip(b, i):
                    r_i["key"] = r_b["key"] = r_b["key"].rsplit("-", 1)[-1]
                if b and i:
                    cmp[f"c{c}"] = call(L.compare, b, i, kind="same_weights", t_pair=0.5, aa_high=None)
            time.sleep(3.0)
            ev = s6_events_settled(rawdir(arm) / "hook", kind)
            per_rank = [int((c or {}).get("draft_nan_injections" if kind == "sent" else "selector_nan_injections") or 0)
                        for c in ev["counters"]]
            nb = _csum(ev, "neighbor_rows_nonfinite") if kind == "sent" else None
            zq = _csum(ev, "zero_q_rows") if kind == "nan" else None
            fcp = _csum(ev, "first_candidate_picks") if kind == "nan" else None
            oor = _csum(ev, "out_of_row_picks") if kind == "nan" else None
            o["measured"].update(per_rank_injections=per_rank, ranks=ev["ranks"], neighbor_rows_nonfinite=nb, zero_q_rows=zq,
                                 first_candidate_picks=fcp, out_of_row_picks_diagnostic=oor, events=ev["events"],
                                 neighbour_logged=(ev["events"].get("neighbor") or 0) > 0,
                                 compare={k: {"status": v.get("status"), "hm": hm(v)} for k, v in cmp.items()})
            if len(per_rank) != TP:
                o["gates"]["injection"] = "INCONCLUSIVE_RANKS_MISSING"
            elif len(set(per_rank)) > 1:
                o["gates"]["injection"] = "INVALID_INJECTION_DESYNC"
            injected = len(per_rank) == TP and min(per_rank) >= 1
            hits = {s: srv.log().count(s) for s in ERR_SIGNS if s in srv.log()}
            v = s6_inject_verdict(kind, patched, injected, nb, zq, fcp, o["measured"]["neighbour_logged"], cmp, srv.alive(), hits,
                                  agree=ev.get("ranks_agree", False))
            o["verdict"], o["log_error_hits"] = v, hits
            o["gates"]["verdict"] = {"CLEAN": "GO", "BUG_EXPOSED": "GO", "FAIL": "NO_GO_PATCHED_ARM_FAILED"}.get(v, v if v.startswith(SEV) else "INCONCLUSIVE_" + v)
            return o
        return lambda: with_server(arm, body, expect_errors=not patched)
    def s6_combined():
        def body(srv):
            o = {"gates": {}, "measured": {}}
            meas(o, "m1", lambda: m_rows("sglang", srv, "s6_combined", "m1", ORACLE, NTOK, "s:"))
            o["m6_tool"] = call(L.tool_probe, "sglang", srv.port, model=srv.model_name)
            ref = _rows("s1_d6", "m1")
            s1c = s1_compare_all()
            tp, tsrc = _tp("s1_d6", "s1_d6_r2")
            c, aa, ast, n = _pair(ref, _rows("s6_combined", "m1"), "same_weights", tp, [(s1c.get("aa_rate") or {}).get("s1_d6")],
                                  [(s1c.get("aa_state_rate") or {}).get("s1_d6")]) if ref else ({}, None, None, 0)
            pre_kill = srv.log().partition("SIGTERM received")[0]
            hits = {s: pre_kill.count(s) for s in ERR_SIGNS if s in pre_kill}
            own = _rows("s6_combined", "m1")
            o["measured"]["vs_s1_d6"] = {"status": c.get("status"), "hm": hm(c), "aa_high": aa, "aa_state_allow": ast, "rows_judged": n,
                                         "t_pair": tp, "t_pair_source": tsrc, "counts": _cls_counts(c), "state_flips": c.get("state_flips")}
            if not ref or not c:
                v = "INCONCLUSIVE_NO_REFERENCE_ROWS"
            elif hits or str(c.get("status")).startswith("FAIL") or (c.get("state_flips") or 0) > (ast or 0):
                v = "FAIL"
            elif not srv.alive() or not own or all(r.get("error") for r in own):
                v = "INCONCLUSIVE_SERVER_DIED"
            elif not o["m6_tool"].get("ok"):
                v = "FAIL"
            elif c.get("status") in ("EXACT", "EXACT_TO_N", "NEAR_TIE"):
                v = "PASS"
            else:
                v = "INCONCLUSIVE_" + str(c.get("status"))
            o["verdict"], o["log_error_hits"] = v, hits
            o["gates"]["verdict"] = {"PASS": "GO", "FAIL": "NO_GO_COMBINED"}.get(v, v)
            return o
        return lambda: with_server("s6_combined", body)
    def s6_unit():
        """GPU unit tests of #41736 / #41737 on overlay copies (never the venv), with the server env minus PYTHONPATH, own
        Triton / Inductor dirs, a timeout and killpg; trigger evidence = the imported module __file__ + sha recorded."""
        E = ENGINES["sglang"]
        raw = rawdir("s6_unit")
        root, cinfo = cache_root("s6_unit", CAT["s6_unit"])
        env = {k: v for k, v in E["env"].items() if k != "PYTHONPATH"}
        env.update(call(L.cache_env, "sglang", str(root)) or {})
        env.update(TRITON_CACHE_DIR=str(raw / "triton"), TORCHINDUCTOR_CACHE_DIR=str(raw / "inductor"), E24_WORK=str(WORK))
        out = {"gates": {}, "tests": {}, "cache": cinfo}
        for t in ("41736", "41737"):
            res_p = raw / f"unit_{t}.json"
            res_p.unlink(missing_ok=True)
            tsec = int(ts(360, 10))
            cmd = [E["vpy"], str(UNIT), "--test", t, "--out", str(res_p), "--patcher", str(MODS / "e24_sglang_patches.py"),
                   "--timeout", str(tsec)] + (["--site", str(PSITE)] if PSITE else [])
            R["argv"]["s6_unit_" + t] = cmd
            with open(LOGS / f"unit_{t}.log", "w") as fh:
                p = subprocess.Popen(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT, start_new_session=True, cwd=str(raw))
                try:
                    rc = p.wait(timeout=2 * (tsec + 120) + 180)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(p.pid, signal.SIGKILL)
                    except Exception:
                        pass
                    p.wait(30)
                    rc = -9
            try:
                res = json.loads(res_p.read_text())
            except Exception:
                res = {"ran": False, "reason": f"rc={rc}: no result file (logs/unit_{t}.log)"}
            res["rc"] = rc
            v = res.get("status") or ("INCONCLUSIVE_NOT_RUN" if not res.get("ran") else "INCONCLUSIVE_NO_STATUS")
            out["tests"][t] = {"verdict": v, **{k: res.get(k) for k in ("ran", "chain", "clean", "reason", "rc", "max_abs_delta_finite", "paths")},
                               "details": small(res.get("details")) if res.get("details") else None}
            if v == "PASS_AST_FALLBACK":
                out["tests"][t]["caveat"] = ("PASS_AST_FALLBACK: `import sglang.srt.models.dflash` failed standalone, so one side "
                                             "ran the overlay's _grouped_conv source text; not the plain module path")
            out["gates"]["unit_" + t] = ("GO" if v in ("PASS", "PASS_AST_FALLBACK") else ("NO_GO_UNIT_" + t + "_" + v) if str(v).startswith("FAIL")
                                         else v if str(v).startswith(SEV) else "INCONCLUSIVE_" + str(v))
        out["status"] = worst(out["gates"].values())
        gpu_gate("s6_unit", out)
        return out
    def s6_pairs():
        sec = R["sections"].get("6") or {}
        P6 = {"43567+43559 forced-fail (target-only)": ("s6_ffu", "s6_ffp"), "43567+43559 forced-fail (DFlash W6)": ("s6_ffud", "s6_ffpd"),
              "43029 grammar (NGRAM)": ("s6_gu", "s6_gp"), "43029 grammar (DFlash extra_buffer fallback)": ("s6_gu_df", "s6_gp_df"),
              "43029 grammar (production lazy, G-L)": ("s6_gl_u", "s6_gl_p"), "41736 sentinels": ("s6_sent_u", "s6_sent_p"),
              "41737 selector NaN": ("s6_nan_u", "s6_nan_p")}
        def vd(a):
            r = sec.get(a)
            if not isinstance(r, dict) or (not r.get("verdict") and str(r.get("status") or "").startswith("SKIPPED")):
                return None
            return r.get("verdict") or f"NO_VERDICT ({r.get('status')})"
        out = {}
        for name, (u, p) in P6.items():
            vu, vp = vd(u), vd(p)
            if vu is None and vp is None:
                continue
            out[name] = {"unpatched": vu, "patched": vp, "verdict": "FAIL" if vp == "FAIL" else
                         ("PASS" if vu == "BUG_EXPOSED" and vp == "CLEAN" else "INCONCLUSIVE")}
        vnp = vd("s6_np")
        if vnp:
            out["43559 natural (unpatched half = E17 run 1's recorded crash, README measured negative)"] = {
                "unpatched": "E17_RUN1_CRASH_PATH_43559 (recorded, not repeated)", "patched": vnp,
                "verdict": "PASS" if vnp == "CLEAN" else ("FAIL" if vnp == "FAIL" else "INCONCLUSIVE")}
        for t, r in ((sec.get("s6_unit") or {}).get("tests") or {}).items():
            out[f"unit {t}"] = {"verdict": r.get("verdict"), **({"caveat": r["caveat"]} if r.get("caveat") else {})}
        if sec.get("s6_combined"):
            out["combined (all five)"] = {"verdict": sec["s6_combined"].get("verdict")}
        return out
    def run_plan():
        sgl, vll = dep_engine("sglang"), dep_engine("vllm")
        for sec, a, est in PLAN:
            opt = a.endswith("?")
            a = a.rstrip("?")
            try:
                if a in S1_MEAS:
                    step(sec, a, est, s1(a), dep=vll if CAT[a]["engine"] == "vllm" else sgl)
                elif a == "s6_unit":
                    step(sec, a, est, s6_unit, dep=sgl)
                elif a in ("s6_ffu", "s6_ffp"):
                    step(sec, a, est, s6_t1_arm(a, a == "s6_ffp", safety=a == "s6_ffp"), dep=sgl)
                elif a == "s6_np":
                    step(sec, a, est, s6_t1_arm(a, True, safety=True, natural=True), dep=sgl)
                elif a in ("s6_ffud", "s6_ffpd"):
                    step(sec, a, est, s6_t1_arm(a, a == "s6_ffpd", safety=a == "s6_ffpd"), dep=sgl)
                elif a in ("s6_gu", "s6_gp", "s6_gl_u", "s6_gl_p"):
                    step(sec, a, est, s6_g_arm(a, a.endswith("p")), dep=sgl)
                elif a in ("s6_gu_df", "s6_gp_df"):
                    ng = {x: ngram_launch_failed(R["sections"]["6"].get(x)) for x in ("s6_gu", "s6_gp")}
                    if FORCE_COND or any(ng.values()):
                        step(sec, a, est, s6_g_arm(a, a == "s6_gp_df"), dep=sgl)
                    else:
                        skip_cond(sec, a, f"NGRAM arms launched ({ng}): the DFLASH extra_buffer fallback is not needed")
                elif a in ("s6_sent_u", "s6_sent_p"):
                    step(sec, a, est, s6_inject_arm(a, "sent", a.endswith("p")), dep=sgl)
                elif a in ("s6_nan_u", "s6_nan_p"):
                    step(sec, a, est, s6_inject_arm(a, "nan", a.endswith("p")), dep=sgl)
                elif a == "s6_combined":
                    step(sec, a, est, s6_combined(), dep=sgl)
                elif a == "s2_w6s20":
                    bands = ("8k",) if SM else ("8k", "40k", "90k", "120k")
                    step(sec, a, est, s2_arm(a, bands, ems=() if SM else ("short", "long", "xl"), role="control"), dep=sgl)
                elif a == "s2_w6s20_aa":
                    step(sec, a, est, s2_arm(a, ("8k", "40k"), role="aa"), dep=sgl)
                elif a in ("s2_w4s20", "s2_w6s16", "s2_w4s16"):
                    bands = ("8k",) if SM else ("8k", "40k")
                    step(sec, a, est, s2_arm(a, bands, ems=("long",) if a == "s2_w4s20" and not SM else ()), dep=sgl)
                    R["comparisons"]["2"] = s2_compare()
                elif a == "s2_w5s20|s2_w7s20":
                    cmpd = R["comparisons"]["2"] = s2_compare()
                    run = ["s2_w5s20", "s2_w7s20"] if FORCE_COND else ([cmpd["branch"]["run"]] if cmpd["branch"].get("run") else [])
                    for x in ("s2_w5s20", "s2_w7s20"):
                        if x in run:
                            step(sec, x, est, s2_arm(x, ("8k", "40k")), dep=sgl)
                        else:
                            skip_cond(sec, x, "branch rule: " + json.dumps(cmpd["branch"]))
                    R["comparisons"]["2"] = s2_compare()
                elif a == "s2_best_long":
                    cmpd = R["comparisons"]["2"] = s2_compare()
                    best = (cmpd["best_or_top"] or S2_CHALLENGERS[0]) if FORCE_COND else cmpd["best_candidate"]
                    if best:
                        step(sec, "s2_best_long__" + best, est, s2_arm("s2_best_long__" + best, ("90k", "120k")), dep=sgl)
                        R["comparisons"]["2"] = s2_compare()
                    else:
                        skip_cond(sec, "s2_best_long", "no eligible challenger beats the control beyond the A/A spread: " + json.dumps(
                            {a: v.get("not_eligible") or v.get("better") for a, v in cmpd["arms"].items() if a in S2_CHALLENGERS})[:400])
                elif a in S3_ARMS:
                    step(sec, a, est, s3_arm(a), dep=sgl)
                    R["comparisons"]["3"] = s3_compare()
                elif a.startswith("s3_conf__best"):
                    c3 = R["comparisons"]["3"] = s3_compare()
                    i = int(a[-1]) - 1
                    pool = c3["best_two"] or ([x for x in S3_ARMS if x != "s3_ctrl"][:2] if FORCE_COND else [])
                    if i < len(pool):
                        step(sec, "s3_conf__" + pool[i], est, s3_arm("s3_conf__" + pool[i], conf=True), dep=sgl)
                    else:
                        skip_cond(sec, a, "fewer than two valid section-3 arms to confirm")
                elif a == "s3_ctrl_rerun":
                    step(sec, a, est, s3_arm(a, rerun=True), dep=sgl)
                elif a in ("s4_m84", "s4_m68", "s4_mem080", "s4_mem086", "s4_lowcap"):
                    loads = ("90k", "116k", "shared") if a == "s4_m84" else ("90k",) if a == "s4_lowcap" else ("90k", "116k")
                    step(sec, a, est, s4_arm(a, loads), dep=sgl)
                elif a in ("s4_m88", "s4_m92"):
                    m84 = (R["sections"]["4"].get("s4_m84") or {}).get("measured") or {}
                    mb = [ld for ld in ("census_90k", "census_116k") if "mamba" in _blocking(m84.get(ld))]
                    if FORCE_COND or mb:
                        step(sec, a, est, s4_arm(a, ("90k", "116k")), dep=sgl)
                    else:
                        skip_cond(sec, a, "the control is not Mamba-bound by the direct gauges: " + json.dumps({k: _blocking(m84.get(k)) for k in ("census_90k", "census_116k")}))
                elif a in ("s5_c4096", "s5_c2048", "s5_c8192", "s5_pdi1", "s5_pdi4", "s5_mixed"):
                    step(sec, a, est, s5_arm(a), dep=sgl)
                    R["comparisons"]["5"] = s5_compare()
                elif a.startswith("aa_ctrl_r"):
                    i = int(a[-1])
                    cmpd = s2_compare()
                    best = cmpd["best_or_top"] or (S2_CHALLENGERS[0] if FORCE_COND else None)
                    if left() < (2 * 18 + 5 + 10) * 60 * TS:
                        skip_cond(sec, a, "A/A rounds stop: less than one round + the report reserve left")
                        continue
                    step(sec, a, est, s2_arm(a, ("8k", "40k"), role="aa_repeat"), dep=sgl)
                    if best:
                        step(sec, f"aa_best_r{i}__{best}", est, s2_arm(f"aa_best_r{i}__{best}", ("8k", "40k"), role="aa_repeat"), dep=sgl)
                else:
                    skip_cond(sec, a, "no runner for this plan entry (builder bug)")
                    err("plan/" + a, None, note="no runner")
            except Exception as e:
                err(f"plan/{a}", e)
                R["sections"].setdefault(sec, {}).setdefault(a, {}).update(status="ERROR_" + type(e).__name__)
            if sec == "6":
                R["comparisons"]["6"] = s6_pairs()
            save()
    if not ABORT["why"]:
        run_plan()
    else:
        for _sec, _a, _est in PLAN:
            step(_sec, _a.rstrip("?"), _est, lambda: {"status": "SKIPPED_ABORT"})


In [ ]:
if not RERUN:
    for _k, _fn in (("1", lambda: s1_compare_all(final=True)), ("2", s2_compare), ("3", s3_compare), ("5", s5_compare), ("6", s6_pairs)):
        if not R["sections"].get(_k):
            continue
        try:
            R["comparisons"][_k] = _fn()
        except Exception as e:
            err("report/comparisons/" + _k, e)
    for _a, _r in (R["sections"].get("1") or {}).items():
        if isinstance(_r, dict) and _r.get("status_measured"):
            try:
                jdump(ARMS_OUT / f"{_a}.json", _r)
            except Exception as e:
                err(f"report/arm_summary/{_a}", e)
    try:
        if not FAKE and not L.LIVE:
            R["orphans_killed_at_end"] = L.kill_orphans(fake=False)
    except Exception as e:
        err("report/kill_orphans", e)
    def _f(x, nd=1):
        return "-" if x is None else (f"{x:.{nd}f}" if isinstance(x, float) else str(x))
    def _bad_gates(rec):
        return ", ".join(f"{k}={v}" for k, v in ((rec or {}).get("gates") or {}).items() if v not in ("GO", None, "SKIPPED_FAKE"))[:160] or "-"
    md = [f"# E24 {VARIANT} report ({R['start_utc']}, {round((time.time() - T_START) / 60, 1)} min, fake={FAKE})", "",
          f"abort: {ABORT['why']} | errors: {len(R['errors'])} | api drift: {json.dumps(R['api_drift'])[:300]}", "",
          "## Steps", "| section | arm | status | ready s | min | not-GO gates |", "|---|---|---|---|---|---|"]
    for _o in R["order"]:
        _s, _a = _o.split("/", 1)
        _r = R["sections"].get(_s, {}).get(_a) or {}
        md.append(f"| {_s} | {_a} | {_r.get('status')} | {_f(_r.get('ready_sec'))} | {_f(_r.get('minutes'), 2)} | {_bad_gates(_r)} |")
    c1 = R["comparisons"].get("1") or {}
    if c1.get("pairs"):
        md += ["", "## Section 1: oracle pairs (aligned by key; T_pair calibrated; A/A allowance scaled to the judged rows)",
               "| pair | kind | status | high-margin | A/A allowance | Fisher p vs A/A | T_pair | positions | attribution |",
               "|---|---|---|---|---|---|---|---|---|"]
        for _k, _p in c1["pairs"].items():
            _at = ",".join(sorted({x.get("attribution") for x in _p.get("attribution") or []})) or "-"
            md.append(f"| {_k} ({_p.get('label')}) | {_p.get('kind')} | {_p.get('status')} | {_f(_p.get('hm'))} | {_f(_p.get('aa_high'))} | "
                      f"{_f(_p.get('p_value'), 4)} | {_f(_p.get('t_pair'), 3)} ({_p.get('t_pair_source', '-')}) | "
                      f"{_f(_p.get('positions_examined'))} | {_at} |")
        md += ["", "Decisive chain a -> b -> c32 -> c -> d6 / d4: " + json.dumps(c1.get("decisive"))[:600],
               "", "M4 decode vs prefill: " + json.dumps({a: {"own": (v.get("own") or {}).get("status"), "vs_base": (v.get("vs_base") or {}).get("status"),
                                                               "state_drift": v.get("state_drift")} for a, v in (c1.get("m4") or {}).items()}),
               "", "Arm verdicts (applied to the arm statuses; STATE_FLIP / STATE_DRIFT / target-only drift / restore): "
               + json.dumps(c1.get("arm_verdicts")), "", "Suggestive only (EMS rule without Fisher significance; not NO-GO): "
               + json.dumps(c1.get("arm_notes")), "", "Cold / warm controls: " + json.dumps(c1.get("cache_controls")),
               "", "| arm | pass 1 <-> pass 2 (restore) | restore | serial vs c16 | c16 A/A | A/A rate (FLIP+STATE / STATE per row) |",
               "|---|---|---|---|---|---|"]
        for _a, _w in (c1.get("within") or {}).items():
            md.append(f"| {_a} | {(_w.get('pass1_pass2') or {}).get('status', '-')} | {_w.get('pass2_restore', '-')} | "
                      f"{(_w.get('serial_vs_c16') or {}).get('status', '-')} | {(_w.get('c16_aa') or {}).get('status', '-')}"
                      f"{'' if _w.get('c16_aa_pure', True) else ' (not pure: a flush failed)'} | "
                      f"{_f((c1.get('aa_rate') or {}).get(_a), 3)} / {_f((c1.get('aa_state_rate') or {}).get(_a), 3)} |")
    c2 = R["comparisons"].get("2") or {}
    if c2.get("arms"):
        md += ["", f"## Section 2: width x streams vs W6 S20 (A/A spread {_f(c2.get('aa_spread'), 4)}; branch {json.dumps(c2.get('branch'))})",
               f"best (confirmed: 8k / 40k beyond the spread + 90k / 116k hold + reproduced): {c2.get('best')} | candidate: "
               f"{c2.get('best_candidate')} | check: {json.dumps(c2.get('best_check'))[:700]}",
               "eligibility: " + json.dumps({_a: (_v.get('not_eligible') or 'eligible') for _a, _v in c2['arms'].items()})[:1200], "",
               "| arm | band | tok/s | control | ratio | tok/step | step ms | running p50 | status | N |", "|---|---|---|---|---|---|---|---|---|---|"]
        for _a, _v in c2["arms"].items():
            for _b, _w in (_v.get("bands") or {}).items():
                md.append(f"| {_a} | {_b} | {_f(_w.get('tok_s'))} | {_f(_w.get('ctrl_tok_s'))} | {_f(_w.get('ratio'), 3)} | {_f(_w.get('tokens_per_step'), 2)} | "
                          f"{_f(_w.get('step_ms'))} | {_f(_w.get('running_p50'))} | {_w.get('status')} | {_w.get('n')} {_w.get('equal_n') or ''} |")
    c3 = R["comparisons"].get("3") or {}
    if c3.get("arms"):
        md += ["", "## Section 3: NCCL arms (8k, N = 20; ranking by mean client step ms)", "| arm | status | window | env proof | step ms mean | p95 | tok/s | tok/s per tok/step |",
               "|---|---|---|---|---|---|---|---|"]
        for _a, _v in c3["arms"].items():
            md.append(f"| {_a} | {_v.get('status')} | {_v.get('window')} | {_v.get('env_proof')} | {_f(_v.get('client_step_ms_mean'))} | "
                      f"{_f(_v.get('client_step_ms_p95'))} | {_f(_v.get('tok_s'))} | {_f(_v.get('tok_s_per_token_per_step'))} |")
        md.append("control mean step ms: " + _f(c3.get("control_step_ms_mean")) + " | best two (GO, faster than the control): "
                  + json.dumps(c3.get("best_two")) + " | confirmations: " + json.dumps(c3.get("confirmations"))[:1200]
                  + " | control cold 40k prefill s: " + str(c3.get("cold_prefill_40k_sec")))
    _s4 = R["sections"].get("4") or {}
    _cen = [(a, k, v) for a, r in _s4.items() for k, v in ((r or {}).get("measured") or {}).items() if k.startswith("census_") and isinstance(v, dict)]
    if SM:
        _cen += [("s2_w6s20", "census_90k", ((R["sections"]["2"].get("s2_w6s20") or {}).get("measured") or {}).get("census_90k") or {})]
    if _cen:
        md += ["", "## Section 4: capacity census (offered = the full logical count)",
               "| arm | load | status | running p50 | blocking | KV pool | retractions | tok/s | new prefill / admitted / band |",
               "|---|---|---|---|---|---|---|---|---|"]
        for _a, _k, _v in _cen:
            _rp = ((_v.get("running_lines") or {}).get("p50") if (_v.get("running_lines") or {}).get("n") else (_v.get("running_gauge") or {}).get("p50"))
            md.append(f"| {_a} | {_k} | {_v.get('status')} | {_f(_rp)} | {_v.get('blocking_reason')} | "
                      f"{_f((_v.get('gauges_end') or {}).get('max_total_num_tokens') or (_v.get('gauges_mean') or {}).get('max_total_num_tokens'))} | "
                      f"{_f(_v.get('retractions'))} | {_f(_v.get('tok_s'))} | "
                      f"{_f((_v.get('prefill_accounting') or {}).get('new_per_admitted_over_band'), 3)} |")
    c5 = R["comparisons"].get("5") or {}
    if c5.get("arms"):
        md += ["", "## Section 5: interference (fixed windows " + json.dumps({k: round(v, 1) for k, v in c5['windows_preset'].items()}) + ")",
               "| arm | load | decode tok/s | control | starvation | max gap s | stall s | prefill s | TPOT p95 | status |", "|---|---|---|---|---|---|---|---|---|---|"]
        for _a, _v in c5["arms"].items():
            for _l, _w in _v.items():
                md.append(f"| {_a} | {_l} | {_f(_w.get('decode_tok_s'))} | {_f(_w.get('ctrl_decode_tok_s'))} | {_w.get('starvation')} | {_f(_w.get('max_gap_s'), 2)} | "
                          f"{_f(_w.get('stall_s'), 2)} | {_f(_w.get('prefill_s'), 1)} | {_f(_w.get('tpot_p95'), 4)} | {_w.get('status')} |")
        md.append("STOP verdicts (starvation on any load, or mixed-window throughput < 0.97 x the 4096 control): "
                  + json.dumps(c5.get("stop")))
    c6 = R["comparisons"].get("6") or {}
    if c6:
        md += ["", "## Section 6: backports (PASS = unpatched chain + patched clean WITH trigger evidence in the patched arm)",
               "| test | unpatched | patched | verdict |", "|---|---|---|---|"]
        for _k, _v in c6.items():
            md.append(f"| {_k} | {_v.get('unpatched', '-')} | {_v.get('patched', '-')} | {_v.get('verdict')}"
                      + (f" (caveat: {_v['caveat'][:90]})" if _v.get("caveat") else "") + " |")
    _sat = [(s, a, k, w.get("tok_s"), w.get("status")) for s in ("2", "3") for a, r in (R["sections"].get(s) or {}).items()
            for k, w in ((r or {}).get("windows") or {}).items() if isinstance(w, dict)]
    _mix = [(a, l, w.get("decode_tok_s"), w.get("status")) for a, v in (c5.get("arms") or {}).items() for l, w in v.items()]
    R["denominators"] = {"saturated_decode": [list(x) for x in _sat], "mixed_serving_window": [list(x) for x in _mix],
                         "agent_end_to_end": "NOT_MEASURED: " + R["not_measured"]["agent_end_to_end_denominator"]}
    md += ["", "## Throughput denominators (Y11)", f"- saturated decode (steady windows): {len(_sat)} windows",
           f"- mixed serving window (section 5 fixed windows): {len(_mix)} loads", "- agent end to end: NOT_MEASURED",
           "", "NOT_MEASURED: " + json.dumps(R["not_measured"])]
    _s = lambda sec, a: (R["sections"].get(sec) or {}).get(a) or {}
    _flag = [r.get("flag_check") or {} for sec in R["sections"].values() for r in sec.values() if isinstance(r, dict) and r.get("flag_check")]
    R["machinery"] = {
        "inputs": R["gates"].get("inputs"), "tokenizer_ids_identical": (R.get("prompts") or {}).get("tokenizer", {}).get("fp8_int4_ids_identical"),
        "sglang_install": not ENGINES["sglang"].get("failed"), "vllm_install": not ENGINES["vllm"].get("failed") if NEED_VLLM else "not needed",
        "staging_sha256": bool(R["identity"].get("int4_shard_sha256")) and R["identity"].get("int4_shard_sha256") != "NOT_STAGED",
        "base_argv_matches_design": (R["setup"].get("base_check") or {}).get("ok"),
        "flag_checks_ran_ok": f"{sum(1 for f in _flag if f.get('ran') and f.get('ok'))}/{len(_flag)}",
        "flag_resolve_advisory_failures": sorted({str(f.get("resolve_error"))[:160] for f in _flag if f.get("resolve_advisory")}),
        "patch_site": R["setup"].get("patch_site"), "patch_broken": PATCH_BROKEN,
        "hook_selftest_go": sum(1 for sec in R["sections"].values() for r in sec.values() if isinstance(r, dict) and (r.get("gates") or {}).get("hook") == "GO"),
        "cache_isolated": {a: (r.get("cache_verdict") or {}).get("status") for a, r in (R["sections"].get("1") or {}).items() if isinstance(r, dict)},
        "tf_preflight": PREFLIGHT_DONE, "traces": {k: len(v.get("cont") or []) for k, v in TRACES.items()},
        "m4_arms": sorted(M4D), "unit_tests": {t: v.get("verdict") for t, v in (_s("6", "s6_unit").get("tests") or {}).items()},
        "census_90k": (((_s("2", "s2_w6s20").get("measured") or {}).get("census_90k")) or {}).get("status") if SM else None,
        "interference_D2": (((_s("2", "s2_w6s20").get("measured") or {}).get("interference_D2")) or {}).get("status") if SM else None,
        "profile": {a: (lambda p: isinstance(p, dict) and not p.get("error") and len(p.get("ranks") or []) >= TP and isinstance(p.get("trace"), dict)
                           and not p["trace"].get("error"))((_s(sec, a).get("measured") or {}).get("profile")) for sec, a in (("2", "s2_w6s20"), ("3", "s3_cta4"))},
        "nccl_env_proof_cta4": {k: (_s("3", "s3_cta4").get("env_proof") or {}).get(k) for k in ("status", "checks")},
    }
    md += ["", "## Machinery (SMOKE gate summary)", "```", json.dumps(R["machinery"], indent=1, default=str)[:6000], "```"]
    R["report_md"] = "\n".join(md)
    (OUT / "e24_report.md").write_text(R["report_md"])
    _skip = {"cache", "mods", "e24_hook_site", "e24_libcuda"}
    _budget, _used, _skipped = 2 * 1024 ** 3, 0, []
    try:
        with tarfile.open(OUT / "e24_raw.tar.gz.tmp", "w:gz") as _tf:
            for _d in sorted(WORK.iterdir()):
                if not _d.is_dir() or _d.name in _skip:
                    continue
                for _p in sorted(_d.rglob("*")):
                    if _p.is_file():
                        _sz = _p.stat().st_size
                        if _used + _sz > _budget:
                            _skipped.append(str(_p))
                            continue
                        _tf.add(_p, arcname=str(_p.relative_to(WORK)))
                        _used += _sz
        (OUT / "e24_raw.tar.gz.tmp").replace(OUT / "e24_raw.tar.gz")
        R["raw_tar"] = {"bytes_in": _used, "skipped_over_budget": _skipped[:50], "n_skipped": len(_skipped)}
        if not _skipped:
            shutil.rmtree(RAW_ARMS, ignore_errors=True)
    except Exception as e:
        err("report/raw_tar", e)
    _n = sum(1 for _p in OUT.rglob("*") if _p.is_file())
    if _n >= 300:
        with tarfile.open(OUT / "e24_arms.tar.gz", "w:gz") as _tf:
            _tf.add(ARMS_OUT, arcname="arms")
        shutil.rmtree(ARMS_OUT, ignore_errors=True)
        _n = sum(1 for _p in OUT.rglob("*") if _p.is_file())
    R["out_files"] = _n
    R["minutes"] = round((time.time() - T_START) / 60, 1)
    save()
    assert _n < 300, f"/kaggle/working/e24 holds {_n} files (>= 300): Kaggle's output listing would truncate"
    print(R["report_md"])
    print("E24", VARIANT, "done in", R["minutes"], "min | abort:", ABORT["why"], "| errors:", len(R["errors"]), "| out files:", _n)
